# v19 — fresh / delta / resume

בתא 3 משנים רק `RUN_MODE`: ברירת המחדל היא `"delta"`; `"fresh"` מתחיל ללא שימוש בתחזיות קודמות, ו־`"resume"` ממשיך את `RESUME_RUN_NAME` עם המקור המקורי שלו.

קלט קבוע: `Data/corpus_paragraphs.parquet`. פלט קבוע בכל המצבים: `Data/georag_selected_v2.parquet` ו־`Data/georag_selected_v2.run.json`. דלתאות מפיקות תוצאה מלאה לקורפוס הנוכחי.

גיבויים חדשים נשמרים באופן מצטבר, עם תכני JSON מקוריים בתוך Parquet דחוס ZSTD ועם הפניות קטנות; ההעלאה ל־GitHub עדיין בחבילות ZIP. ניקוי מקומי מסיר רק עותקי ZIP שאומתו בגיבוי המרוחק. ניקוי מרוחק מוגבל לגיבוי legacy של אותה ריצה אחרי שחזור עצמאי ובדיקת כיסוי. הוא אינו מוחק ריצות אחרות או חבילות הדרושות לנקודת השחזור. חבילות שלא זוהו משאירות את הגיבוי הישן במקומו; דוח הבדיקה שבוצע בנפרד במחשב אינו מיובא אוטומטית למחברת כהיתר מחיקה.

אין צורך ב־Google Drive. זיהוי GitHub אוטומטי דורש אישור נגיש בסביבה שבה פועל ה־kernel; סוד באתר Colab אינו משותף אוטומטית ל־kernel מקומי ב־VS Code. הטוקן אינו מודפס.

# v18: atomic Git-protocol publication; final filename georag_selected_v2.parquet

# v17: raw owners are validated once in cell 11; all span/export checks remain.

# GeoRAG v16 — compact Parquet checkpoints and verified retirement

Based on v15. The models, pinned revisions, batches, cleaning, alignment, selection,
offset checks and fixed final output Data/georag_selected_v2.parquet are unchanged.

Default: resume the saved run, preferring matching local restored_* folders. Source:
Data/corpus_paragraphs.parquet. Fresh/resume remain adjacent in configuration cell 3.
No Drive, manual token input or token printing. Automatic GitHub credentials are required
for durable backups; access available only in the Colab web UI cannot be read via VS Code.

Storage migration: legacy model JSON payloads -> tiny JSON pointers plus ZSTD Parquet.
Each Parquet row contains logical_path, payload_sha256, payload_bytes and payload_json
(original UTF-8 bytes). This is a lossless checkpoint container, not a normalized entity
table. Scores, repeated occurrences, original raw, invalid output and references survive
exactly. NER schema stays ner-selected-3.2; pointer/storage schemas are explicitly versioned.
Old JSON backups can be read; compact backups require v16 or the included reader.

Independent backups use Data/NER_checkpoints_v16/<run>. Legacy current-run backups
are removed from the active Git branch ONLY after the new complete snapshot is downloaded,
restored and every logical payload is validated. Other runs are untouched. No Git history
rewrite: deleting old branch entries does not reclaim historical Git objects.
Confirmed local ZIP copies are removed after complete-index publication. All packets
required by the current compact recovery chain remain on GitHub; indexes and dependencies
are not pruned merely because they are old. Cleanup failures are reported and preserve files.

Migration prints 10% progress and exact before/after sizes. It does not need another
whole uncompressed copy of the run. Restore verification intentionally needs space for
one additional COMPACT copy. Library/model compatibility guards remain enforced.


### 1 — התקנות

מתקין את אותן ספריות וגרסאות. פלט: בדיקת תלויות וזמן התקנה.


In [ ]:
import sys
import subprocess
import time
import json
import secrets
from contextlib import contextmanager

# Cell 1 starts a new execution; repeating other cells preserves their attempts.
SESSION_WALL_START = time.monotonic()
CELL_TIMINGS = []
TIMING_EXECUTION_ID = secrets.token_hex(12)
ACTIVE_TIMING_RUN_NAME = None
INSTALLATION_LOG = ""

def print_cell_timings():
    print("\nCELL TIMINGS — measured code blocks (logging/download bookkeeping excluded)")
    for row in CELL_TIMINGS:
        print(f"{row['title']} | {row['seconds']:.2f}s | {row['minutes']:.2f}min | {row['status']}")
    print(f"Total measured: {sum(row['seconds'] for row in CELL_TIMINGS):.2f}s")

@contextmanager
def cell(name):
    started = time.perf_counter()
    at_started = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    status, error = "success", None
    try:
        yield
    except BaseException as exc:
        status, error = "failed", f"{type(exc).__name__}: {exc}"
        raise
    finally:
        elapsed = time.perf_counter() - started
        CELL_TIMINGS.append({"attempt": len(CELL_TIMINGS)+1, "title": name,
            "execution_id": TIMING_EXECUTION_ID, "run_name": ACTIVE_TIMING_RUN_NAME,
            "seconds": elapsed, "minutes": elapsed/60, "status": status,
            "error": error, "at_started_utc": at_started,
            "at_finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})
        print(f"{'SUCCESS' if status == 'success' else 'FAILED'} {name}: "
              f"{error or 'completed'}; {elapsed:.2f}s ({elapsed/60:.2f}min)", flush=True)
        if status == "failed":
            print_cell_timings()

# Runs in a fresh Python process, so installed metadata is not cached by the kernel.
DEPENDENCY_CHECK_CODE = r"""
import json, sys
from importlib.metadata import distribution, PackageNotFoundError
from packaging.requirements import Requirement
from packaging.specifiers import SpecifierSet
from packaging.utils import canonicalize_name

def check_dependencies(roots, get_distribution=distribution):
    queue = [Requirement(value) for value in roots]
    visited, checked, errors = set(), set(), []
    while queue:
        req = queue.pop()
        name = canonicalize_name(req.name)
        try:
            dist = get_distribution(req.name)
        except PackageNotFoundError:
            errors.append(f"Missing dependency: {req}")
            continue
        if req.specifier and dist.version not in req.specifier:
            errors.append(f"{req}: installed {dist.version}")
        key = (name, tuple(sorted(req.extras)))
        if key in visited:
            continue
        visited.add(key)
        checked.add(name)
        requires_python = dist.metadata.get("Requires-Python")
        if requires_python and '.'.join(map(str, sys.version_info[:3])) not in SpecifierSet(requires_python):
            errors.append(f"{name} requires Python {requires_python}; actual {sys.version.split()[0]}")
        for value in dist.requires or []:
            child = Requirement(value)
            if child.marker is None or any(child.marker.evaluate({"extra": extra}) for extra in ["", *req.extras]):
                queue.append(child)
    return {"checked_packages": sorted(checked), "errors": sorted(set(errors))}

if __name__ == "__main__":
    report = check_dependencies(json.loads(sys.argv[1]))
    print(json.dumps(report))
    sys.exit(1 if report["errors"] else 0)
"""

with cell("1 — התקנות"):
    packages = ["ipywidgets>=8.1,<9", "torch>=2.2,<3", "tzdata>=2024.1", "transformers==4.49.0", "gliner2[local]==2.1.0",
        "huggingface-hub>=0.30,<1", "peft==0.14.0", "packaging>=24,<27",
        "pandas>=2.2,<3", "pyarrow>=18,<24", "jedi==0.19.2"]
    def installation_command(args):
        global INSTALLATION_LOG
        result = subprocess.run(args, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        INSTALLATION_LOG += f"\nCOMMAND: {args}\n{result.stdout}\n"
        if result.stdout:
            print(result.stdout, flush=True)
        if result.returncode:
            raise RuntimeError(f"Installation command failed ({result.returncode}): {args}\n{result.stdout}")
    print("Python:", sys.version)
    installation_command([sys.executable, "-m", "pip", "install", "--quiet", *packages])
    full_check = subprocess.run([sys.executable, "-m", "pip", "check"], text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    INSTALLATION_LOG += "\nGLOBAL PIP CHECK:\n" + full_check.stdout
    print("GLOBAL PIP CHECK:\n" + full_check.stdout, flush=True)
    if full_check.returncode not in [0, 1]:
        raise RuntimeError(f"pip check itself failed: {full_check.stdout}")
    relevant = subprocess.run([sys.executable, "-c", DEPENDENCY_CHECK_CODE,
        json.dumps(packages)], text=True,
        stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    INSTALLATION_LOG += "\nNOTEBOOK DEPENDENCY CHECK:\n" + relevant.stdout + relevant.stderr
    if relevant.returncode:
        raise RuntimeError("Notebook dependency conflict. Full details:\n" + relevant.stdout + relevant.stderr)
    DEPENDENCY_REPORT = {"global_pip_check_returncode": full_check.returncode,
        "global_pip_check_output": full_check.stdout, "notebook_dependencies": json.loads(relevant.stdout)}
    if full_check.returncode:
        print("WARNING: global conflicts were reported, but the notebook dependency closure passed. "
              "These diagnostics will be retained; model imports are checked in cell 2.")


### 2 — ספריות וסביבת הריצה

טוען ספריות, מזהה Colab/VS Code ומציג GPU וגרסאות. לא מבקש גישה ל־Drive.


In [ ]:
with cell("2 — imports and environment"):
    import os, copy, inspect, shutil
    import base64, urllib.request, urllib.error, urllib.parse
    import gc, hashlib, io, logging, math, platform, random, re, tempfile, zipfile
    from collections import Counter, defaultdict, deque, OrderedDict
    from itertools import islice
    from pathlib import PurePosixPath
    from datetime import datetime, timezone
    from importlib.metadata import version
    from pathlib import Path
    from zoneinfo import ZoneInfo
    import ipywidgets as widgets
    from IPython.display import display
    import pandas as pd
    import pyarrow as pa
    import pyarrow.parquet as pq
    import torch
    from gliner2 import AutoExtractor
    from huggingface_hub import model_info, snapshot_download
    from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
    VERSIONS = {n: version(n) for n in ["torch", "transformers", "gliner2", "peft",
        "huggingface-hub", "pandas", "pyarrow", "numpy", "tokenizers"]}
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    log = logging.getLogger("georag_ner_v15")
    print(json.dumps(VERSIONS, indent=2))
    print("Python:", platform.python_version(), "CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
    if torch.cuda.is_available():
        print("GPU free/total bytes:", torch.cuda.mem_get_info())

    """Notebook runtime adapters. Embedded in the notebook, with no local imports."""
    def detect_notebook_runtime():
        import os
        import platform
        import sys
        vscode = any(os.environ.get(key) for key in (
            "VSCODE_PID", "VSCODE_CWD", "VSCODE_IPC_HOOK_CLI", "VSCODE_INJECTION"))
        vscode = vscode or os.environ.get("TERM_PROGRAM") == "vscode"
        shell_module = ""
        try:
            from IPython import get_ipython
            shell = get_ipython()
            if shell is not None:
                shell_module = type(shell).__module__
        except ImportError:
            pass
        colab_kernel = shell_module.startswith("google.colab") or bool(
            os.environ.get("COLAB_RELEASE_TAG") or os.environ.get("COLAB_BACKEND_VERSION"))
        # The kernel's host does not prove which frontend submitted this cell.
        return {"platform": platform.system(), "python": platform.python_version(),
            "python_executable": sys.executable, "colab_kernel": colab_kernel,
            "frontend_hint": "vscode" if vscode else "unknown",
            "colab_frontend_confirmed": False}


    def default_output_root():
        runtime = globals().get("RUNTIME_INFO", {})
        if runtime.get("colab_kernel") and Path("/content").is_dir():
            return Path("/content/GeoRAG_NER")
        # Keep default Windows output paths short even when the project path is long.
        return (Path.home() if os.name == "nt" else Path.cwd()) / "GeoRAG_NER"


    def read_github_token(_prompt=None):
        """Reuse existing credentials only. Never ask for, print or serialize a token."""
        import os
        import subprocess
        secret_name = globals().get("GITHUB_TOKEN_SECRET", "COLAB")
        cache = globals().setdefault("_GITHUB_TOKEN_RUNTIME", {})

        def remember(value, source):
            if not isinstance(value, str) or not value.strip():
                return None
            cache[secret_name] = value.strip()
            globals()["_GITHUB_AUTH_SOURCE"] = source
            return value.strip()

        value = cache.get(secret_name)
        if isinstance(value, str) and value.strip():
            return value.strip()
        for name in dict.fromkeys([secret_name, "GITHUB_TOKEN", "GH_TOKEN"]):
            value = remember(os.environ.get(name), "environment")
            if value:
                return value
        command_env = os.environ.copy()
        command_env.update({"GIT_TERMINAL_PROMPT": "0", "GCM_INTERACTIVE": "0",
            "GIT_ASKPASS": "", "SSH_ASKPASS": "", "GH_PROMPT_DISABLED": "1",
            "GIT_TRACE": "0", "GIT_CURL_VERBOSE": "0", "GCM_TRACE": "0"})
        try:
            result = subprocess.run(["gh", "auth", "token", "--hostname", "github.com"],
                capture_output=True, text=True, timeout=15, env=command_env)
            if result.returncode == 0:
                value = remember(result.stdout, "github_cli")
                if value:
                    return value
        except (OSError, subprocess.TimeoutExpired):
            pass  # Optional credential source is unavailable; try the next source.
        try:
            result = subprocess.run(["git", "-c", "credential.interactive=false",
                "-c", "core.askPass=", "credential", "fill"],
                input="protocol=https\nhost=github.com\n\n", capture_output=True,
                text=True, timeout=15, env=command_env)
            if result.returncode == 0:
                fields = dict(line.split("=", 1) for line in result.stdout.splitlines() if "=" in line)
                value = remember(fields.get("password"), "git_credential_helper")
                if value:
                    return value
        except (OSError, subprocess.TimeoutExpired):
            pass
        runtime = globals().get("RUNTIME_INFO", {})
        reason = "No saved credential is accessible to this Python kernel."
        if runtime.get("colab_kernel") and runtime.get("frontend_hint") != "vscode":
            try:
                from google.colab import userdata
            except ImportError:
                reason = "Colab Secrets is unavailable in this kernel."
            else:
                # One bounded request, not repeated retries against an absent frontend.
                try:
                    value = userdata.get(secret_name)
                except userdata.TimeoutException:
                    runtime["colab_frontend_confirmed"] = False
                    runtime["colab_bridge_unavailable"] = True
                    reason = "Colab Secrets did not respond; this kernel may be connected through VS Code."
                except userdata.SecretNotFoundError:
                    reason = f"Colab secret {secret_name!r} does not exist."
                except userdata.NotebookAccessError:
                    reason = f"Enable notebook access for the existing Colab secret {secret_name!r}."
                else:
                    runtime["colab_frontend_confirmed"] = True
                    value = remember(value, "colab_secrets")
                    if value:
                        return value
                    reason = f"Colab secret {secret_name!r} is empty."
        raise RuntimeError(reason + " Automatic sources: runtime memory, environment, GitHub CLI, "
            "Git credential helper, and Colab Secrets when its web frontend is reachable. "
            "A VS Code extension or a Colab website secret is not automatically shared with a remote kernel. "
            "No manual token entry is requested. Existing output files are unchanged.") from None


    def read_colab_github_token(_prompt=None):
        # Keep the existing publication function's interface backward compatible.
        return read_github_token(_prompt)


    def deliver_run_archive(path):
        """Request a browser transfer only when the Colab frontend responds."""
        from pathlib import Path
        path = Path(path).resolve()
        if not path.is_file():
            raise FileNotFoundError(path)
        runtime = globals().get("RUNTIME_INFO", {})
        frontend = runtime.get("colab_frontend_confirmed", False)
        if runtime.get("colab_kernel") and runtime.get("frontend_hint") != "vscode" and not frontend:
            try:
                from google.colab import output
                frontend = bool(output.eval_js(
                    "typeof google !== 'undefined' && !!google.colab && !!google.colab.kernel",
                    timeout_sec=2))
            except ImportError:
                frontend = False
            except Exception as exc:
                # ZIP is already verified. A browser transfer failure must not invalidate it.
                print("Colab browser transfer unavailable:", type(exc).__name__)
                frontend = False
        if frontend:
            try:
                from google.colab import files
                files.download(str(path))
            except Exception as exc:
                print("Browser download could not be requested:", type(exc).__name__)
            else:
                print("Colab browser download requested; verified ZIP:", path)
                return "browser_download_requested"
        print("Verified ZIP saved in the Python kernel filesystem:", path)
        print("In VS Code use Explorer to open/save this file. For a remote kernel, download it from its host.")
        return "saved_in_kernel_filesystem"
    RUNTIME_INFO = detect_notebook_runtime()
    print("Runtime:", json.dumps(RUNTIME_INFO, ensure_ascii=False))
    print("Automatic output root:", default_output_root())

    """Resource probes and bounded browser transfers; no keep-alive or external storage."""
    class SafeCheckpointStop(RuntimeError):
        pass


    def resource_snapshot():
        result={'at_processed':datetime.now(timezone.utc).isoformat(),'ram_available_gb':None,
            'disk_free_gb':None,'gpu_free_gb':None,'gpu_name':None,
            'session_hours':(time.monotonic()-SESSION_WALL_START)/3600}
        info=Path('/proc/meminfo')
        if info.is_file():
            values=dict(line.split(':',1) for line in info.read_text().splitlines())
            if 'MemAvailable' in values:
                result['ram_available_gb']=int(values['MemAvailable'].split()[0])*1024/1024**3
        maximum,current=Path('/sys/fs/cgroup/memory.max'),Path('/sys/fs/cgroup/memory.current')
        if maximum.is_file() and current.is_file():
            cap=maximum.read_text().strip()
            if cap.isdigit():
                free=max(0,int(cap)-int(current.read_text().strip()))/1024**3
                old=result['ram_available_gb']
                result['ram_available_gb']=free if old is None else min(old,free)
        target=Path(globals().get('OUTPUT_ROOT') or Path.cwd()).expanduser().resolve()
        while not target.exists() and target!=target.parent:target=target.parent
        result['disk_free_gb']=shutil.disk_usage(target).free/1024**3
        if torch.cuda.is_available():
            result['gpu_name']=torch.cuda.get_device_name(0)
            free,total=torch.cuda.mem_get_info()
            result.update(gpu_free_gb=free/1024**3,gpu_total_gb=total/1024**3)
        return result


    def initial_resource_check():
        snapshot=resource_snapshot()
        print('Colab resource check:',json.dumps(snapshot))
        if REQUIRE_GPU and not torch.cuda.is_available():
            raise RuntimeError('Select GPU in Colab Runtime > Change runtime type. TPU/CPU is not the CUDA execution path for this notebook.')
        ram=snapshot['ram_available_gb']
        if ram is not None and ram<MIN_INITIAL_RAM_GB:
            raise RuntimeError(f'Only {ram:.2f} GiB RAM available; need {MIN_INITIAL_RAM_GB} before loading the corpus/models.')
        if snapshot['disk_free_gb']<DISK_RESERVE_GB:
            raise RuntimeError('Insufficient free disk for checkpoints, exports and backup packets')
        return snapshot


    def request_checkpoint_download(path):
        """Transfer small blocks via eval_js even while the inference cell is running.

        A browser download request is not proof that the browser saved the file.
        """
        import base64
        path=Path(path)
        if not AUTO_DOWNLOAD_BACKUPS:
            return {'status':'server_file_only','bytes_sent':0}
        if not RUNTIME_INFO.get('colab_kernel'):
            if REQUIRE_BROWSER_BACKUP:
                raise RuntimeError('Browser checkpoint backups require the Colab web interface. Use local persistent storage explicitly or run from Colab.')
            return {'status':'kernel_filesystem_only','bytes_sent':0}
        from google.colab import output
        key='__georag_checkpoint_'+secrets.token_hex(12)
        encoded_key=json.dumps(key)
        sent=0
        try:
            output.eval_js(f'window[{encoded_key}]={{chunks:[],received:0}}; true',timeout_sec=30)
            with path.open('rb') as handle:
                for block in iter(lambda:handle.read(1024*1024),b''):
                    encoded=json.dumps(base64.b64encode(block).decode('ascii'))
                    script='''(()=>{const q=window[KEY];const s=atob(DATA);
                        const a=new Uint8Array(s.length);for(let i=0;i<s.length;i++)a[i]=s.charCodeAt(i);
                        q.chunks.push(a);q.received+=a.length;return q.received;})()'''
                    received=output.eval_js(script.replace('KEY',encoded_key).replace('DATA',encoded),timeout_sec=30)
                    sent+=len(block)
                    if received!=sent:raise RuntimeError('Checkpoint browser transfer byte count mismatch')
            script='''(()=>{const q=window[KEY];const blob=new Blob(q.chunks,{type:'application/zip'});
                const url=URL.createObjectURL(blob);const a=document.createElement('a');
                a.href=url;a.download=NAME;document.body.appendChild(a);a.click();a.remove();
                const bytes=blob.size;q.chunks=[];delete window[KEY];
                setTimeout(()=>URL.revokeObjectURL(url),60000);return {bytes:bytes,requested:true};})()'''
            reply=output.eval_js(script.replace('KEY',encoded_key).replace('NAME',json.dumps(path.name)),timeout_sec=30)
            if not reply or reply.get('bytes')!=path.stat().st_size:
                raise RuntimeError('Checkpoint browser handoff was not acknowledged')
            print('CHECKPOINT DOWNLOAD REQUESTED:',path.name,'— verify it was saved on your computer.',flush=True)
            return {'status':'browser_download_requested','bytes_sent':sent,
                'disk_save_confirmed':False}
        except Exception as exc:
            print('Checkpoint browser transfer failed:',type(exc).__name__,str(exc))
            raise RuntimeError(f'Checkpoint ZIP remains on the server: {path}. Restore the browser connection and retry pending downloads before continuing.') from exc
        finally:
            try:output.eval_js(f'delete window[{encoded_key}]; true',timeout_sec=2)
            except Exception:pass  # Only remove this temporary browser buffer; transfer errors already propagate.


### 3 — הגדרות מקור, מודלים וגיבוי

מגדיר את כל הפרמטרים. פלט: מזהה ריצה, ספים, משאבים ותדירות הגיבוי.


In [ ]:
with cell("3 — configuration"):
    # Change only this value: "fresh", "delta", or "resume".
    RUN_MODE = "delta"
    # Used ONLY in resume mode; ignored by fresh/delta.
    RESUME_RUN_NAME = "georag_selected_v2_full_n736275_r985292856_20261026"

    # fresh: latest input, new run, infer all eligible unique texts.
    # delta: latest input, new run, reuse compatible predictions from previous output pointer.
    # resume: continue the named run with its frozen input/configuration.
    if RUN_MODE in {"fresh", "delta"}:
        GITHUB_RESUME_RUN_NAME = None
        RESUME_RUN_DIR = None
        DELTA_REUSE_ENABLED = RUN_MODE == "delta"
        PREFER_MATCHING_LOCAL_RESUME = False
        PROCESS_ALL = True
        RESTORE_FROM_BACKUPS = False
    elif RUN_MODE == "resume":
        GITHUB_RESUME_RUN_NAME = RESUME_RUN_NAME
        RESUME_RUN_DIR = None
        DELTA_REUSE_ENABLED = True
        PREFER_MATCHING_LOCAL_RESUME = True
        PROCESS_ALL = True
        RESTORE_FROM_BACKUPS = False
    else:
        raise ValueError('RUN_MODE must be "fresh", "delta", or "resume"')

    print("Requested execution mode:", RUN_MODE)

    REPO_URL = "https://github.com/danielbr9382/Corpus2GeoRAG_Pacific.git"
    INPUT_SOURCE = "github"  # pinned Parquet, no local upload or Drive
    SOURCE_RELATIVE_PATH = "Data/corpus_paragraphs.parquet"
    SOURCE_REF = "main"  # resolve latest once per NEW run; resume uses the saved snapshot
    SOURCE_COMMIT = None  # populated automatically; no manual commit updates
    GITHUB_REPOSITORY = "danielbr9382/Corpus2GeoRAG_Pacific"
    GITHUB_CHECKPOINT_ROOT = "Data/NER_checkpoints"
    GITHUB_CHECKPOINT_BRANCH = "main"
    PREPARATION_PART_ROWS = 5000
    PREPARATION_BACKUP_EVERY_PARTS = 10
    RECONSTRUCTION_CACHE_ROWS = 256
    EQUIVALENCE_SEED = 17
    CACHE_ENABLED = True
    CACHE_LRU_SHARDS = 2
    TEXT_PREPROCESSING = {"version": "wiki-mask-1.0", "enabled": True,
        "mask_urls": True, "mask_comments": True, "mask_html_tags": True,
        "mask_wiki_syntax": True, "decode_html_entities": True, "skip_nonlinguistic": True}
    CONFLICT_POLICY = "quarantine_exact_span"
    CONFLICT_DIAGNOSTIC_MIN_SCORE = 0.50  # diagnostics only; does not override selection thresholds
    MODELS = {
        "bert": {"id": "dslim/bert-base-NER", "revision": "d1a3e8f13f8c3566299d95fcfc9a8d2382a9affc"},
        "gliner": {"id": "fastino/gliner2.5-base-v1", "revision": "54785d51df8d86a0d8f2eb212fec15871aad2c4d"},
    }
    GITHUB_UPLOAD_ENABLED = True
    GITHUB_OUTPUT_DIR = "Data"
    GITHUB_FINAL_FILENAME = "georag_selected_v2.parquet"
    ALLOW_SAMPLE_PUBLICATION = False
    DELTA_BASE_RUN_NAME = None  # None = previous run from the fixed .run.json pointer
    DELTA_BASE_RUN_DIR = None  # optional existing LOCAL v10/v11 run instead of GitHub
    DELTA_REUSE_ERRORS = False
    GITHUB_UPLOAD_BRANCH = None  # None = repository default branch
    GITHUB_TOKEN_SECRET = "COLAB"
    SEED = 17
    SAMPLE_SIZE = 1000  # paragraphs, not articles
    ENABLE_GLINER_SAFE_DOT_REPAIR = True  # exact trailing-dot proof only; raw remains unchanged
    RUN_RANDOM_ID = None  # None generates a new ID; does not affect sampling
    OUTPUT_ROOT = None  # automatic; optionally set a local/remote kernel Path
    BATCH_ENABLED = True
    BATCH_SIZES = {"bert": 8, "gliner": 2}
    BATCH_SCORE_ATOL = 0.0001
    PREFLIGHT_PARAGRAPHS = 32
    PREFLIGHT_MAX_PARAGRAPH_CHARS = 20000
    REQUIRE_GPU = True
    MIN_INITIAL_RAM_GB = 4.0
    RAM_RESERVE_GB = 1.5
    DISK_RESERVE_GB = 10.0
    MAX_SESSION_HOURS = None  # no artificial time stop; Colab may still terminate the runtime
    ENABLE_CHECKPOINT_BACKUPS = True
    AUTO_DOWNLOAD_BACKUPS = False  # GitHub delivery replaces browser checkpoint downloads
    REQUIRE_BROWSER_BACKUP = False
    BACKUP_INTERVAL_MINUTES = 30
    # ZIP target now uses compressed bytes; no removal of raw data.
    RELEASE_CONFIRMED_COMPRESSION_CACHE = True  # keep raw/source/ZIP; free only derived stage after confirmed index
    BACKUP_RAW_PACKET_MIB = 384  # bounded unpacked data, below the 512 MiB restore guard
    GITHUB_MIN_WRITE_INTERVAL_SECONDS = 2.0
    GITHUB_MAX_WRITES_PER_MINUTE = 30
    GITHUB_MAX_WRITES_PER_HOUR = 240
    GITHUB_RETRY_ATTEMPTS = 8
    GITHUB_RETRY_BASE_SECONDS = 60.0
    GITHUB_RETRY_BACKOFF_CAP_SECONDS = 1800.0
    GITHUB_RETRY_MAX_WAIT_SECONDS = 7200.0
    BACKUP_PACKET_MIB = 20
    BACKUP_SEGMENT_MIB = 8
    TOKEN_COUNT_CACHE_SIZE = 256
    RESTORE_BACKUP_DIR = Path("/content/restore_packets")
    CHECKPOINT_PARAGRAPHS = 100
    RESELECT_V1_ZIP = None  # optional: existing v1 ZIP, selection only; no inference
    FINAL_OUTPUT_PREFIX = "georag_selected_v2"
    ADD_BERT_PER = True
    BERT_PER_MIN_SCORE = 0.90
    EVENT_HIGH_PRIORITY_SCORE = 0.90
    GLINER_THRESHOLD = 0.01  # decoding floor, independent of selection thresholds
    GLINER_LABELS = ["person", "organization", "military unit", "location",
        "geopolitical entity", "facility", "nationality or political group", "event",
        "date", "time", "weapon", "vehicle", "product", "work of art", "law",
        "language", "money", "quantity", "percent", "ordinal", "cardinal",
        "miscellaneous named entity"]
    LABEL_MAP = {
        "PER": "PER", "LOC": "LOC", "ORG": "ORG", "MISC": "MISC",
        "person": "PER", "organization": "ORG", "military unit": "ORG",
        "location": "LOC", "geopolitical entity": "LOC", "facility": "LOC",
        "event": "EVENT", "date": "DATE", "time": "TIME",
        **{k: "MISC" for k in ["nationality or political group", "weapon", "vehicle",
            "product", "work of art", "law", "language", "money", "quantity", "percent",
            "ordinal", "cardinal", "miscellaneous named entity"]},
    }
    SELECTION_THRESHOLDS = {"PER": 0.50, "DATE": 0.50, "LOC": 0.0, "ORG": 0.0, "MISC": 0.0}
    # Initial v2 rule, not final calibration for the full corpus.
    PRIMARY = {"PER": "gliner", "LOC": "bert", "ORG": "bert", "DATE": "gliner",
               "TIME": "gliner", "EVENT": "gliner", "MISC": "bert"}
    WINDOWS = {n: {"max_encoded_tokens": 480, "max_words": 384, "overlap_words": 64}
               for n in MODELS}
    for window_settings in WINDOWS.values():
        window_settings["long_unit_overlap_chars"] = 64
    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
    SCHEMA_VERSION = "ner-selected-3.2"
    SPLIT_VERSION = "nonempty_nonheading_lines_v1"  # unchanged from v4
    assert isinstance(SEED, int) and isinstance(SAMPLE_SIZE, int) and SAMPLE_SIZE > 0
    assert re.fullmatch(r"[A-Za-z0-9_-]+", FINAL_OUTPUT_PREFIX), "Use letters, digits, _ or -"
    assert all(0 <= t <= 1 for t in [BERT_PER_MIN_SCORE, EVENT_HIGH_PRIORITY_SCORE, GLINER_THRESHOLD])
    assert all(0 <= w["overlap_words"] < w["max_words"] for w in WINDOWS.values())
    assert all(re.fullmatch(r"[0-9a-f]{40}", m["revision"]) for m in MODELS.values())
    if RUN_RANDOM_ID is None:
        RUN_RANDOM_ID = secrets.randbelow(900_000_000)+100_000_000
    assert isinstance(RUN_RANDOM_ID, int) and RUN_RANDOM_ID >= 0
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    assert isinstance(CHECKPOINT_PARAGRAPHS, int) and CHECKPOINT_PARAGRAPHS > 0
    assert all(0 <= x <= 1 for x in SELECTION_THRESHOLDS.values())
    OUTPUT_ROOT = default_output_root() if OUTPUT_ROOT is None else Path(OUTPUT_ROOT).expanduser().resolve()
    assert INPUT_SOURCE in {"local", "github"}
    assert all(isinstance(v,int) and v>0 for v in BATCH_SIZES.values())
    assert BATCH_SCORE_ATOL>=0 and PREFLIGHT_PARAGRAPHS>0
    assert 0<BACKUP_SEGMENT_MIB<=BACKUP_PACKET_MIB<=256
    assert BACKUP_INTERVAL_MINUTES>0 and (MAX_SESSION_HOURS is None or MAX_SESSION_HOURS>0) and TOKEN_COUNT_CACHE_SIZE>0
    assert CACHE_LRU_SHARDS >= 1 and CONFLICT_POLICY == "quarantine_exact_span"
    assert 0 <= CONFLICT_DIAGNOSTIC_MIN_SCORE <= 1
    assert PREPARATION_PART_ROWS > 0 and PREPARATION_BACKUP_EVERY_PARTS > 0 and RECONSTRUCTION_CACHE_ROWS > 0
    assert 1 <= BACKUP_RAW_PACKET_MIB <= 480
    assert 0 < BACKUP_SEGMENT_MIB <= BACKUP_RAW_PACKET_MIB
    assert 0 < BACKUP_PACKET_MIB <= 20
    assert GITHUB_MIN_WRITE_INTERVAL_SECONDS >= 1
    assert 1 <= GITHUB_MAX_WRITES_PER_MINUTE <= 60 and 1 <= GITHUB_MAX_WRITES_PER_HOUR <= 400
    assert GITHUB_RETRY_ATTEMPTS >= 1 and GITHUB_RETRY_BASE_SECONDS >= 60
    assert GITHUB_RETRY_BACKOFF_CAP_SECONDS >= GITHUB_RETRY_BASE_SECONDS and GITHUB_RETRY_MAX_WAIT_SECONDS > 0
    CONFIG = {
        "alignment_validation": {"schema_version":"georag-span-alignment-1.0",
            "rule":"one_extra_terminal_dot_exact_source_slice",
            "enabled":ENABLE_GLINER_SAFE_DOT_REPAIR,"raw_preserved":True},
        "checkpoint_efficiency": {"schema_version": "georag-checkpoint-upload-plan-2.0",
            "compression": "single_deflate_stage_standard_zip", "all_raw_and_occurrences_retained": True,
            "upload_verification_cache": "kernel_local_sha_size_signature",
            "release_confirmed_compression_cache": RELEASE_CONFIRMED_COMPRESSION_CACHE},
        "checkpoint_transport": {"schema_version": "georag-checkpoint-upload-plan-2.0",
            "packing": "compressed_size_with_unpacked_guard", "raw_packet_mib": BACKUP_RAW_PACKET_MIB,
            "min_write_interval_seconds": GITHUB_MIN_WRITE_INTERVAL_SECONDS,
            "max_writes_per_minute": GITHUB_MAX_WRITES_PER_MINUTE, "max_writes_per_hour": GITHUB_MAX_WRITES_PER_HOUR,
            "retry_attempts": GITHUB_RETRY_ATTEMPTS, "retry_base_seconds": GITHUB_RETRY_BASE_SECONDS,
            "retry_backoff_cap_seconds": GITHUB_RETRY_BACKOFF_CAP_SECONDS,
            "retry_max_wait_seconds": GITHUB_RETRY_MAX_WAIT_SECONDS},
        "preparation": {"schema_version": "georag-prepared-input-1.0", "part_rows": PREPARATION_PART_ROWS,
            "backup_every_parts": PREPARATION_BACKUP_EVERY_PARTS, "equivalence_seed": EQUIVALENCE_SEED,
            "reconstruction_cache_rows": RECONSTRUCTION_CACHE_ROWS},
        "source_ref": SOURCE_REF,
        "fixed_publication": {"filename": GITHUB_FINAL_FILENAME, "allow_sample": ALLOW_SAMPLE_PUBLICATION},
        "delta_reuse": {"enabled": DELTA_REUSE_ENABLED, "base_run_name": DELTA_BASE_RUN_NAME,
            "base_run_dir": str(DELTA_BASE_RUN_DIR) if DELTA_BASE_RUN_DIR else None, "reuse_errors": DELTA_REUSE_ERRORS},
        "text_preprocessing": TEXT_PREPROCESSING,
        "prediction_cache": {"enabled": CACHE_ENABLED, "version": "prediction-reuse-1.0", "lru_shards": CACHE_LRU_SHARDS},
        "conflicts": {"policy": CONFLICT_POLICY, "diagnostic_min_score": CONFLICT_DIAGNOSTIC_MIN_SCORE},
        "github_checkpoint_publication": {"repository": GITHUB_REPOSITORY,
            "branch": GITHUB_CHECKPOINT_BRANCH, "root": GITHUB_CHECKPOINT_ROOT,
            "enabled": ENABLE_CHECKPOINT_BACKUPS},
        "resilience": {"require_gpu":REQUIRE_GPU,"initial_ram_gb":MIN_INITIAL_RAM_GB,
            "ram_reserve_gb":RAM_RESERVE_GB,"disk_reserve_gb":DISK_RESERVE_GB,
            "max_session_hours":MAX_SESSION_HOURS,"backups_enabled":ENABLE_CHECKPOINT_BACKUPS,
            "auto_download":AUTO_DOWNLOAD_BACKUPS,"backup_interval_minutes":BACKUP_INTERVAL_MINUTES,
            "backup_packet_mib":BACKUP_PACKET_MIB,"backup_segment_mib":BACKUP_SEGMENT_MIB,
            "token_count_cache_size":TOKEN_COUNT_CACHE_SIZE},
        "batch_execution": {"enabled":BATCH_ENABLED,"sizes":BATCH_SIZES,
            "score_atol":BATCH_SCORE_ATOL,"preflight_paragraphs":PREFLIGHT_PARAGRAPHS,
            "preflight_max_paragraph_chars":PREFLIGHT_MAX_PARAGRAPH_CHARS},
        "input_source": INPUT_SOURCE,
        "table_output_formats": ["parquet"],
        "diagnostic_output_format": "json",
        "github_publication": {"enabled": GITHUB_UPLOAD_ENABLED,
            "folder": GITHUB_OUTPUT_DIR, "branch": GITHUB_UPLOAD_BRANCH},"schema_version": SCHEMA_VERSION, "split_version": SPLIT_VERSION,
        "selection_thresholds": SELECTION_THRESHOLDS, "checkpoint_paragraphs": CHECKPOINT_PARAGRAPHS,
        "repo_url": REPO_URL, "source_path": SOURCE_RELATIVE_PATH, "source_commit": SOURCE_COMMIT,
        "models": MODELS, "seed": SEED, "sample_size_requested": SAMPLE_SIZE,
        "process_all": PROCESS_ALL, "run_random_id": RUN_RANDOM_ID,
        "output_root": str(OUTPUT_ROOT), "final_output_prefix": FINAL_OUTPUT_PREFIX,
        "add_bert_per": ADD_BERT_PER, "bert_per_min_score": BERT_PER_MIN_SCORE,
        "event_high_priority_score": EVENT_HIGH_PRIORITY_SCORE, "gliner_threshold": GLINER_THRESHOLD,
        "gliner_labels": GLINER_LABELS, "label_map": LABEL_MAP, "primary_v2": PRIMARY,
        "windows": WINDOWS, "device": DEVICE, "aggregation_strategy": "average",
        "window_dedup": "same model/label/exact span; first window wins; preserve all raw",
        "selection_dedup": "same paragraph/group/exact span; no score averaging",
        "semantic_validation": "not performed; EVENT/TIME are candidates only"}
    MODEL_OUTPUTS = {}
    print("Run ID:", RUN_RANDOM_ID, "device:", DEVICE, "policy:", PRIMARY)

    _GITHUB_TOKEN_RUNTIME = globals().get("_GITHUB_TOKEN_RUNTIME", {})
    GITHUB_UPLOAD_RESULT = {"status": "not_attempted" if GITHUB_UPLOAD_ENABLED else "disabled"}
    print("GitHub upload:", "will be attempted after validated exports; failure will not stop ZIP creation"
          if GITHUB_UPLOAD_ENABLED else "disabled")
    print("Output root:", OUTPUT_ROOT)

    print("GeoRAG v16 compact storage | batch enabled:",BATCH_ENABLED,"| requested sizes:",BATCH_SIZES,"| score tolerance:",BATCH_SCORE_ATOL)

    COMPACT_PRUNE_LEGACY_REMOTE = True  # current run only, AFTER independent compact restore
    COMPACT_CLEAN_LOCAL_CONFIRMED = True  # verified ZIP copies only
    CONFIG["compact_storage"] = {"schema_version":"georag-lossless-checkpoint-parquet-1.0",
        "codec":"zstd", "level":3, "logical_json_bytes_preserved":True,
        "prune_legacy_current_run":COMPACT_PRUNE_LEGACY_REMOTE,
        "clean_confirmed_local_zips":COMPACT_CLEAN_LOCAL_CONFIRMED}
    # Bootstrap may need to recover a VM stopped just below the ordinary disk reserve.
    # This exception is limited to initial checks; normal guards resume after migration.
    _normal_reserve = DISK_RESERVE_GB
    try:
        if RUN_MODE == "resume":
            DISK_RESERVE_GB = min(DISK_RESERVE_GB, 0.5)
        INITIAL_RESOURCES = initial_resource_check()
    finally:
        DISK_RESERVE_GB = _normal_reserve

    print("Remote checkpoints (v16):", GITHUB_CHECKPOINT_ROOT + "_v16", "every", BACKUP_INTERVAL_MINUTES, "minutes and at model boundaries")


### 4 — חלונות, היסטים ופרוטוקול חידוש

מגדיר את אותם adapters וחלונות, קורא Parquet ומוסיף העלאה ושחזור של חבילות.


In [ ]:
with cell('4 — source, offsets and adapters'):
    def utc_now():
        return datetime.now(timezone.utc).isoformat()

    def digest(payload):
        return hashlib.sha256(payload).hexdigest()

    def digest_file(path):
        hasher = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                hasher.update(block)
        return hasher.hexdigest()

    def jsonable(value):
        if hasattr(value, "item"):
            return value.item()
        raise TypeError(type(value).__name__)

    def encoded(obj):
        return json.dumps(obj, ensure_ascii=False, sort_keys=True, allow_nan=False,
                          default=jsonable).encode("utf-8")

    def save_json(path, obj):
        with Path(path).open("xb") as handle:
            handle.write(encoded(obj))

    def load_json(path):
        return json.loads(Path(path).read_bytes())

    def progress(name, done, total, errors, started, last):
        bucket = done * 10 // total
        for mark in range(last+1, bucket+1):
            print(f"{name}: {mark*10}% reached; processed={done}/{total}; "
                  f"success={done-errors}; failed={errors}; elapsed={time.perf_counter()-started:.2f}s", flush=True)
        return bucket

    def text_units(title, article):
        offset, index = 0, 0
        for line in article.splitlines(keepends=True):
            text = line.rstrip("\r\n")  # שומרים רווחים ותווים מקוריים
            if text.strip() and not re.fullmatch(r"=+\s*.*?\s*=+", text.strip()):
                end = offset + len(text)
                assert article[offset:end] == text
                yield {"Source": title, "Index": index, "Text": text,
                       "start_document": offset, "end_document": end}
                index += 1
            offset += len(line)

    def windows(text, settings, measure, limit):
        words = list(re.finditer(r"\S+", text))
        i = 0
        budget = min(settings["max_encoded_tokens"], limit)
        while i < len(words):
            left = 0 if i == 0 else words[i].start()
            high = min(len(words), i + settings["max_words"])
            def right(j):
                return words[j].start() if j < len(words) else len(text)
            # הפחתה עד שהקידוד המלא נכנס; לא מפעילים truncation
            j = high
            while j > i and measure(text[left:right(j)]) > budget:
                j -= max(1, (j-i)//8)
            if j == i:
                # A URL/encoded identifier can exceed the limit without containing spaces.
                # Keep every source character: split only this unit, with overlap and
                # a real tokenizer-length check for every emitted character window.
                stop = right(i + 1)
                cursor = left
                overlap_chars = settings["long_unit_overlap_chars"]
                while cursor < stop:
                    end = min(stop, cursor + max(1, budget*4))
                    while end > cursor and measure(text[cursor:end]) > budget:
                        end -= max(1, (end-cursor)//8)
                    if end == cursor:
                        raise ValueError(f"One source character at {cursor} cannot fit encoded token budget {budget}")
                    count = measure(text[cursor:end])
                    assert count <= budget
                    yield {"start": cursor, "end": end, "Text": text[cursor:end], "encoded_tokens": count}
                    if end == stop:
                        break
                    # Bound the overlap to a quarter of this window so progress is guaranteed.
                    overlap = min(overlap_chars, (end-cursor)//4)
                    cursor = max(cursor+1, end-overlap)
                i += 1
                continue
            end = right(j)
            count = measure(text[left:end])
            assert count <= budget
            yield {"start": left, "end": end, "Text": text[left:end], "encoded_tokens": count}
            if j == len(words):
                break
            i = max(i + 1, j - settings["overlap_words"])

    def group(label):
        return LABEL_MAP.get(re.sub(r"^[BI]-", "", str(label)))

    def mention(p, name, prediction, base, ref):
        result = {"model": name, "model_id": MODELS[name]["id"],
            "Source": p["Source"], "Index": p["Index"], "sample_position": p["sample_position"],
            "Text": p["Text"], "corpus_position": p.get("corpus_position"), "source_sha256": SOURCE_META["sha256"],
            "source_commit": SOURCE_META["commit_sha"], "source_path": SOURCE_META["path"],
            "text_sha256": digest(p["Text"].encode()), "label_original": prediction.get("label"),
            "subtype": prediction.get("label"), "entity_group": group(prediction.get("label")),
            "score": None, "start": None, "end": None, "word": prediction.get("text"),
            "start_document": None, "end_document": None, "status_validation": "invalid",
            "error": None, "ref_output_raw": [ref], "prediction": prediction}
        try:
            score = prediction.get("score")
            if score is not None:
                score = float(score)
                if not math.isfinite(score) or not 0 <= score <= 1:
                    raise ValueError("Non-finite score")
            result["score"] = score
            if prediction.get("alignment_error"):
                raise ValueError(prediction["alignment_error"])
            a, b = prediction["start"], prediction["end"]
            if not isinstance(a, int) or not isinstance(b, int) or isinstance(a, bool) or isinstance(b, bool):
                raise ValueError("Offsets must be integers")
            a, b = base + a, base + b
            result.update(start=a, end=b)
            if not 0 <= a < b <= len(p["Text"]):
                raise ValueError("Span outside paragraph")
            surface = p["Text"][a:b]
            # HF subword rendering is diagnostic only; other adapters verify model surface
            if prediction.get("surface_checked", True) and surface != prediction.get("text"):
                raise ValueError("Model surface differs from original source slice")
            da, db = p["start_document"] + a, p["start_document"] + b
            assert_document_span(p, da, db, surface)
            if not isinstance(prediction.get("label"), str) or not prediction["label"]:
                raise ValueError("Missing label")
            result.update(word=surface, score=score, start_document=da, end_document=db,
                          status_validation="valid" if result["entity_group"] else "pending")
            if result["entity_group"] is None:
                result["error"] = "Unmapped label; update LABEL_MAP explicitly"
            result["semantic_validation"] = "not_assessed"
        except Exception as exc:
            result["error"] = f"{type(exc).__name__}: {exc}"
        return result

    def deduplicate(items):
        output, seen = [], {}
        for item in items:
            key = (item["Source"], item["Index"], item["model"], item["start"],
                   item["end"], item["label_original"])
            if item["status_validation"] == "valid" and key in seen:
                seen[key]["ref_output_raw"].extend(item["ref_output_raw"])
            else:
                output.append(item)
                if item["status_validation"] == "valid":
                    seen[key] = item  # בחירה ראשונה לפי סדר חלונות; לא לפי ממוצע ציון
        return output

    def process_paragraph(p, name, predict, measure, limit):
        started, raw, found, errors = time.perf_counter(), [], [], []
        try:
            for wi, w in enumerate(windows(p["Text"], WINDOWS[name], measure, limit)):
                record = {"window_id": wi, **w, "at_processed": utc_now(), "output": None,
                          "error": None, "status_processing": "pending", "status_validation": "pending"}
                raw.append(record)
                try:
                    output = predict(w["Text"])
                    record["output"] = output  # שומרים לפני פירוש/ולידציה
                    output_errors = output.get("errors", []) if isinstance(output, dict) else []
                    errors.extend(output_errors)
                    ref = f"{ACTIVE_RAW_FILE}#/results/{ACTIVE_RAW_INDEX}/raw/{wi}/output"
                    predictions = adapt_output(name, output, w["Text"])
                    for ordinal, prediction in enumerate(predictions):
                        prediction = dict(prediction)
                        a, b = prediction.get("start"), prediction.get("end")
                        if isinstance(a, int) and isinstance(b, int) and not 0 <= a < b <= len(w["Text"]):
                            synthetic = (name == "gliner" and b == len(w["Text"])+1
                                and prediction.get("text", "").endswith("."))
                            prediction["alignment_error"] = (
                                "GLiNER synthetic trailing punctuation outside source window"
                                if synthetic else "Span outside source window")
                        item = mention(p, name, prediction, w["start"], ref)
                        item["raw_ordinal"] = ordinal
                        found.append(item)
                    record["status_processing"] = "error" if output_errors else "success"
                    record["error"] = "; ".join(output_errors) or None
                    record["status_validation"] = "invalid" if output_errors or any(
                        m["status_validation"] != "valid" for m in found if ref in m["ref_output_raw"]
                    ) else "valid"
                except Exception as exc:
                    record["error"] = f"{type(exc).__name__}: {exc}"
                    record["status_processing"] = "error"
                    record["status_validation"] = "invalid"
                    errors.append(record["error"])
                    log.error(json.dumps({"model": name, "sample_position": p["sample_position"],
                                          "window": wi, "error": record["error"]}))
        except Exception as exc:
            errors.append(f"{type(exc).__name__}: {exc}")
            log.error(json.dumps({"model": name, "sample_position": p["sample_position"], "error": errors[-1]}))
        before_dedup = len(found)
        found = deduplicate(found)
        invalid = [m for m in found if m["status_validation"] != "valid"]
        status = "error" if errors else "partial_invalid" if invalid else "success"
        return {"paragraph": p, "raw": raw, "mentions": found, "duplicates_resolved": before_dedup-len(found),
            "status_processing": status, "errors": errors + [m["error"] for m in invalid],
            "at_processed": utc_now(), "processing_seconds": time.perf_counter()-started}

    def release():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def tokenizer_measure(tokenizer):
        return lambda text: len(tokenizer(text, add_special_tokens=True, truncation=False, verbose=False)["input_ids"])


    def hf_revision(name):
        return model_info(MODELS[name]["id"], revision=MODELS[name]["revision"]).sha

    def adapt_output(name, output, text):
        if name == "bert":
            return [{"label": x["entity_group"], "start": int(x["start"]), "end": int(x["end"]),
                     "score": x.get("score"), "text": x.get("word"), "surface_checked": False}
                    for x in output]
        if name == "gliner":
            return [{"label": label, "start": x["start"], "end": x["end"],
                     "text": x["text"], "score": x.get("confidence")}
                    for label, values in output["entities"].items() for x in values]
        raise ValueError(f"No adapter registered: {name}")

    print("v4 source/window/offset helpers ready; raw API outputs retained, not full logits.")

    """Incremental, immutable, hash-chained checkpoint packets and streaming restore."""
    PACKET_SCHEMA='georag-checkpoint-packets-3.1'
    PACKET_MANIFEST='_CHECKPOINT_PACKET.json'


    def sha_file_stream(path):
        h=hashlib.sha256()
        with Path(path).open('rb') as handle:
            for block in iter(lambda:handle.read(1024*1024),b''):h.update(block)
        return h.hexdigest()


    def checkpoint_relative_path(name):
        if not isinstance(name,str) or '\\' in name or ':' in name:
            raise ValueError('Invalid checkpoint member path')
        path=PurePosixPath(name)
        if path.is_absolute() or '..' in path.parts or not path.parts:
            raise ValueError('Checkpoint path escapes its restore root')
        if name not in {'source.parquet','sample.json','run.json'} and not (
            path.parts[0]=='models' and path.suffix=='.json') and not (path.parts[0]=='prepared' and path.suffix in {'.json','.parquet'}):
            raise ValueError('Unexpected file in checkpoint packet')
        return path


    class CheckpointPackets:
        def __init__(self,run_dir,packet_dir,config,source_hash,sample_hash,inference_hash,
                     packet_bytes=64*1024**2,segment_bytes=16*1024**2,restore_info=None):
            self.root=Path(run_dir).resolve();self.output=Path(packet_dir).resolve()
            self.output.mkdir(parents=True,exist_ok=True)
            self.config=copy.deepcopy(config)
            self.ids={'run_name':RUN_NAME,'source_sha256':source_hash,
                'sample_sha256':sample_hash,'inference_sha256':inference_hash}
            self.packet_bytes=packet_bytes;self.segment_bytes=min(segment_bytes,packet_bytes)
            if self.segment_bytes<=0:raise ValueError('Backup segment size must be positive')
            restore_info=restore_info or {}
            self.sequence=restore_info.get('sequence',0)
            self.parent=restore_info.get('head_sha256')
            self.covered={}
            for name,detail in restore_info.get('files',{}).items():
                stat=(self.root/name).stat()
                self.covered[name]=(stat.st_size,stat.st_mtime_ns,detail['sha256'])
            self.receipts=[];self.pending=[];self.last_cycle=time.monotonic()

        def candidate_files(self):
            files=[self.root/name for name in ['source.parquet','sample.json','run.json']]
            files+=sorted((self.root/'models').rglob('*.json'))
            files+=sorted(p for p in (self.root/'prepared').rglob('*') if p.is_file() and p.suffix in {'.json','.parquet'})
            return [p for p in files if p.is_file()]

        def backup(self,reason,download):
            new=[];observed={}
            for path in self.candidate_files():
                name=path.relative_to(self.root).as_posix();checkpoint_relative_path(name)
                stat=path.stat();prior=self.covered.get(name)
                if prior:
                    if (stat.st_size,stat.st_mtime_ns)!=prior[:2]:
                        raise RuntimeError(f'Previously backed-up immutable checkpoint was modified: {name}')
                    continue
                sha=sha_file_stream(path)
                observed[name]=(stat.st_size,stat.st_mtime_ns,sha)
                if stat.st_size==0:raise ValueError('Empty checkpoint file')
                for offset in range(0,stat.st_size,self.segment_bytes):
                    new.append({'path':name,'offset':offset,'bytes':min(self.segment_bytes,stat.st_size-offset),
                        'file_bytes':stat.st_size,'file_sha256':sha})
            if not new:
                self.deliver_pending(download)
                self.last_cycle=time.monotonic()
                return []
            groups=[];part=[];size=0
            for segment in new:
                if part and size+segment['bytes']>self.packet_bytes:
                    groups.append(part);part=[];size=0
                part.append(segment);size+=segment['bytes']
            if part:groups.append(part)
            cycle=secrets.token_hex(8);created=[]
            sequence,parent=self.sequence,self.parent
            for group_index,segments in enumerate(groups):
                sequence+=1
                path=self.output/f'{RUN_NAME}_cp_{sequence:06d}_{cycle}.zip'
                saved_segments=[]
                with zipfile.ZipFile(path,'x',zipfile.ZIP_DEFLATED,compresslevel=1) as archive:
                    for index,segment in enumerate(segments):
                        entry=f'segment_{index:06d}.bin';h=hashlib.sha256();remaining=segment['bytes']
                        with (self.root/segment['path']).open('rb') as src,archive.open(entry,'w') as dst:
                            src.seek(segment['offset'])
                            while remaining:
                                block=src.read(min(1024*1024,remaining))
                                if not block:raise IOError('Checkpoint shortened during backup')
                                h.update(block);dst.write(block);remaining-=len(block)
                        saved_segments.append({**segment,'entry':entry,'segment_sha256':h.hexdigest()})
                    manifest={'schema_version':PACKET_SCHEMA,**self.ids,'sequence':sequence,
                        'parent_sha256':parent,'cycle_id':cycle,'cycle_complete':group_index==len(groups)-1,
                        'reason':reason,'resume_config':self.config,'versions':VERSIONS,
                        'segments':saved_segments,'at_created':utc_now()}
                    archive.writestr(PACKET_MANIFEST,json.dumps(manifest,ensure_ascii=False,allow_nan=False))
                with zipfile.ZipFile(path) as archive:
                    if archive.testzip() is not None:raise IOError('Checkpoint ZIP integrity check failed')
                sha=sha_file_stream(path)
                parent=sha
                receipt={'path':str(path),'sha256':sha,'sequence':sequence,
                    'cycle_complete':manifest['cycle_complete'],'status':'server_file_created',
                    'bytes':path.stat().st_size,'reason':reason}
                created.append(receipt)
                print('CHECKPOINT PACKET VERIFIED:',path.name,flush=True)
            for name,detail in observed.items():
                stat=(self.root/name).stat()
                if (stat.st_size,stat.st_mtime_ns)!=detail[:2]:raise RuntimeError('Checkpoint changed during backup')
            self.covered.update(observed)
            self.sequence,self.parent=sequence,parent
            self.receipts.extend(created);self.pending.extend(created)
            self.last_cycle=time.monotonic()
            self.deliver_pending(download)
            return created

        def deliver_pending(self,download):
            while self.pending:
                receipt=self.pending[0]
                result=download(Path(receipt['path']))
                receipt.update(delivery=result,status=result['status'])
                folder=self.root/'recovery';folder.mkdir(exist_ok=True)
                atomic_json(folder/f'packet_receipt_{receipt["sequence"]:06d}_{secrets.token_hex(4)}.json',receipt)
                self.pending.pop(0)


    def restore_checkpoint_packets(paths,output_root):
        paths=[Path(p) for p in paths]
        if not paths:raise FileNotFoundError('No checkpoint packets. Upload all downloaded *_cp_*.zip files to RESTORE_BACKUP_DIR using the Colab Files panel, then rerun the restore cell.')
        packets=[]
        for path in paths:
            with zipfile.ZipFile(path) as archive:
                info=archive.getinfo(PACKET_MANIFEST)
                if info.file_size>8*1024**2:raise ValueError('Checkpoint manifest is unexpectedly large')
                if sum(i.file_size for i in archive.infolist())>512*1024**2:
                    raise ValueError('Checkpoint packet exceeds restore safety size')
                manifest=json.loads(archive.read(info))
            if manifest.get('schema_version') not in {PACKET_SCHEMA,'georag-checkpoint-packets-3.0'}:raise ValueError('Unsupported checkpoint packet schema')
            packets.append((manifest['sequence'],path,manifest,sha_file_stream(path)))
        packets.sort(key=lambda x:x[0])
        parent=None;last_complete=0;identity=None
        for expected,(sequence,path,manifest,sha) in enumerate(packets,1):
            if sequence!=expected or manifest['parent_sha256']!=parent:
                raise ValueError('Checkpoint chain has a missing, duplicate or altered packet; provide the complete sequence starting at 1')
            current=tuple(manifest[k] for k in ['run_name','source_sha256','sample_sha256','inference_sha256'])
            if identity is None:identity=current
            if current!=identity:raise ValueError('Packets belong to different runs')
            parent=sha
            if manifest['cycle_complete']:last_complete=expected
        if not last_complete:raise ValueError('No complete backup cycle; additional packets are required')
        if last_complete<len(packets):print('Ignoring incomplete final backup cycle; restoring the last complete cycle only')
        packets=packets[:last_complete]
        output_root=Path(output_root).resolve();output_root.mkdir(parents=True,exist_ok=True)
        needed={segment['path']:segment['file_bytes'] for _,_,manifest,_ in packets for segment in manifest['segments']}
        if sum(needed.values())+globals().get('DISK_RESERVE_GB',0)*1024**3>shutil.disk_usage(output_root).free:
            raise RuntimeError('Insufficient disk to restore all checkpoint files with the configured reserve')
        # Never use archive.extractall: original paths are validated and written in bounded blocks.
        staging=output_root/f'restored_{secrets.token_hex(12)}.restoring'
        staging.mkdir(exist_ok=False);files={};active={}
        for sequence,path,manifest,sha in packets:
            with zipfile.ZipFile(path) as archive:
                names=[item['entry'] for item in manifest['segments']]+[PACKET_MANIFEST]
                if len(set(names))!=len(names) or set(archive.namelist())!=set(names):
                    raise ValueError('Unexpected or duplicate checkpoint ZIP entries')
                for segment in manifest['segments']:
                    relative=checkpoint_relative_path(segment['path']).as_posix()
                    target=(staging/relative).resolve()
                    if not target.is_relative_to(staging.resolve()):raise ValueError('Restore target escapes staging directory')
                    if relative in files:raise ValueError('Completed immutable file appears again in the backup chain')
                    state=active.setdefault(relative,{'bytes':0,'hash':hashlib.sha256(),
                        'total':segment['file_bytes'],'sha':segment['file_sha256']})
                    if segment['offset']!=state['bytes'] or segment['bytes']<=0 or segment['file_bytes']!=state['total'] or segment['file_sha256']!=state['sha']:
                        raise ValueError('Missing/overlapping checkpoint file segment')
                    target.parent.mkdir(parents=True,exist_ok=True)
                    h=hashlib.sha256();remaining=segment['bytes']
                    mode='xb' if state['bytes']==0 else 'ab'
                    with archive.open(segment['entry']) as src,target.open(mode) as dst:
                        while remaining:
                            block=src.read(min(1024*1024,remaining))
                            if not block:raise IOError('Incomplete checkpoint segment')
                            dst.write(block);h.update(block);state['hash'].update(block)
                            state['bytes']+=len(block);remaining-=len(block)
                        if src.read(1):raise ValueError('Extra bytes in checkpoint segment')
                    if h.hexdigest()!=segment['segment_sha256']:raise ValueError('Checkpoint segment hash mismatch')
                    if state['bytes']>state['total']:raise ValueError('Checkpoint file size exceeded')
                    if state['bytes']==state['total']:
                        if state['hash'].hexdigest()!=state['sha']:raise ValueError('Checkpoint file hash mismatch')
                        files[relative]={'bytes':state['bytes'],'sha256':state['sha']}
                        del active[relative]
        if active:raise ValueError('Backup cycle contains an incomplete file')
        for required in ['source.parquet','sample.json','run.json']:
            if required not in files:raise ValueError('Backup is missing a required run file')
        last=packets[-1][2]
        if files['source.parquet']['sha256']!=last['source_sha256']:raise ValueError('Source snapshot hash mismatch')
        run=json.loads((staging/'run.json').read_text(encoding='utf-8'))
        if run['sample_sha256']!=last['sample_sha256'] or run['inference_sha256']!=last['inference_sha256']:
            raise ValueError('Restored run identity differs from packet manifest')
        target=output_root/f'restored_{secrets.token_hex(12)}'
        if not staging.resolve().is_relative_to(output_root) or not target.resolve().is_relative_to(output_root):
            raise ValueError('Restore rename target escapes output root')
        staging.rename(target)
        return {'run_dir':str(target),'run_name':last['run_name'],'sequence':last_complete,
            'head_sha256':packets[-1][3],'files':files,'resume_config':last['resume_config'],
            'versions':last['versions'],'source_sha256':last['source_sha256']}

    """Checkpoint hooks; invoked only after an immutable shard has been committed."""
    CHECKPOINT_BACKUPS=None


    def checkpoint_safety_hook(force=False,reason='checkpoint_committed'):
        manager=globals().get('CHECKPOINT_BACKUPS')
        snapshot=resource_snapshot()
        trouble=[]
        ram=snapshot['ram_available_gb']
        if ram is not None and ram<RAM_RESERVE_GB:trouble.append(f'available RAM {ram:.2f} GiB')
        if snapshot['disk_free_gb']<DISK_RESERVE_GB:trouble.append(f'free disk {snapshot["disk_free_gb"]:.2f} GiB')
        if MAX_SESSION_HOURS is not None and snapshot['session_hours']>=MAX_SESSION_HOURS:trouble.append('configured session time budget reached')
        if manager is not None:
            due=(time.monotonic()-manager.last_cycle)>=BACKUP_INTERVAL_MINUTES*60
            if trouble or force or due:
                actual_reason='safe_stop: '+', '.join(trouble) if trouble else reason
                print('Checkpoint backup boundary:',actual_reason,flush=True)
                manager.backup(actual_reason,request_checkpoint_download)
        if trouble:
            folder=RUN_DIR/'recovery';folder.mkdir(exist_ok=True)
            atomic_json(folder/f'safe_stop_{SESSION_ID}_{secrets.token_hex(4)}.json',
                {'schema_version':'georag-resource-stop-1.0','reason':trouble,'resources':snapshot,
                 'backup_head_sha256':manager.parent if manager else None,
                 'last_backup_sequence':manager.sequence if manager else None,
                 'backup_delivery_scope':'Requests only; verify ZIP files were saved on your computer.'})
            raise SafeCheckpointStop('Stopped after a committed checkpoint: '+', '.join(trouble)+
                '. Save all checkpoint packet ZIPs. Start a fresh GPU runtime and restore them; do not discard the original packets.')


    def initialize_checkpoint_backups():
        global CHECKPOINT_BACKUPS
        if not ENABLE_CHECKPOINT_BACKUPS:
            print('WARNING checkpoint downloads disabled: VM loss may lose all local-only outputs')
            return
        if not AUTO_DOWNLOAD_BACKUPS:
            print('WARNING backup ZIPs remain on the server only; download them manually or VM loss can lose both checkpoints and backups')
        restore_info=globals().get('RESTORE_INFO')
        packet_dir=OUTPUT_ROOT/'checkpoint_backups'/RUN_NAME/SESSION_ID
        CHECKPOINT_BACKUPS=CheckpointPackets(RUN_DIR,packet_dir,CONFIG,SOURCE_META['sha256'],
            SAMPLE_HASH,INFERENCE_HASH,packet_bytes=BACKUP_PACKET_MIB*1024**2,
            segment_bytes=BACKUP_SEGMENT_MIB*1024**2,restore_info=restore_info)
        checkpoint_safety_hook(force=True,reason='run_initialized')


    def restore_saved_parameters(info):
        """Apply a fixed allowlist of JSON configuration keys; never execute archive code."""
        saved=copy.deepcopy(info['resume_config'])
        if saved.get('repo_url')!=REPO_URL:
            raise ValueError('Saved repository differs from the explicitly configured destination; update the configuration intentionally before restoring')
        if saved.get('schema_version')!=SCHEMA_VERSION:
            raise ValueError('Checkpoint NER schema differs; use the notebook version that created it')
        if set(saved['models'])!={'bert','gliner'}:raise ValueError('Unexpected model adapter keys')
        for model in saved['models'].values():
            if not isinstance(model.get('id'),str) or not re.fullmatch(r'[0-9a-f]{40}',model.get('revision','')):
                raise ValueError('Invalid pinned model configuration')
        mapping={'MODELS':'models','SEED':'seed','SAMPLE_SIZE':'sample_size_requested',
            'PROCESS_ALL':'process_all','RUN_RANDOM_ID':'run_random_id','FINAL_OUTPUT_PREFIX':'final_output_prefix',
            'ADD_BERT_PER':'add_bert_per','BERT_PER_MIN_SCORE':'bert_per_min_score',
            'EVENT_HIGH_PRIORITY_SCORE':'event_high_priority_score','GLINER_THRESHOLD':'gliner_threshold',
            'GLINER_LABELS':'gliner_labels','LABEL_MAP':'label_map','PRIMARY':'primary_v2',
            'WINDOWS':'windows','CHECKPOINT_PARAGRAPHS':'checkpoint_paragraphs',
            'INPUT_SOURCE':'input_source','REPO_URL':'repo_url','SOURCE_RELATIVE_PATH':'source_path',
            'SOURCE_COMMIT':'source_commit','SELECTION_THRESHOLDS':'selection_thresholds'}
        for target,key in mapping.items():globals()[target]=copy.deepcopy(saved[key])
        batch=saved['batch_execution']
        for target,key in [('BATCH_ENABLED','enabled'),('BATCH_SIZES','sizes'),('BATCH_SCORE_ATOL','score_atol'),
            ('PREFLIGHT_PARAGRAPHS','preflight_paragraphs'),('PREFLIGHT_MAX_PARAGRAPH_CHARS','preflight_max_paragraph_chars')]:
            globals()[target]=copy.deepcopy(batch[key])
        if not isinstance(SEED,int) or not isinstance(CHECKPOINT_PARAGRAPHS,int) or CHECKPOINT_PARAGRAPHS<=0:
            raise ValueError('Invalid saved seed/checkpoint settings')
        if not re.fullmatch(r'[A-Za-z0-9_-]+',FINAL_OUTPUT_PREFIX):raise ValueError('Invalid saved filename prefix')
        if INPUT_SOURCE not in {'local','github'}:raise ValueError('Invalid saved input mode')
        if not all(isinstance(n,int) and n>0 for n in BATCH_SIZES.values()):raise ValueError('Invalid saved batch sizes')
        globals()['TEXT_PREPROCESSING'] = copy.deepcopy(saved['text_preprocessing'])
        globals()['CACHE_ENABLED'] = saved['prediction_cache']['enabled']
        globals()['CACHE_LRU_SHARDS'] = CONFIG['prediction_cache']['lru_shards']
        globals()['CONFLICT_POLICY'] = saved['conflicts']['policy']
        globals()['CONFLICT_DIAGNOSTIC_MIN_SCORE'] = saved['conflicts']['diagnostic_min_score']
        globals()['SOURCE_REF'] = saved.get('source_ref', 'main')
        globals()['DELTA_REUSE_ENABLED'] = saved['delta_reuse']['enabled']
        globals()['DELTA_BASE_RUN_NAME'] = saved['delta_reuse']['base_run_name']
        globals()['DELTA_BASE_RUN_DIR'] = saved['delta_reuse']['base_run_dir']
        globals()['DELTA_REUSE_ERRORS'] = saved['delta_reuse']['reuse_errors']
        saved['fixed_publication'] = copy.deepcopy(CONFIG['fixed_publication'])
        saved['preparation'] = copy.deepcopy(CONFIG['preparation'])
        saved['checkpoint_transport'] = copy.deepcopy(CONFIG['checkpoint_transport'])
        saved['checkpoint_efficiency'] = copy.deepcopy(CONFIG['checkpoint_efficiency'])
        saved['alignment_validation'] = copy.deepcopy(CONFIG['alignment_validation'])
        saved['compact_storage'] = copy.deepcopy(CONFIG['compact_storage'])
        # Resource/backup thresholds may change across hosts; the NER settings may not.
        saved['resilience']=copy.deepcopy(CONFIG['resilience'])
        saved['github_publication']=copy.deepcopy(CONFIG['github_publication'])
        saved['github_checkpoint_publication']=copy.deepcopy(CONFIG['github_checkpoint_publication'])
        saved['output_root']=str(OUTPUT_ROOT);saved['device']=DEVICE
        saved['restored_backup_head_sha256']=info['head_sha256']
        globals()['CONFIG']=saved
        globals()['RESUME_RUN_DIR']=info['run_dir']
        torch.manual_seed(SEED)
        if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
        if info['versions']!=VERSIONS:
            raise RuntimeError('Library versions differ from the saved run. Restore the recorded package versions '
                'before inference; the notebook will not mix incompatible checkpoints. Saved: '+json.dumps(info['versions']))

    class FatalWindowGPUOOM(BaseException):
        """Resource abort intentionally bypasses per-paragraph extraction error handlers."""
        pass

    """Disk-backed paragraph sequence: exact existing rows, bounded row-group cache."""
    from bisect import bisect_right
    from operator import index as integer_index


    class ParquetParagraphs:
        def __init__(self, path, positions=None):
            self.path = Path(path)
            expected = pa.schema([("Source", pa.string()), ("Index", pa.int64()),
                ("Text", pa.string()), ("start_document", pa.int64()),
                ("end_document", pa.int64()), ("corpus_position", pa.int64())])
            self.ends = []
            total = 0
            with pq.ParquetFile(self.path) as source:
                if not source.schema_arrow.equals(expected, check_metadata=False):
                    raise ValueError("Expected the six-column corpus_paragraphs.parquet input schema")
                for i in range(source.num_row_groups):
                    total += source.metadata.row_group(i).num_rows
                    self.ends.append(total)
            if total <= 0:
                raise ValueError("Empty paragraph corpus")
            self.positions = range(total) if positions is None else positions
            if positions is not None and (len(set(positions)) != len(positions) or any(
                    not isinstance(p, int) or not 0 <= p < total for p in positions)):
                raise ValueError("Invalid or duplicate sample positions")
            self.cached_group = None
            self.cached_rows = None

        def __len__(self):
            return len(self.positions)

        def __getitem__(self, key):
            if isinstance(key, slice):
                return [self[i] for i in range(*key.indices(len(self)))]
            key = integer_index(key)
            if key < 0:
                key += len(self)
            if not 0 <= key < len(self):
                raise IndexError(key)
            position = self.positions[key]
            group = bisect_right(self.ends, position)
            if group != self.cached_group:
                self.cached_rows = None  # release old group before allocating the next one
                self.cached_group = None
                with pq.ParquetFile(self.path) as source:
                    self.cached_rows = source.read_row_group(group).to_pylist()
                self.cached_group = group
            base = self.ends[group - 1] if group else 0
            return {**self.cached_rows[position - base], "sample_position": key}

        def __iter__(self):
            for i in range(len(self)):
                yield self[i]


    def validate_paragraph_source(rows):
        started = time.perf_counter()
        last = articles = 0
        prior_source, prior_index, prior_end = None, -1, 0
        for position, p in enumerate(rows):
            if p["corpus_position"] != position:
                raise ValueError(f"Non-contiguous corpus_position at row {position}")
            source, index, text = p["Source"], p["Index"], p["Text"]
            a, b = p["start_document"], p["end_document"]
            if (not isinstance(source, str) or not isinstance(text, str) or not text.strip()
                    or not isinstance(index, int) or not isinstance(a, int) or not isinstance(b, int)
                    or index < 0 or a < 0 or b - a != len(text)):
                raise ValueError(f"Invalid paragraph anchor at corpus row {position}")
            if source != prior_source:
                if index != 0 or (prior_source is not None and source <= prior_source):
                    raise ValueError(f"Article ordering/index mismatch at corpus row {position}")
                articles += 1
            elif index != prior_index + 1 or a < prior_end:
                raise ValueError(f"Paragraph ordering/overlap mismatch at corpus row {position}")
            prior_source, prior_index, prior_end = source, index, b
            last = progress("input_validation", position + 1, len(rows), 0, started, last)
        return {"paragraphs": len(rows), "articles_with_eligible_paragraphs": articles,
                "paragraph_anchors": "valid", "full_article_slice_validation": "not_available",
                "scope": "Exact paragraph text and document anchor arithmetic; full articles are not in this input."}


    def assert_document_span(p, start_document, end_document, word):
        a, b = start_document - p["start_document"], end_document - p["start_document"]
        if (p["end_document"] - p["start_document"] != len(p["Text"])
                or not 0 <= a < b <= len(p["Text"]) or p["Text"][a:b] != word):
            raise ValueError("Document coordinates do not match the exact paragraph source anchor")

    """Immutable GitHub checkpoint packets plus complete-cycle indexes; no Drive."""


    class CheckpointGitHub:
        def __init__(self, repository=None, branch=None, root=None, token_provider=None, transport=None):
            self.repository = repository or GITHUB_REPOSITORY
            self.branch = branch or GITHUB_CHECKPOINT_BRANCH
            self.root = (root or GITHUB_CHECKPOINT_ROOT).strip("/")
            if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", self.repository):
                raise ValueError("Invalid checkpoint repository")
            self.remote_path(self.root)
            self.token = (token_provider or read_github_token)()
            if not isinstance(self.token, str) or not self.token.strip():
                raise RuntimeError("An accessible GitHub write credential is required for remote checkpoints")
            self.token = self.token.strip()

            class NoRedirect(urllib.request.HTTPRedirectHandler):
                def redirect_request(self, req, fp, code, msg, headers, newurl):
                    return None
            self.open = transport or urllib.request.build_opener(NoRedirect()).open
            repo = self.api("GET", "")
            self.branch = self.branch or repo["default_branch"]
            self.head = self.api("GET", "/branches/" + urllib.parse.quote(self.branch, safe=""))["commit"]["sha"]
            self.packets = {}
            self.published_head = None

        @staticmethod
        def remote_path(value):
            if not isinstance(value, str) or "\\" in value or not value or value.startswith("/") or any(
                    part in {"", ".", ".."} for part in value.split("/")):
                raise ValueError("Invalid repository-relative checkpoint path")
            return value

        def api(self, method, suffix, payload=None, missing=False, raw=False):
            request = urllib.request.Request("https://api.github.com/repos/" + self.repository + suffix,
                data=encoded(payload) if payload is not None else None, method=method,
                headers={"Authorization": "Bearer " + self.token,
                    "Accept": "application/vnd.github.raw+json" if raw else (
                        "application/vnd.github.object+json" if method == "GET" and suffix.startswith("/contents/")
                        else "application/vnd.github+json"),
                    "X-GitHub-Api-Version": "2026-03-10", "User-Agent": "GeoRAG-v9",
                    "Content-Type": "application/json"})
            try:
                response = self.open(request, timeout=180)
                if raw:
                    return response
                with response:
                    return json.load(response)
            except urllib.error.HTTPError as exc:
                status = exc.code
                exc.close()
                if missing and status == 404:
                    return None
                hints = {401: "Credential invalid/expired", 403: "Check Contents: write, branch policy or rate limit",
                    404: "Repository/branch/path not found or not accessible", 409: "Conflict: retry the backup",
                    422: "Creation rejected; check destination, branch policy and file size"}
                raise RuntimeError(f"GitHub HTTP {status}: " + hints.get(status, "Request failed")) from None
            except (urllib.error.URLError, TimeoutError, OSError):
                raise RuntimeError("GitHub checkpoint connection failed/timed out; local files are retained") from None

        def metadata(self, remote, ref=None, missing=False):
            endpoint = "/contents/" + urllib.parse.quote(self.remote_path(remote), safe="/")
            return self.api("GET", endpoint + "?" + urllib.parse.urlencode({"ref": ref or self.branch}), missing=missing)

        @staticmethod
        def blob_sha(path):
            h = hashlib.sha1(f"blob {Path(path).stat().st_size}\0".encode())
            with Path(path).open("rb") as src:
                for block in iter(lambda: src.read(1024 * 1024), b""):
                    h.update(block)
            return h.hexdigest()

        def put(self, path, remote):
            path = Path(path)
            remote = self.remote_path(remote)
            size = path.stat().st_size
            if not 0 < size < 25 * 1024**2:
                raise ValueError("Checkpoint packet/index must be smaller than 25 MiB")
            sha, digest_before = self.blob_sha(path), sha_file_stream(path)
            old = self.metadata(remote, missing=True)
            if old is not None:
                if not isinstance(old, dict) or old.get("type") != "file" or old.get("sha") != sha:
                    raise FileExistsError("Different remote checkpoint exists; never overwritten: " + remote)
                commit = None
            else:
                result = self.api("PUT", "/contents/" + urllib.parse.quote(remote, safe="/"), {
                    "branch": self.branch, "message": "Add immutable GeoRAG checkpoint " + path.name,
                    "content": base64.b64encode(path.read_bytes()).decode("ascii")})
                if result["content"]["sha"] != sha:
                    raise RuntimeError("Unexpected uploaded Git blob SHA")
                commit = result["commit"]["sha"]
                time.sleep(1.1)  # serial, paced writes rather than parallel commit bursts
            saved = self.metadata(remote, ref=commit)
            if saved.get("sha") != sha or saved.get("size") != size or sha_file_stream(path) != digest_before:
                raise RuntimeError("Checkpoint remote hash/size verification failed")
            return {"path": remote, "blob_sha": sha, "sha256": digest_before, "bytes": size}

        def upload_packet(self, path):
            with zipfile.ZipFile(path) as archive:
                manifest = json.loads(archive.read(PACKET_MANIFEST))
            if manifest["run_name"] != RUN_NAME or manifest["schema_version"] != PACKET_SCHEMA:
                raise ValueError("Checkpoint packet run/schema mismatch")
            sequence = manifest["sequence"]
            remote = f"{self.root}/{RUN_NAME}/packets/{(sequence - 1) // 500:06d}/{path.name}"
            receipt = self.put(path, remote)
            descriptor = {**receipt, "sequence": sequence, "cycle_complete": manifest["cycle_complete"]}
            if sequence in self.packets and self.packets[sequence] != descriptor:
                raise RuntimeError("Checkpoint sequence points to different remote bytes")
            self.packets[sequence] = descriptor
            print("GITHUB CHECKPOINT VERIFIED:", sequence, remote, flush=True)
            return {"status": "github_uploaded_verified", **receipt}

        def publish_index(self, manager):
            if self.published_head == manager.parent:
                return
            ordered = [self.packets[i] for i in range(1, manager.sequence + 1)]
            if not ordered or not ordered[-1]["cycle_complete"] or ordered[-1]["sha256"] != manager.parent:
                raise ValueError("Cannot publish an incomplete checkpoint cycle")
            index = {"schema_version": "georag-github-checkpoint-index-1.0", **manager.ids,
                "sequence": manager.sequence, "head_sha256": manager.parent, "packets": ordered,
                "files": {name: {"bytes": detail[0], "sha256": detail[2]}
                          for name, detail in manager.covered.items()}}
            name = f"checkpoint_{manager.sequence:08d}_{manager.parent[:12]}.json"
            local = RUN_DIR / "recovery" / name
            local.parent.mkdir(exist_ok=True)
            if local.exists():
                if load_json(local) != index:
                    raise FileExistsError("Existing local checkpoint index differs")
            else:
                atomic_json(local, index)
            self.put(local, f"{self.root}/{RUN_NAME}/indexes/{name}")
            self.published_head = manager.parent
            print("REMOTE RECOVERY POINT CONFIRMED:", RUN_NAME, "packet", manager.sequence, flush=True)

        def download(self, descriptor, target):
            target = Path(target)
            sha = descriptor["blob_sha"]
            if not re.fullmatch(r"[0-9a-f]{40}", sha) or not 0 < descriptor["bytes"] < 25 * 1024**2:
                raise ValueError("Invalid remote checkpoint descriptor")
            with self.api("GET", "/git/blobs/" + sha, raw=True) as src, target.open("xb") as dst:
                count = 0
                while True:
                    block = src.read(1024 * 1024)
                    if not block:
                        break
                    count += len(block)
                    if count > descriptor["bytes"]:
                        raise ValueError("Remote checkpoint exceeds declared size")
                    dst.write(block)
            if count != descriptor["bytes"] or sha_file_stream(target) != descriptor["sha256"]:
                raise ValueError("Downloaded checkpoint hash/size mismatch")

        def latest_index(self, run_name, missing=False):
            if not re.fullmatch(r"[A-Za-z0-9_-]+", run_name):
                raise ValueError("Invalid resume run name")
            self.head = self.api("GET", "/branches/" + urllib.parse.quote(self.branch, safe=""))["commit"]["sha"]
            listing = self.metadata(f"{self.root}/{run_name}/indexes", ref=self.head, missing=missing)
            if listing is None:
                return None
            entries = listing.get("entries", []) if isinstance(listing, dict) else listing
            if len(entries) >= 1000:
                raise RuntimeError("Checkpoint index listing may be truncated; choose an explicit index implementation")
            entries = sorted([e for e in entries if e.get("type") == "file" and re.fullmatch(
                r"checkpoint_\d{8}_[0-9a-f]{12}\.json", e["name"])], key=lambda e: e["name"])
            if not entries:
                raise FileNotFoundError("No confirmed checkpoint index for this run; do not start a different run accidentally")
            with self.api("GET", "/git/blobs/" + entries[-1]["sha"], raw=True) as src:
                payload = src.read(16 * 1024**2 + 1)
            if len(payload) > 16 * 1024**2:
                raise ValueError("Remote checkpoint index too large")
            index = json.loads(payload)
            if index.get("schema_version") != "georag-github-checkpoint-index-1.0" or index["run_name"] != run_name:
                raise ValueError("Remote checkpoint index run/schema mismatch")
            packets = index["packets"]
            if [p["sequence"] for p in packets] != list(range(1, index["sequence"] + 1)):
                raise ValueError("Incomplete remote packet sequence")
            for name, detail in index["files"].items():
                checkpoint_relative_path(name)
                if not re.fullmatch(r"[0-9a-f]{64}", detail["sha256"]) or detail["bytes"] <= 0:
                    raise ValueError("Invalid indexed checkpoint file")
            return index

        def restore(self, run_name, output_root):
            index = self.latest_index(run_name)
            packets = index["packets"]
            target_root = Path(output_root).resolve()
            target_root.mkdir(parents=True, exist_ok=True)
            needed = sum(p["bytes"] for p in packets) + sum(f["bytes"] for f in index["files"].values())
            if needed + DISK_RESERVE_GB * 1024**3 > shutil.disk_usage(target_root).free:
                raise RuntimeError("Insufficient disk for downloaded packets and restored files with the configured reserve")
            folder = Path(output_root) / "restore_packets" / secrets.token_hex(4)
            folder.mkdir(parents=True, exist_ok=False)
            paths = []
            for p in packets:
                remote = self.remote_path(p["path"])
                if not remote.startswith(f"{self.root}/{run_name}/packets/") or not remote.endswith(".zip"):
                    raise ValueError("Checkpoint descriptor points outside the configured run")
                path = folder / f"p_cp_{p['sequence']:08d}.zip"
                self.download(p, path)
                paths.append(path)
                print("RESTORE packet downloaded:", p["sequence"], "/", len(packets), flush=True)
            info = restore_checkpoint_packets(paths, output_root)
            if info["head_sha256"] != index["head_sha256"] or info["sequence"] != index["sequence"]:
                raise ValueError("Downloaded packet chain differs from the committed recovery index")
            if info["files"] != index["files"]:
                raise ValueError("Restored files differ from the committed recovery index")
            self.packets = {p["sequence"]: p for p in packets}
            self.published_head = index["head_sha256"]
            info["remote_index"] = index
            return info


    CHECKPOINT_GITHUB = None


    def checkpoint_safety_hook(force=False, reason="checkpoint_committed"):
        manager = globals().get("CHECKPOINT_BACKUPS")
        snapshot = resource_snapshot()
        trouble = []
        ram = snapshot["ram_available_gb"]
        if ram is not None and ram < RAM_RESERVE_GB:
            trouble.append(f"available RAM {ram:.2f} GiB")
        if snapshot["disk_free_gb"] < DISK_RESERVE_GB:
            trouble.append(f"free disk {snapshot['disk_free_gb']:.2f} GiB")
        if MAX_SESSION_HOURS is not None and snapshot["session_hours"] >= MAX_SESSION_HOURS:
            trouble.append("configured session time budget reached")
        if manager is not None:
            due = time.monotonic() - manager.last_cycle >= BACKUP_INTERVAL_MINUTES * 60
            if trouble or force or due:
                try:
                    manager.backup(reason, CHECKPOINT_GITHUB.upload_packet)
                    CHECKPOINT_GITHUB.publish_index(manager)
                except Exception as exc:
                    folder = RUN_DIR / "recovery"
                    folder.mkdir(exist_ok=True)
                    atomic_json(folder / f"github_failure_{secrets.token_hex(8)}.json", {
                        "schema_version": "georag-github-backup-failure-1.0", "status": "failed",
                        "error": {"type": type(exc).__name__, "message": str(exc)},
                        "last_confirmed_head": CHECKPOINT_GITHUB.published_head,
                        "at_processed": utc_now()})
                    print("GitHub backup failed. Local committed checkpoints are retained:", RUN_DIR, flush=True)
                    raise SafeCheckpointStop("Remote checkpoint not confirmed. Processing stopped to avoid "
                        "continuing without a durable backup. Fix authentication/network and rerun the model cell "
                        "on this VM, or restore the last confirmed GitHub recovery point on a fresh VM.") from exc
        if trouble:
            raise SafeCheckpointStop("Stopped after a committed checkpoint and backup: " + ", ".join(trouble))


    def initialize_checkpoint_backups():
        global CHECKPOINT_BACKUPS, CHECKPOINT_GITHUB
        if not ENABLE_CHECKPOINT_BACKUPS:
            print("WARNING: remote checkpoint backups explicitly disabled; outputs remain local")
            return
        if CHECKPOINT_GITHUB is None:
            CHECKPOINT_GITHUB = CheckpointGitHub()
        restore_info = globals().get("RESTORE_INFO")
        if restore_info is None:
            prior = CHECKPOINT_GITHUB.latest_index(RUN_NAME, missing=True)
            if prior is not None:
                for key, expected in [("source_sha256", SOURCE_META["sha256"]),
                                      ("sample_sha256", SAMPLE_HASH), ("inference_sha256", INFERENCE_HASH)]:
                    if prior[key] != expected:
                        raise ValueError("Remote backup identity differs from the local resumed run")
                for name, detail in prior["files"].items():
                    local = RUN_DIR / name
                    if not local.is_file() or local.stat().st_size != detail["bytes"] or sha_file_stream(local) != detail["sha256"]:
                        raise ValueError("Local resumed run lacks verified files; use GITHUB_RESUME_RUN_NAME to restore from GitHub")
                restore_info = {"sequence": prior["sequence"], "head_sha256": prior["head_sha256"],
                                "files": prior["files"]}
                CHECKPOINT_GITHUB.packets = {p["sequence"]: p for p in prior["packets"]}
                CHECKPOINT_GITHUB.published_head = prior["head_sha256"]
        packet_dir = OUTPUT_ROOT / "checkpoint_backups" / RUN_NAME / "delivery_v13"
        CHECKPOINT_BACKUPS = CheckpointPackets(RUN_DIR, packet_dir, CONFIG, SOURCE_META["sha256"],
            SAMPLE_HASH, INFERENCE_HASH, packet_bytes=BACKUP_PACKET_MIB * 1024**2,
            segment_bytes=BACKUP_SEGMENT_MIB * 1024**2, restore_info=restore_info)
        checkpoint_safety_hook(force=True, reason="run_initialized")

    PROCESS_PARAGRAPH_V9 = process_paragraph
    """Conservative offset-preserving text masking and occurrence-safe prediction reuse."""
    import html
    import sqlite3

    URL_PATTERN = re.compile(r'''(?:https?://|ftp://|www\.)[^\s<>{}\[\]|"']+''', re.IGNORECASE)
    MODEL_CACHES = {}
    RAW_REFERENCE_CACHE = OrderedDict()


    def prepare_inference_text(text):
        """Keep every original character position; never overwrite the source text."""
        settings = TEXT_PREPROCESSING
        chars = list(text)
        actions = Counter()

        def mask(a, b, reason):
            for i in range(a, b):
                if not chars[i].isspace():
                    chars[i] = " "
            actions[reason] += 1

        if settings["enabled"]:
            if settings["mask_comments"]:
                for m in re.finditer(r"<!--[\s\S]*?(?:-->|$)", text):
                    mask(m.start(), m.end(), "comment")
            if settings["mask_urls"]:
                for m in URL_PATTERN.finditer(text):
                    end = m.end()
                    while end > m.start() and text[end - 1] in ".,;!?":
                        end -= 1
                    mask(m.start(), end, "url")
                for m in re.finditer(r"\[(?:https?://|ftp://|www\.)[^\]\n]*\]", text, re.IGNORECASE):
                    mask(m.start(), m.start() + 1, "external_link_delimiter")
                    mask(m.end() - 1, m.end(), "external_link_delimiter")
            if settings["mask_html_tags"]:
                for m in re.finditer(r"</?[A-Za-z][^<>\n]*>", text):
                    mask(m.start(), m.end(), "html_tag")
            if settings["mask_wiki_syntax"]:
                for m in re.finditer(r"\[\[([^\[\]\n]+)\]\]", text):
                    inner = m.group(1)
                    bar = inner.rfind("|")
                    mask(m.start(), m.start() + 2, "wiki_link_delimiter")
                    mask(m.end() - 2, m.end(), "wiki_link_delimiter")
                    if bar >= 0 and inner[bar + 1:].strip():
                        mask(m.start() + 2, m.start() + 3 + bar, "wiki_link_target")
                for m in re.finditer(r"\{\{[^{}|\n]+(?=\|)", text):
                    mask(m.start(), m.end(), "template_function")
                field = re.match(r"^[ \t]*\|[ A-Za-z0-9_\-]+\s*=", text)
                if field:
                    mask(0, field.end(), "field_name")
                if re.match(r"^\s*\{\{[^{}|\n]+$", text) or re.match(r"^\s*\{\|", text):
                    mask(0, len(text), "structural_header")
                if re.fullmatch(r"\s*\|[-}](?:\s*(?:class|style|id)\s*=.*)?\s*", text):
                    mask(0, len(text), "table_row_separator")
                for m in re.finditer(r"\{\{|\}\}|\[\[|\]\]|'{2,5}|\||!!|^[ \t]*[!*#;]+", text):
                    mask(m.start(), m.end(), "wiki_delimiter")
            if settings["decode_html_entities"]:
                for m in re.finditer(r"&(?:#[0-9]+|#x[0-9A-Fa-f]+|[A-Za-z][A-Za-z0-9]+);", text):
                    # Do not revive an entity inside an already masked URL/comment/tag.
                    if "".join(chars[m.start():m.end()]) != m.group():
                        continue
                    decoded = html.unescape(m.group())
                    if decoded != m.group() and len(decoded) <= m.end() - m.start() and "\ufffd" not in decoded:
                        chars[m.start():m.end()] = list(decoded + " " * (m.end() - m.start() - len(decoded)))
                        actions["html_entity"] += 1
        cleaned = "".join(chars)
        if len(cleaned) != len(text):
            raise ValueError("Offset-preserving preprocessing changed character count")
        no_content = settings["skip_nonlinguistic"] and not any(c.isalnum() for c in cleaned)
        return cleaned, {"version": settings["version"], "actions": dict(actions),
            "characters_changed": sum(a != b for a, b in zip(text, cleaned)),
            "inference_sha256": digest(cleaned.encode("utf-8")), "skipped_no_content": no_content,
            "mapping": "identity_character_positions", "scope": "Conservative masking, not a complete MediaWiki renderer"}


    def process_uncached_occurrence(p, name, predict, measure, limit):
        started = time.perf_counter()
        cleaned, normalization = prepare_inference_text(p["Text"])
        if normalization["skipped_no_content"]:
            result = {"paragraph": p, "raw": [], "mentions": [], "duplicates_resolved": 0,
                "status_processing": "success", "errors": [], "at_processed": utc_now(), "processing_seconds": 0.0}
        else:
            result = PROCESS_PARAGRAPH_V9({**p, "Text": cleaned}, name, predict, measure, limit)
        result["paragraph"] = p
        result["normalization"] = normalization
        result["cache_hit"] = False
        result["inference_evaluated"] = not normalization["skipped_no_content"]
        for item in result["mentions"]:
            item["word_inference"] = item["word"]
            item["Text"] = p["Text"]
            item["text_sha256"] = digest(p["Text"].encode("utf-8"))
            item["inference_text_sha256"] = normalization["inference_sha256"]
            a, b = item["start"], item["end"]
            if isinstance(a, int) and isinstance(b, int) and 0 <= a < b <= len(p["Text"]):
                item["word"] = p["Text"][a:b]
                item["start_document"], item["end_document"] = p["start_document"] + a, p["start_document"] + b
                item["source_span_contains_masked_characters"] = p["Text"][a:b] != cleaned[a:b]
                if item["status_validation"] == "valid":
                    assert_document_span(p, item["start_document"], item["end_document"], item["word"])
        result["processing_seconds"] = time.perf_counter() - started
        return result


    def prediction_cache_namespace(name):
        return digest(encoded({"inference": INFERENCE_HASH, "model_name": name, "model": MODELS[name],
            "window": WINDOWS[name], "labels": GLINER_LABELS if name == "gliner" else "pinned_model_config",
            "raw_threshold": GLINER_THRESHOLD if name == "gliner" else None,
            "text_preprocessing": TEXT_PREPROCESSING, "versions": VERSIONS, "device": DEVICE,
            "cache_version": "prediction-reuse-1.0"}))


    class PredictionReuse:
        """SQLite contains only pointers. Immutable raw checkpoints are authoritative."""
        def __init__(self, name):
            self.name = name
            self.namespace = prediction_cache_namespace(name)
            folder = RUN_DIR / "runtime_cache"
            folder.mkdir(exist_ok=True)
            self.db = sqlite3.connect(folder / f"{name}.sqlite")
            self.db.execute("CREATE TABLE IF NOT EXISTS owners (key TEXT PRIMARY KEY, file TEXT NOT NULL, slot INTEGER NOT NULL)")
            self.pending = {}
            self.parts = OrderedDict()

        def key(self, text):
            return digest((self.namespace + "\0" + text).encode("utf-8"))

        def reset_for_run(self):
            self.pending.clear()
            self.parts.clear()
            self.db.execute("DELETE FROM owners")
            for path in sorted((RUN_DIR / "models" / self.name).glob("cache_import_*.json")):
                filename = path.relative_to(RUN_DIR).as_posix()
                chunk = load_prediction_owner(filename)
                self.register(filename, chunk["results"])
            # Rebuild unique output owners independently of occurrence reconstruction.
            for start in range(0, len(PREPARED), CHECKPOINT_PARAGRAPHS):
                path = unique_shard_path(self.name, start)
                if path.is_file():
                    chunk = load_unique_shard(self.name, start)
                    self.register(path.relative_to(RUN_DIR).as_posix(), chunk["results"])
            # Rebuild from complete, verified shards, including ones restored from GitHub.
            for start in range(0, len(SAMPLE), CHECKPOINT_PARAGRAPHS):
                path = shard_path(self.name, start)
                if not path.is_file():
                    continue
                chunk = load_shard(self.name, start)
                self.register(path.relative_to(RUN_DIR).as_posix(), chunk["results"])
            self.db.commit()

        def register(self, filename, records):
            values = []
            for slot, record in enumerate(records):
                if not record.get("cache_hit", False):
                    expected = self.key(record["paragraph"]["Text"])
                    if record.get("cache_key") != expected:
                        raise ValueError("Prediction cache key/config mismatch")
                    values.append((expected, filename, slot))
            self.db.executemany("INSERT OR IGNORE INTO owners VALUES (?, ?, ?)", values)
            self.db.commit()
            self.pending.clear()

        def contains(self, text):
            key = self.key(text)
            return key in self.pending or self.db.execute("SELECT 1 FROM owners WHERE key=?", (key,)).fetchone() is not None

        def lookup(self, text):
            key = self.key(text)
            if key in self.pending:
                return self.pending[key]
            row = self.db.execute("SELECT file, slot FROM owners WHERE key=?", (key,)).fetchone()
            if row is None:
                return None
            filename, slot = row
            if filename not in self.parts:
                while len(self.parts) >= CACHE_LRU_SHARDS:
                    self.parts.popitem(last=False)
                self.parts[filename] = load_prediction_owner(filename)
            self.parts.move_to_end(filename)
            record = self.parts[filename]["results"][slot]
            if record["paragraph"]["Text"] != text or record.get("cache_hit"):
                raise ValueError("Prediction cache owner is not the exact original paragraph")
            return filename, slot, record

        def close(self):
            self.pending.clear()
            self.parts.clear()
            self.db.close()


    def prediction_cache(name):
        value = MODEL_CACHES.get(name)
        if value is None or value.namespace != prediction_cache_namespace(name):
            if value is not None:
                value.close()
            value = MODEL_CACHES[name] = PredictionReuse(name)
        return value


    def clone_occurrence(owner, p, filename, slot):
        if owner["paragraph"]["Text"] != p["Text"]:
            raise ValueError("Only exact full paragraph text can reuse predictions")
        result = {k: copy.deepcopy(v) for k, v in owner.items() if k not in {"paragraph", "raw", "mentions"}}
        result.update(paragraph=p, raw=[], mentions=copy.deepcopy(owner["mentions"]), cache_hit=True,
                      inference_evaluated=False, at_processed=utc_now(),
                      raw_shared_ref=f"{filename}#/results/{slot}/raw",
                      cache_owner_sample_position=owner["paragraph"]["sample_position"])
        for item in result["mentions"]:
            item.update(Source=p["Source"], Index=p["Index"], Text=p["Text"], sample_position=p["sample_position"],
                        corpus_position=p.get("corpus_position"))
            a, b = item["start"], item["end"]
            if isinstance(a, int) and isinstance(b, int):
                item["start_document"], item["end_document"] = p["start_document"] + a, p["start_document"] + b
            if item["status_validation"] == "valid":
                assert_document_span(p, item["start_document"], item["end_document"], item["word"])
        return result


    def process_paragraph(p, name, predict, measure, limit):
        started = time.perf_counter()
        use_cache = CACHE_ENABLED and re.fullmatch(r"models/" + re.escape(name) + r"/part_\d+\.json", ACTIVE_RAW_FILE)
        cache = prediction_cache(name) if use_cache else None
        hit = cache.lookup(p["Text"]) if cache else None
        if hit is not None:
            filename, slot, owner = hit
            result = clone_occurrence(owner, p, filename, slot)
            result["processing_seconds"] = time.perf_counter() - started
            return result
        result = process_uncached_occurrence(p, name, predict, measure, limit)
        if cache:
            result["cache_key"] = cache.key(p["Text"])
            cache.pending[result["cache_key"]] = (ACTIVE_RAW_FILE, ACTIVE_RAW_INDEX, result)
        else:
            result["cache_key"] = None
        return result


    def resolve_raw_reference(ref, current_chunk=None, current_file=None):
        filename, pointer = ref.split("#", 1)
        match = re.fullmatch(r"models/(bert|gliner)/part_(\d+)\.json", filename)
        if not match:
            raise ValueError("Unexpected model raw-reference owner")
        if filename == current_file and current_chunk is not None:
            node = current_chunk
        else:
            if filename not in RAW_REFERENCE_CACHE:
                while len(RAW_REFERENCE_CACHE) >= CACHE_LRU_SHARDS:
                    RAW_REFERENCE_CACHE.popitem(last=False)
                RAW_REFERENCE_CACHE[filename] = load_shard(match.group(1), int(match.group(2)))
            RAW_REFERENCE_CACHE.move_to_end(filename)
            node = RAW_REFERENCE_CACHE[filename]
        for token in pointer.strip("/").split("/"):
            node = node[int(token)] if isinstance(node, list) else node[token]
        return node

    CLONE_OCCURRENCE_V10 = clone_occurrence
    """Cross-source prediction reuse with per-model compatibility and current-only export."""
    DELTA_IMPORT_MANIFEST = {"status": "not_initialized"}


    def fingerprint_from_config(name, config, versions, device):
        return digest(encoded({"prediction_format": "source-independent-1.0", "model_name": name,
            "model": config["models"][name], "window": config["windows"][name],
            "labels": config["gliner_labels"] if name == "gliner" else "pinned_model_config",
            "raw_threshold": config["gliner_threshold"] if name == "gliner" else None,
            "text_preprocessing": config["text_preprocessing"], "versions": versions, "device": device,
            "batch_enabled": config["batch_execution"]["enabled"],
            "batch_size_requested": config["batch_execution"]["sizes"][name], "aggregation": "average"}))


    def prediction_cache_namespace(name):
        # Deliberately excludes source, article names, positions, sample and selection thresholds.
        # These change the occurrence view, not raw model predictions for an identical text.
        actual = {**CONFIG, "models": MODELS, "windows": WINDOWS, "gliner_labels": GLINER_LABELS,
            "gliner_threshold": GLINER_THRESHOLD, "text_preprocessing": TEXT_PREPROCESSING,
            "batch_execution": {"enabled": BATCH_ENABLED, "sizes": BATCH_SIZES}}
        return fingerprint_from_config(name, actual, VERSIONS, DEVICE)


    def load_prediction_owner(filename):
        match = re.fullmatch(r"models/(bert|gliner)/(?:part_(\d+)|cache_import_(\d+))\.json", filename)
        if not match:
            raise ValueError("Unexpected prediction owner path")
        if match.group(2) is not None:
            return load_shard(match.group(1), int(match.group(2)))
        chunk = load_json(RUN_DIR / filename)
        if (chunk.get("schema_version") != "ner-delta-cache-import-1.0" or chunk["model"] != match.group(1)
                or chunk["prediction_fingerprint"] != prediction_cache_namespace(match.group(1))
                or chunk["payload_sha256"] != digest(encoded(chunk["results"]))):
            raise ValueError("Imported prediction owner hash/config mismatch")
        return chunk


    def clone_occurrence(owner, p, filename, slot):
        result = CLONE_OCCURRENCE_V10(owner, p, filename, slot)
        for item in result["mentions"]:
            item.update(source_sha256=SOURCE_META["sha256"], source_commit=SOURCE_META["commit_sha"], source_path=SOURCE_META["path"])
        return result


    def resolve_raw_reference(ref, current_chunk=None, current_file=None):
        filename, pointer = ref.split("#", 1)
        if filename == current_file and current_chunk is not None:
            node = current_chunk
        else:
            if filename not in RAW_REFERENCE_CACHE:
                while len(RAW_REFERENCE_CACHE) >= CACHE_LRU_SHARDS:
                    RAW_REFERENCE_CACHE.popitem(last=False)
                RAW_REFERENCE_CACHE[filename] = load_prediction_owner(filename)
            RAW_REFERENCE_CACHE.move_to_end(filename)
            node = RAW_REFERENCE_CACHE[filename]
        for token in pointer.strip("/").split("/"):
            node = node[int(token)] if isinstance(node, list) else node[token]
        return node


    def initialize_delta_reuse():
        global DELTA_IMPORT_MANIFEST
        manifest_path = RUN_DIR / "models/delta_manifest.json"
        manifest_path.parent.mkdir(exist_ok=True)
        config_path = RUN_DIR / "models/delta_source_config.json"
        if not config_path.exists():
            atomic_json(config_path, {"configuration": CONFIG, "versions": VERSIONS})
        fingerprints = {n: prediction_cache_namespace(n) for n in MODELS}
        if manifest_path.exists():
            DELTA_IMPORT_MANIFEST = load_json(manifest_path)
            if DELTA_IMPORT_MANIFEST.get("fingerprints") != fingerprints:
                raise ValueError("Restored delta import fingerprints differ")
            for entry in DELTA_IMPORT_MANIFEST.get("imports", []):
                if digest_file(RUN_DIR / entry["path"]) != entry["sha256"]:
                    raise ValueError("Restored delta cache file hash differs")
            print("Delta state restored:", DELTA_IMPORT_MANIFEST["status"])
            return
        manifest = {"schema_version": "ner-delta-manifest-1.0", "status": "disabled",
            "fingerprints": fingerprints, "imports": [], "compatible_models": {}, "models": {}}
        base_dir = Path(DELTA_BASE_RUN_DIR).expanduser().resolve() if DELTA_BASE_RUN_DIR else None
        base_name = DELTA_BASE_RUN_NAME
        if not DELTA_REUSE_ENABLED:
            atomic_json(manifest_path, manifest)
            DELTA_IMPORT_MANIFEST = manifest
            return
        if base_dir is None:
            store = CheckpointGitHub()
            checkpoint = None
            if not base_name:
                meta = store.metadata(current_pointer_path(), missing=True)
                if meta:
                    with store.api("GET", "/git/blobs/" + meta["sha"], raw=True) as stream:
                        body = stream.read(16 * 1024**2 + 1)
                    if len(body) > 16 * 1024**2:
                        raise ValueError("Current-run pointer too large")
                    pointer = json.loads(body)
                    if pointer.get("schema_version") != "georag-current-output-1.0":
                        raise ValueError("Unsupported current-run pointer")
                    checkpoint = pointer.get("checkpoint")
                    if checkpoint:
                        if checkpoint["repository"] != GITHUB_REPOSITORY:
                            raise ValueError("Delta checkpoint repository differs")
                        base_name = checkpoint["run_name"]
                if not base_name:
                    manifest["status"] = "no_previous_durable_run"
                    atomic_json(manifest_path, manifest)
                    DELTA_IMPORT_MANIFEST = manifest
                    print("Delta: no previous durable run; this is the first full extraction")
                    return
            if checkpoint:
                store = CheckpointGitHub(branch=checkpoint["branch"], root=checkpoint["root"])
            info = store.restore(base_name, OUTPUT_ROOT / "delta_sources")
            base_dir = Path(info["run_dir"])
            saved_config, saved_versions = info["resume_config"], info["versions"]
        else:
            # Local immutable backup index carries the exact extraction configuration.
            run = load_json(base_dir / "run.json")
            # An explicit local baseline requires a complete saved configuration file.
            config_file = base_dir / "models/delta_source_config.json"
            if config_file.is_file():
                saved = load_json(config_file)
                saved_config, saved_versions = saved["configuration"], saved["versions"]
            else:
                packets = sorted((OUTPUT_ROOT / "checkpoint_backups" / run["run_name"]).rglob("*_cp_*.zip"))
                if not packets:
                    raise ValueError("Local baseline has no saved config/backup packet. Use DELTA_BASE_RUN_NAME for GitHub restore")
                with zipfile.ZipFile(packets[0]) as archive:
                    saved = json.loads(archive.read(PACKET_MANIFEST))
                saved_config, saved_versions = saved["resume_config"], saved["versions"]
        prior_run = load_json(base_dir / "run.json")
        if saved_config.get("schema_version") not in {"ner-selected-3.0", "ner-selected-3.1", "ner-selected-3.2"}:
            raise ValueError("Delta baseline must use v10/v11 prediction records")
        if base_dir.resolve() == RUN_DIR.resolve():
            raise ValueError("Delta baseline must be a different run; use RESUME_RUN_DIR to resume the same run")
        manifest.update(status="imported", base_run_name=prior_run["run_name"], base_source=prior_run["source"],
                        base_inference_sha256=prior_run["inference_sha256"])
        database = sqlite3.connect(RUN_DIR / "runtime_delta.sqlite")
        try:
            database.execute("CREATE TABLE IF NOT EXISTS texts (hash TEXT PRIMARY KEY, text TEXT NOT NULL, current_count INTEGER NOT NULL, old_count INTEGER NOT NULL DEFAULT 0)")
            database.execute("DELETE FROM texts")
            for row in PREPARED.iter_unique():
                database.execute("INSERT INTO texts(hash,text,current_count) VALUES (?,?,1)", (row["text_sha256"], row["Text"]))
            database.commit()
            with pq.ParquetFile(base_dir / "source.parquet") as prior:
                for batch in prior.iter_batches(batch_size=5000, columns=["Text"]):
                    for text in batch.column(0).to_pylist():
                        key = digest(text.encode("utf-8"))
                        database.execute("INSERT INTO texts(hash,text,current_count,old_count) VALUES (?,?,0,1) ON CONFLICT(hash) DO UPDATE SET old_count=old_count+1", (key, text))
            database.commit()
            manifest["text_delta"] = {"unchanged_unique_texts": database.execute("SELECT COUNT(*) FROM texts WHERE current_count>0 AND old_count>0").fetchone()[0],
                "new_unique_texts": database.execute("SELECT COUNT(*) FROM texts WHERE current_count>0 AND old_count=0").fetchone()[0],
                "removed_unique_texts": database.execute("SELECT COUNT(*) FROM texts WHERE current_count=0 AND old_count>0").fetchone()[0],
                "scope": "Exact full original text; positions and article identities are rebound from the current input"}
            for name in MODELS:
                compatible = fingerprint_from_config(name, saved_config, saved_versions, saved_config["device"]) == fingerprints[name]
                manifest["compatible_models"][name] = compatible
                if not compatible:
                    manifest["models"][name] = {"imported_unique_texts": 0, "reason": "model/preprocessing/window/library/execution configuration changed"}
                    continue
                seen = set()
                records, ordinal = [], 0

                def flush():
                    nonlocal records, ordinal
                    if not records:
                        return
                    relative = f"models/{name}/cache_import_{ordinal:08d}.json"
                    for slot, entry in enumerate(records):
                        old_raw = entry.pop("_old_raw_prefix")
                        new_raw = relative + f"#/results/{slot}/raw/"
                        for m in entry["mentions"]:
                            m["ref_output_raw"] = [ref.replace(old_raw, new_raw, 1) for ref in m["ref_output_raw"]]
                    content = {"schema_version": "ner-delta-cache-import-1.0", "model": name,
                        "prediction_fingerprint": fingerprints[name], "base_run_name": prior_run["run_name"],
                        "results": records, "payload_sha256": digest(encoded(records))}
                    target = RUN_DIR / relative
                    target.parent.mkdir(parents=True, exist_ok=True)
                    if target.exists():
                        if load_json(target) != content:
                            raise ValueError("Existing imported cache differs; cannot mix baselines")
                    else:
                        atomic_json(target, content)
                    manifest["imports"].append({"path": relative, "sha256": digest_file(target), "records": len(records)})
                    ordinal += 1
                    records = []

                files = sorted((base_dir / "models" / name).glob("cache_import_*.json")) + sorted((base_dir / "models" / name).glob("unique_part_*.json")) + sorted((base_dir / "models" / name).glob("part_*.json"))
                for path in files:
                    chunk = load_json(path)
                    if chunk.get("payload_sha256") != digest(encoded(chunk["results"])):
                        raise ValueError("Delta baseline model checkpoint payload mismatch")
                    for slot, record in enumerate(chunk["results"]):
                        if record.get("cache_hit") or (not DELTA_REUSE_ERRORS and record["status_processing"] != "success"):
                            continue
                        text = record["paragraph"]["Text"]
                        key = digest(text.encode("utf-8"))
                        if key in seen:
                            continue
                        current = database.execute("SELECT text,current_count FROM texts WHERE hash=?", (key,)).fetchone()
                        if current is None or not current[1]:
                            continue
                        if current[0] != text:
                            raise ValueError("Text digest collision")
                        seen.add(key)
                        entry = copy.deepcopy(record)
                        entry["cache_key"] = digest((fingerprints[name] + "\0" + text).encode("utf-8"))
                        entry["reused_from_run"] = prior_run["run_name"]
                        entry["reused_from_source_sha256"] = prior_run["source"]["sha256"]
                        entry["_old_raw_prefix"] = path.relative_to(base_dir).as_posix() + f"#/results/{slot}/raw/"
                        records.append(entry)
                        if len(records) >= CHECKPOINT_PARAGRAPHS:
                            flush()
                flush()
                manifest["models"][name] = {"imported_unique_texts": len(seen)}
        finally:
            database.close()
        atomic_json(manifest_path, manifest)
        DELTA_IMPORT_MANIFEST = manifest
        print("Delta summary:", json.dumps({k:v for k,v in manifest.items() if k != "imports"}, indent=2))

    """Publish a fixed Parquet and its provenance pointer in one Git commit."""


    def current_pointer_path():
        return str(PurePosixPath(GITHUB_OUTPUT_DIR) / (PurePosixPath(GITHUB_FINAL_FILENAME).stem + ".run.json"))


    def publish_current_parquet(path, expected_rows, client=None):
        path = Path(path)
        if not path.is_file() or not 0 < path.stat().st_size < 100_000_000:
            raise ValueError("GitHub publication requires a nonempty Parquet smaller than 100 MB")
        with pq.ParquetFile(path) as parquet:
            if parquet.schema_arrow != ARROW_SCHEMA or parquet.metadata.num_rows != expected_rows:
                raise ValueError("Current Parquet schema/count differs from validated export")
        if not PROCESS_ALL and not ALLOW_SAMPLE_PUBLICATION:
            return {"status": "skipped_sample", "reason": "Test samples do not replace the full current output"}
        client = client or CheckpointGitHub(branch=GITHUB_UPLOAD_BRANCH)
        if PurePosixPath(GITHUB_FINAL_FILENAME).name != GITHUB_FINAL_FILENAME or not GITHUB_FINAL_FILENAME.endswith(".parquet"):
            raise ValueError("GITHUB_FINAL_FILENAME must be a Parquet filename without a folder")
        target = client.remote_path(str(PurePosixPath(GITHUB_OUTPUT_DIR) / GITHUB_FINAL_FILENAME))
        pointer_path = client.remote_path(current_pointer_path())
        local_sha, local_hash = client.blob_sha(path), digest_file(path)
        manager = globals().get("CHECKPOINT_BACKUPS")
        checkpoint_client = globals().get("CHECKPOINT_GITHUB")
        checkpoint = None
        if manager is not None and checkpoint_client is not None:
            checkpoint_safety_hook(force=True, reason="before_fixed_publication")
            if checkpoint_client.published_head != manager.parent or manager.pending:
                raise RuntimeError("Cannot publish a delta pointer before the recovery point is confirmed")
            checkpoint = {"repository": checkpoint_client.repository, "branch": checkpoint_client.branch,
                "root": checkpoint_client.root, "run_name": RUN_NAME, "head_sha256": manager.parent,
                "sequence": manager.sequence}
        pointer = {"schema_version": "georag-current-output-1.0", "run_name": RUN_NAME,
            "ner_schema_version": SCHEMA_VERSION, "source": SOURCE_META, "sample_sha256": SAMPLE_HASH,
            "inference_sha256": INFERENCE_HASH, "prediction_fingerprints": {n: prediction_cache_namespace(n) for n in MODELS},
            "output": {"path": target, "git_blob_sha": local_sha, "sha256": local_hash,
                       "bytes": path.stat().st_size, "rows": expected_rows},
            "checkpoint": checkpoint, "extraction_status": RUN_STATUS,
            "delta_reuse": DELTA_IMPORT_MANIFEST}
        pointer_bytes = encoded(pointer)
        pointer_sha = hashlib.sha1(f"blob {len(pointer_bytes)}\0".encode() + pointer_bytes).hexdigest()
        # Resolve one base commit. Both files below will become visible together.
        branch_suffix = "/branches/" + urllib.parse.quote(client.branch, safe="")
        base = client.api("GET", branch_suffix)["commit"]["sha"]
        old = client.metadata(target, ref=base, missing=True)
        old_pointer = client.metadata(pointer_path, ref=base, missing=True)
        same = old is not None and old.get("sha") == local_sha and old_pointer is not None and old_pointer.get("sha") == pointer_sha
        if same:
            commit, status = base, "already_present_identical"
        else:
            for metadata in [old, old_pointer]:
                if metadata is not None and metadata.get("type") != "file":
                    raise ValueError("Publication target is not a regular file")
            if old is None or old["sha"] != local_sha:
                result = client.api("POST", "/git/blobs", {"content": base64.b64encode(path.read_bytes()).decode("ascii"), "encoding": "base64"})
                if result["sha"] != local_sha:
                    raise RuntimeError("Parquet Git blob hash mismatch")
            result = client.api("POST", "/git/blobs", {"content": base64.b64encode(pointer_bytes).decode("ascii"), "encoding": "base64"})
            if result["sha"] != pointer_sha:
                raise RuntimeError("Run pointer Git blob hash mismatch")
            base_commit = client.api("GET", "/git/commits/" + base)
            tree = client.api("POST", "/git/trees", {"base_tree": base_commit["tree"]["sha"],
                "tree": [{"path": target, "mode": "100644", "type": "blob", "sha": local_sha},
                         {"path": pointer_path, "mode": "100644", "type": "blob", "sha": pointer_sha}]})
            new_commit = client.api("POST", "/git/commits", {"message": "Update GeoRAG current Parquet: " + RUN_NAME,
                "tree": tree["sha"], "parents": [base]})
            commit = new_commit["sha"]
            # Fast-forward only: a concurrent writer is not silently overwritten.
            client.api("PATCH", "/git/refs/heads/" + urllib.parse.quote(client.branch, safe=""),
                       {"sha": commit, "force": False})
            status = "updated" if old is not None else "created"
        saved = client.metadata(target, ref=commit)
        saved_pointer = client.metadata(pointer_path, ref=commit)
        if (saved["sha"] != local_sha or saved["size"] != path.stat().st_size
                or saved_pointer["sha"] != pointer_sha or digest_file(path) != local_hash):
            raise RuntimeError("Fixed output/provenance verification failed")
        return {"schema_version": "github-current-publication-1.0", "status": status,
            "path": target, "url": f"https://github.com/{client.repository}/blob/{client.branch}/{target}",
            "commit_sha": commit, "git_blob_sha": local_sha, "sha256": local_hash,
            "rows": expected_rows, "bytes": path.stat().st_size, "pointer_path": pointer_path,
            "source": SOURCE_META, "at_published": utc_now()}

    LEGACY_PREPARE_TEXT = prepare_inference_text
    LEGACY_PROCESS_UNCACHED = process_uncached_occurrence
    """Prepare each exact original text once; immutable Parquet parts and occurrence map."""
    from bisect import bisect_right

    PREPARED = None
    PREPARATION_REPORT = {}
    STAGE_TIMINGS = {"models": {}}
    PREPARED_UNIQUE_SCHEMA = pa.schema([
        ("unique_id", pa.int64()), ("text_sha256", pa.string()), ("Text", pa.string()),
        ("Text_inference", pa.string()), ("normalization_json", pa.string()),
        ("eligible", pa.bool_()), ("representative_sample_position", pa.int64())])
    PREPARED_OCCURRENCE_SCHEMA = pa.schema([
        ("sample_position", pa.int64()), ("corpus_position", pa.int64()), ("unique_id", pa.int64()),
        ("Source", pa.string()), ("Index", pa.int64()), ("start_document", pa.int64()), ("end_document", pa.int64())])


    def preparation_identity():
        selection = {"mode": "full", "count": len(SAMPLE)} if PROCESS_ALL else {
            "mode": "sample", "positions": list(SAMPLE.positions)}
        return {"schema_version": "georag-prepared-input-1.0", "source_sha256": SOURCE_META["sha256"],
                "selection": selection, "cleaning": copy.deepcopy(TEXT_PREPROCESSING)}


    def prepared_path(root, relative):
        if not isinstance(relative, str) or "\\" in relative or ":" in relative:
            raise ValueError("Invalid preparation path")
        path = PurePosixPath(relative)
        if path.is_absolute() or ".." in path.parts or not path.parts:
            raise ValueError("Preparation path escapes its root")
        result = (Path(root) / relative).resolve()
        if not result.is_relative_to(Path(root).resolve()):
            raise ValueError("Preparation file escapes root")
        return result


    def write_prepared_part(path, rows, schema):
        table = pa.Table.from_pylist(rows, schema=schema)
        path.parent.mkdir(parents=True, exist_ok=True)
        if path.exists():
            with path.open("rb") as src:
                if not pq.read_table(src).equals(table):
                    raise ValueError("Uncommitted preparation part differs from the deterministic replay")
        else:
            temp = path.with_name(path.name + "." + secrets.token_hex(4) + ".tmp")
            with temp.open("xb") as dst:
                pq.write_table(table, dst, compression="zstd", compression_level=3)
                dst.flush()
                os.fsync(dst.fileno())
            if path.exists():
                raise FileExistsError(path)
            os.rename(temp, path)
        return {"path": path.name, "rows": len(rows), "sha256": digest_file(path), "bytes": path.stat().st_size}


    def validate_prepared_parts(folder, manifest, expected_identity):
        if manifest.get("identity") != expected_identity or manifest.get("cache_key") != digest(encoded(expected_identity)):
            raise ValueError("Preparation source/selection/cleaning hash mismatch")
        if manifest.get("schema_version") != "georag-prepared-input-1.0":
            raise ValueError("Unsupported prepared input schema")
        for kind, schema in [("unique_parts", PREPARED_UNIQUE_SCHEMA), ("occurrence_parts", PREPARED_OCCURRENCE_SCHEMA)]:
            for part in manifest[kind]:
                path = prepared_path(folder, part["path"])
                if not path.is_file() or digest_file(path) != part["sha256"]:
                    raise ValueError("Preparation part missing or changed: " + part["path"])
                with pq.ParquetFile(path) as file:
                    if not file.schema_arrow.equals(schema) or file.metadata.num_rows != part["rows"]:
                        raise ValueError("Preparation schema/count mismatch")
        if sum(p["rows"] for p in manifest["occurrence_parts"]) != len(SAMPLE):
            raise ValueError("Preparation does not cover every current occurrence")
        if sum(p["rows"] for p in manifest["unique_parts"]) != manifest["counts"]["unique_texts"]:
            raise ValueError("Preparation unique count mismatch")


    class PreparedInput:
        def __init__(self, folder, manifest):
            self.folder = Path(folder)
            self.manifest = manifest
            self.key = manifest["cache_key"]
            self.ends = []
            total = 0
            for part in manifest["unique_parts"]:
                total += part["rows"]
                self.ends.append(total)
            self.cached_part = None
            self.cached_rows = None
            runtime = RUN_DIR / "runtime_cache"
            runtime.mkdir(exist_ok=True)
            self.db = sqlite3.connect(runtime / "prepared_lookup.sqlite")
            # A bounded page cache keeps the unique hash index from thrashing disk.
            self.db.execute("PRAGMA cache_size=-32768")
            self.db.execute("CREATE TABLE IF NOT EXISTS texts(hash TEXT PRIMARY KEY, original TEXT NOT NULL, uid INTEGER NOT NULL)")
            self.db.execute("DELETE FROM texts")
            self.db.execute("CREATE TABLE IF NOT EXISTS identity(value TEXT)")
            self.db.execute("DELETE FROM identity")
            self.db.execute("INSERT INTO identity VALUES (?)", (self.key,))
            expected = 0
            for row in self.iter_unique():
                if (row["unique_id"] != expected or row["text_sha256"] != digest(row["Text"].encode("utf-8"))
                        or len(row["Text"]) != len(row["Text_inference"])):
                    raise ValueError("Invalid prepared unique text/offset identity")
                normalization = json.loads(row["normalization_json"])
                if normalization["inference_sha256"] != digest(row["Text_inference"].encode("utf-8")):
                    raise ValueError("Prepared cleaned text hash mismatch")
                self.db.execute("INSERT INTO texts VALUES (?,?,?)", (row["text_sha256"], row["Text"], expected))
                expected += 1
            if expected != len(self):
                raise ValueError("Prepared unique IDs are not contiguous")
            self.db.commit()

        def __len__(self):
            return self.manifest["counts"]["unique_texts"]

        def __getitem__(self, key):
            if isinstance(key, slice):
                return [self[i] for i in range(*key.indices(len(self)))]
            if not 0 <= key < len(self):
                raise IndexError(key)
            part = bisect_right(self.ends, key)
            if self.cached_part != part:
                self.cached_rows = None
                path = prepared_path(self.folder, self.manifest["unique_parts"][part]["path"])
                with pq.ParquetFile(path) as file:
                    self.cached_rows = file.read().to_pylist()
                self.cached_part = part
            base = self.ends[part - 1] if part else 0
            return self.cached_rows[key - base]

        def iter_unique(self):
            for part in self.manifest["unique_parts"]:
                with pq.ParquetFile(prepared_path(self.folder, part["path"])) as file:
                    for batch in file.iter_batches(batch_size=PREPARATION_PART_ROWS):
                        yield from batch.to_pylist()

        def iter_occurrences(self):
            expected = 0
            for part in self.manifest["occurrence_parts"]:
                with pq.ParquetFile(prepared_path(self.folder, part["path"])) as file:
                    for batch in file.iter_batches(batch_size=PREPARATION_PART_ROWS):
                        for row in batch.to_pylist():
                            if row["sample_position"] != expected or not 0 <= row["unique_id"] < len(self):
                                raise ValueError("Prepared occurrence map has a gap/invalid unique ID")
                            expected += 1
                            yield row
            if expected != len(SAMPLE):
                raise ValueError("Prepared occurrence map does not cover the current corpus")

        def lookup(self, text):
            found = self.db.execute("SELECT original,uid FROM texts WHERE hash=?", (digest(text.encode("utf-8")),)).fetchone()
            if found is None or found[0] != text:
                raise ValueError("Text was not present in the verified prepared input")
            return self[found[1]]

        def close(self):
            self.cached_rows = None
            self.db.close()


    def prepare_input_once():
        global PREPARED, PREPARATION_REPORT
        started = time.perf_counter()
        if digest_file(RUN_DIR / "source.parquet") != SOURCE_META["sha256"]:
            raise ValueError("Frozen source bytes changed; preparation cache cannot be reused")
        if PREPARED is not None:
            PREPARED.close()
            PREPARED = None
        folder = RUN_DIR / "prepared"
        folder.mkdir(exist_ok=True)
        identity = preparation_identity()
        key = digest(encoded(identity))
        cleaning_seconds = saving_seconds = 0.0
        initial_cleaning_seconds = initial_saving_seconds = 0.0
        manifest_path = folder / "manifest.json"
        created = not manifest_path.exists()
        if created:
            runtime = RUN_DIR / "runtime_cache"
            runtime.mkdir(exist_ok=True)
            db = sqlite3.connect(runtime / "preparation_build.sqlite")
            db.execute("PRAGMA cache_size=-32768")
            unique_parts, occurrence_parts = [], []
            done = unique_count = eligible_unique = skipped_occurrences = 0
            parent = None
            try:
                db.execute("CREATE TABLE IF NOT EXISTS texts(hash TEXT PRIMARY KEY, original TEXT NOT NULL, uid INTEGER NOT NULL, eligible INTEGER NOT NULL)")
                db.execute("DELETE FROM texts")
                # Rebuild the transient index from immutable preparation checkpoints.
                checkpoints = sorted(folder.glob("progress_*.json"))
                for ordinal, checkpoint in enumerate(checkpoints):
                    saved = load_json(checkpoint)
                    if saved["identity"] != identity or saved["parent_sha256"] != parent or saved["sequence"] != ordinal:
                        raise ValueError("Preparation progress chain/source mismatch")
                    for name in ["unique_part", "occurrence_part"]:
                        item = saved[name]
                        path = prepared_path(folder, item["path"])
                        if digest_file(path) != item["sha256"]:
                            raise ValueError("Preparation progress part hash mismatch")
                    with pq.ParquetFile(prepared_path(folder, saved["unique_part"]["path"])) as file:
                        for row in file.read().to_pylist():
                            if row["unique_id"] != unique_count:
                                raise ValueError("Preparation progress unique ID gap")
                            db.execute("INSERT INTO texts VALUES (?,?,?,?)", (row["text_sha256"], row["Text"], row["unique_id"], row["eligible"]))
                            unique_count += 1
                            eligible_unique += row["eligible"]
                    unique_parts.append(saved["unique_part"])
                    occurrence_parts.append(saved["occurrence_part"])
                    done, skipped_occurrences = saved["processed_occurrences"], saved["skipped_occurrences"]
                    cleaning_seconds = saved.get("cleaning_seconds", 0.0)
                    saving_seconds = saved.get("saving_seconds", 0.0)
                    parent = digest_file(checkpoint)
                db.commit()
                initial_done = done
                initial_cleaning_seconds, initial_saving_seconds = cleaning_seconds, saving_seconds
                last = progress("preparation", done, len(SAMPLE), 0, started, 0) if done else 0
                for start in range(done, len(SAMPLE), PREPARATION_PART_ROWS):
                    ordinal = len(unique_parts)
                    uniques, occurrences = [], []
                    for p in SAMPLE[start:start + PREPARATION_PART_ROWS]:
                        text = p["Text"]
                        text_hash = digest(text.encode("utf-8"))
                        found = db.execute("SELECT original,uid,eligible FROM texts WHERE hash=?", (text_hash,)).fetchone()
                        if found:
                            if found[0] != text:
                                raise ValueError("Full-text hash collision")
                            uid, eligible = found[1], bool(found[2])
                        else:
                            cleaning_started = time.perf_counter()
                            cleaned, normalization = LEGACY_PREPARE_TEXT(text)
                            cleaning_seconds += time.perf_counter() - cleaning_started
                            uid, eligible = unique_count, not normalization["skipped_no_content"]
                            uniques.append({"unique_id": uid, "text_sha256": text_hash, "Text": text,
                                "Text_inference": cleaned, "normalization_json": encoded(normalization).decode("utf-8"),
                                "eligible": eligible, "representative_sample_position": p["sample_position"]})
                            db.execute("INSERT INTO texts VALUES (?,?,?,?)", (text_hash, text, uid, eligible))
                            unique_count += 1
                            eligible_unique += eligible
                        skipped_occurrences += not eligible
                        occurrences.append({k: p[k] for k in ["sample_position", "corpus_position", "Source", "Index", "start_document", "end_document"]} | {"unique_id": uid})
                    save_started = time.perf_counter()
                    unique_part = write_prepared_part(folder / f"unique_{ordinal:06d}.parquet", uniques, PREPARED_UNIQUE_SCHEMA)
                    occurrence_part = write_prepared_part(folder / f"occurrences_{ordinal:06d}.parquet", occurrences, PREPARED_OCCURRENCE_SCHEMA)
                    done = start + len(occurrences)
                    checkpoint = folder / f"progress_{ordinal:06d}.json"
                    saved = {"schema_version": "georag-preparation-progress-1.0", "identity": identity,
                        "sequence": ordinal, "parent_sha256": parent, "unique_part": unique_part,
                        "occurrence_part": occurrence_part, "processed_occurrences": done,
                        "skipped_occurrences": skipped_occurrences, "cleaning_seconds": cleaning_seconds,
                        "saving_seconds": saving_seconds + time.perf_counter() - save_started}
                    atomic_json(checkpoint, saved)
                    parent = digest_file(checkpoint)
                    db.commit()
                    saving_seconds += time.perf_counter() - save_started
                    unique_parts.append(unique_part)
                    occurrence_parts.append(occurrence_part)
                    last = progress("preparation", done, len(SAMPLE), 0, started, last)
                    checkpoint_safety_hook(force=(ordinal + 1) % PREPARATION_BACKUP_EVERY_PARTS == 0,
                                           reason="preparation_part_committed")
                manifest = {"schema_version": "georag-prepared-input-1.0", "identity": identity,
                    "cache_key": key, "unique_parts": unique_parts, "occurrence_parts": occurrence_parts,
                    "counts": {"occurrences": len(SAMPLE), "unique_texts": unique_count,
                        "unique_eligible": eligible_unique, "unique_skipped": unique_count - eligible_unique,
                        "duplicate_occurrences": len(SAMPLE) - unique_count, "skipped_occurrences": skipped_occurrences},
                    "producer_seconds": time.perf_counter() - started, "resumed_from_occurrence": initial_done,
                    "cleaning_seconds": cleaning_seconds, "saving_seconds": saving_seconds}
                atomic_json(manifest_path, manifest)
            finally:
                db.close()
        else:
            manifest = load_json(manifest_path)
        verify_started = time.perf_counter()
        validate_prepared_parts(folder, manifest, identity)
        PREPARED = PreparedInput(folder, manifest)
        PREPARATION_REPORT = {"status": "created" if created else "reused_verified", "cache_key": key,
            "counts": manifest["counts"], "seconds_current_session": time.perf_counter() - started,
            "original_producer_seconds": manifest["producer_seconds"],
            "current_session_cleaning_seconds": cleaning_seconds - initial_cleaning_seconds,
            "current_session_preparation_save_seconds": saving_seconds - initial_saving_seconds,
            "verification_and_index_seconds": time.perf_counter() - verify_started}
        STAGE_TIMINGS["preparation"] = PREPARATION_REPORT
        print("Prepared input verified:", json.dumps(PREPARATION_REPORT, indent=2))
        checkpoint_safety_hook(force=True, reason="prepared_input_ready")
        return PREPARED


    def prepare_inference_text(text):
        if PREPARED is None:
            raise RuntimeError("Run the one-time preparation cell before model processing")
        row = PREPARED.lookup(text)
        return row["Text_inference"], json.loads(row["normalization_json"])


    def process_uncached_occurrence(p, name, predict, measure, limit, prepared=None):
        started = time.perf_counter()
        row = prepared if prepared is not None else PREPARED.lookup(p["Text"])
        if row["Text"] != p["Text"]:
            raise ValueError("Prepared text differs from source occurrence")
        cleaned, normalization = row["Text_inference"], json.loads(row["normalization_json"])
        if not row["eligible"]:
            result = {"paragraph": p, "raw": [], "mentions": [], "duplicates_resolved": 0,
                "status_processing": "success", "errors": [], "at_processed": utc_now(), "processing_seconds": 0.0}
        else:
            result = PROCESS_PARAGRAPH_V9({**p, "Text": cleaned}, name, predict, measure, limit)
        result.update(paragraph=p, normalization=normalization, cache_hit=False,
                      inference_evaluated=row["eligible"], prepared_unique_id=row["unique_id"])
        for item in result["mentions"]:
            item["word_inference"] = item["word"]
            item["Text"] = p["Text"]
            item["text_sha256"] = row["text_sha256"]
            item["inference_text_sha256"] = normalization["inference_sha256"]
            a, b = item["start"], item["end"]
            if isinstance(a, int) and isinstance(b, int) and 0 <= a < b <= len(p["Text"]):
                item["word"] = p["Text"][a:b]
                item["start_document"], item["end_document"] = p["start_document"] + a, p["start_document"] + b
                item["source_span_contains_masked_characters"] = p["Text"][a:b] != cleaned[a:b]
                if item["status_validation"] == "valid":
                    assert_document_span(p, item["start_document"], item["end_document"], item["word"])
        result["processing_seconds"] = time.perf_counter() - started
        return result

    """Paced, resumable GitHub delivery and compression-aware checkpoint packets."""
    import zlib
    from email.utils import parsedate_to_datetime

    GITHUB_HTTP_STATE = {"writes": deque(), "last_write": None}
    GITHUB_HTTP_STATS = Counter()


    def github_clock():
        return globals().get("GITHUB_HTTP_MONOTONIC", time.monotonic)()


    def github_wall_time():
        return globals().get("GITHUB_HTTP_WALL_TIME", time.time)()


    def github_wait(seconds, reason):
        seconds = max(0.0, float(seconds))
        if not seconds:
            return
        print(f"GitHub waiting {seconds:.1f}s: {reason}", flush=True)
        GITHUB_HTTP_STATS["wait_seconds"] += seconds
        remaining = seconds
        while remaining > 0:
            step = min(30.0, remaining)
            globals().get("GITHUB_HTTP_SLEEP", time.sleep)(step)
            remaining -= step
        print("GitHub wait completed; checking/retrying the pending operation", flush=True)


    def github_pace_write():
        now = github_clock()
        writes = GITHUB_HTTP_STATE["writes"]
        while writes and now - writes[0] >= 3600:
            writes.popleft()
        delay = 0.0
        last = GITHUB_HTTP_STATE["last_write"]
        if last is not None:
            delay = max(delay, GITHUB_MIN_WRITE_INTERVAL_SECONDS - (now - last))
        minute = [stamp for stamp in writes if now - stamp < 60]
        if len(minute) >= GITHUB_MAX_WRITES_PER_MINUTE:
            delay = max(delay, minute[-GITHUB_MAX_WRITES_PER_MINUTE] + 60.1 - now)
        if len(writes) >= GITHUB_MAX_WRITES_PER_HOUR:
            delay = max(delay, writes[-GITHUB_MAX_WRITES_PER_HOUR] + 3600.1 - now)
        github_wait(delay, "serial write pacing/window budget")
        now = github_clock()
        writes.append(now)
        GITHUB_HTTP_STATE["last_write"] = now
        GITHUB_HTTP_STATS["write_attempts"] += 1


    class GitHubDeliveryError(RuntimeError):
        def __init__(self, detail, retryable=False):
            self.detail = detail
            self.retryable = retryable
            self.exhausted = False
            super().__init__(json.dumps(detail, ensure_ascii=False))


    CheckpointGitHubV12 = CheckpointGitHub


    class CheckpointGitHub(CheckpointGitHubV12):
        def _safe_message(self, value):
            value = str(value).replace(self.token, "[REDACTED]")
            value = re.sub(r"(?i)Bearer\s+\S+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9_]+",
                           "[REDACTED]", value)
            return value[:4000]

        def _record_error(self, error):
            GITHUB_HTTP_STATS["errors"] += 1
            root = Path(globals().get("RUN_DIR", OUTPUT_ROOT)) / "models" / "github_transport"
            root.mkdir(parents=True, exist_ok=True)
            path = root / f"http_error_{secrets.token_hex(8)}.json"
            detail = {
                "schema_version": "georag-github-http-diagnostic-1.0", "at_processed": utc_now(),
                "error": error.detail}
            temp = path.with_name(path.name + ".tmp")
            with temp.open("xb") as dst:
                dst.write(encoded(detail))
                dst.flush()
                os.fsync(dst.fileno())
            os.rename(temp, path)

        def _delay(self, error, attempt):
            detail = error.detail
            delay = min(GITHUB_RETRY_BASE_SECONDS * 2 ** attempt, GITHUB_RETRY_BACKOFF_CAP_SECONDS)
            headers = detail.get("response_headers", {})
            retry = headers.get("Retry-After")
            if retry:
                try:
                    instructed = float(retry)
                except (ValueError, TypeError):
                    try:
                        instructed = parsedate_to_datetime(retry).timestamp() - github_wall_time()
                    except (ValueError, TypeError, OverflowError):
                        instructed = 0.0
                delay = max(delay, instructed)
            if headers.get("X-RateLimit-Remaining") == "0":
                try:
                    delay = max(delay, float(headers["X-RateLimit-Reset"]) - github_wall_time() + 1)
                except (ValueError, TypeError, KeyError):
                    pass
            return max(0.0, delay)

        def api(self, method, suffix, payload=None, missing=False, raw=False):
            waited = 0.0
            for attempt in range(GITHUB_RETRY_ATTEMPTS):
                if method in {"POST", "PUT", "PATCH", "DELETE"}:
                    github_pace_write()
                request = urllib.request.Request("https://api.github.com/repos/" + self.repository + suffix,
                    data=encoded(payload) if payload is not None else None, method=method,
                    headers={"Authorization": "Bearer " + self.token,
                        "Accept": "application/vnd.github.raw+json" if raw else (
                            "application/vnd.github.object+json" if method == "GET" and suffix.startswith("/contents/")
                            else "application/vnd.github+json"), "X-GitHub-Api-Version": "2026-03-10",
                        "User-Agent": "GeoRAG-v13", "Content-Type": "application/json"})
                try:
                    response = self.open(request, timeout=180)
                    GITHUB_HTTP_STATS["successful_requests"] += 1
                    if raw:
                        return response
                    with response:
                        return json.load(response)
                except urllib.error.HTTPError as exc:
                    status = exc.code
                    if missing and status == 404:
                        exc.close()
                        return None
                    headers = {name: exc.headers.get(name) for name in (
                        "Retry-After", "X-RateLimit-Remaining", "X-RateLimit-Reset", "X-GitHub-Request-Id")}
                    headers = {name: self._safe_message(value) if value is not None else None
                               for name, value in headers.items()}
                    try:
                        body = exc.read(16384).decode("utf-8", errors="replace")
                        try:
                            parsed = json.loads(body)
                        except ValueError:
                            parsed = {"message": body}
                        message = self._safe_message(parsed.get("message", body)) if isinstance(parsed, dict) else self._safe_message(body)
                    finally:
                        exc.close()
                    limited = status == 429 or (status == 403 and (
                        headers.get("X-RateLimit-Remaining") == "0" or headers.get("Retry-After") is not None
                        or any(term in message.lower() for term in ("rate limit", "abuse", "temporarily blocked"))))
                    permission = status in {401, 403} and not limited
                    explanation = ("Rate limit: automatic header-aware wait/retry" if limited else
                        "Access denied: check token Contents: write, repository access, token expiry/SSO and branch policy; waiting does not grant permission"
                        if permission else "GitHub rejected the request")
                    error = GitHubDeliveryError({"status": status, "method": method,
                        "endpoint": suffix.split("?", 1)[0], "response_message": message,
                        "response_headers": headers, "rate_limited": limited, "explanation": explanation},
                        retryable=limited or status in {408, 409, 500, 502, 503, 504})
                    self._record_error(error)
                    # Uncertain mutating failures are reconciled by put(), not blindly repeated.
                    if not limited and method != "GET":
                        raise error from None
                except (urllib.error.URLError, TimeoutError, OSError) as exc:
                    error = GitHubDeliveryError({"status": None, "method": method,
                        "endpoint": suffix.split("?", 1)[0], "response_message": type(exc).__name__,
                        "response_headers": {}, "rate_limited": False,
                        "explanation": "Connection failed; local packet remains available"}, retryable=True)
                    self._record_error(error)
                    if method != "GET":
                        raise error from None
                if not error.retryable:
                    raise error from None
                delay = self._delay(error, attempt)
                if attempt + 1 >= GITHUB_RETRY_ATTEMPTS or waited + delay > GITHUB_RETRY_MAX_WAIT_SECONDS:
                    error.exhausted = True
                    raise error from None
                print(f"GitHub retry {attempt + 1}/{GITHUB_RETRY_ATTEMPTS - 1}: "
                      f"HTTP {error.detail['status']}; {error.detail['explanation']}", flush=True)
                github_wait(delay, "server rate limit/transient error")
                waited += delay
            raise AssertionError("Unreachable GitHub retry state")

        def put(self, path, remote):
            path = Path(path)
            remote = self.remote_path(remote)
            size = path.stat().st_size
            if not 0 < size < 25 * 1024**2:
                raise ValueError("Checkpoint ZIP/index must be smaller than 25 MiB")
            sha, before = self.blob_sha(path), sha_file_stream(path)
            waited = 0.0
            for attempt in range(GITHUB_RETRY_ATTEMPTS):
                try:
                    old = self.metadata(remote, missing=True)
                    if old is not None:
                        if old.get("type") != "file" or old.get("sha") != sha or old.get("size") != size:
                            raise FileExistsError("Different immutable remote checkpoint exists: " + remote)
                        GITHUB_HTTP_STATS["existing_files_reused"] += 1
                        saved = old
                    else:
                        result = self.api("PUT", "/contents/" + urllib.parse.quote(remote, safe="/"), {
                            "branch": self.branch, "message": "Add immutable GeoRAG checkpoint " + path.name,
                            "content": base64.b64encode(path.read_bytes()).decode("ascii")})
                        if result["content"]["sha"] != sha:
                            raise RuntimeError("Unexpected uploaded checkpoint Git blob SHA")
                        saved = self.metadata(remote, ref=result["commit"]["sha"])
                    if saved.get("sha") != sha or saved.get("size") != size or sha_file_stream(path) != before:
                        raise RuntimeError("Local/remote checkpoint hash/size verification failed")
                    return {"path": remote, "blob_sha": sha, "sha256": before, "bytes": size}
                except GitHubDeliveryError as error:
                    if not error.retryable or error.exhausted:
                        raise
                    delay = self._delay(error, attempt)
                    if attempt + 1 >= GITHUB_RETRY_ATTEMPTS or waited + delay > GITHUB_RETRY_MAX_WAIT_SECONDS:
                        raise
                    github_wait(delay, "uncertain write; next attempt first checks existing remote SHA/size")
                    waited += delay
            raise AssertionError("Unreachable immutable upload retry state")


    CheckpointPacketsV12 = CheckpointPackets


    class CheckpointPackets(CheckpointPacketsV12):
        """Same ZIP contract; journal a complete plan, upload each verified ZIP immediately."""
        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.active_plan = None

        def _find_plan(self):
            for path in sorted(self.output.glob("cycle_plan_*.json")):
                plan = load_json(path)
                if plan["base_sequence"] == self.sequence and plan["base_parent"] == self.parent:
                    if plan["ids"] != self.ids:
                        raise ValueError("Local pending backup plan belongs to another run/source")
                    return plan
            return None

        def _make_plan(self, reason):
            observed, groups, group = {}, [], []
            compressed = raw_bytes = 0
            for path in self.candidate_files():
                name = path.relative_to(self.root).as_posix()
                checkpoint_relative_path(name)
                stat = path.stat()
                prior = self.covered.get(name)
                if prior:
                    if (stat.st_size, stat.st_mtime_ns) != tuple(prior[:2]):
                        raise RuntimeError("Previously backed-up immutable checkpoint changed: " + name)
                    continue
                if not stat.st_size:
                    raise ValueError("Empty checkpoint file")
                file_hash = sha_file_stream(path)
                observed[name] = [stat.st_size, stat.st_mtime_ns, file_hash]
                with path.open("rb") as src:
                    for offset in range(0, stat.st_size, self.segment_bytes):
                        content = src.read(min(self.segment_bytes, stat.st_size - offset))
                        compressor = zlib.compressobj(1, zlib.DEFLATED, -15)
                        compressed_size = len(compressor.compress(content)) + len(compressor.flush())
                        segment = {"path": name, "offset": offset, "bytes": len(content),
                            "file_bytes": stat.st_size, "file_sha256": file_hash,
                            "segment_sha256": hashlib.sha256(content).hexdigest()}
                        # Account conservatively for ZIP headers and the JSON segment descriptor.
                        cost = compressed_size + len(encoded(segment)) + 256
                        if group and (compressed + cost > self.packet_bytes or
                                raw_bytes + len(content) > BACKUP_RAW_PACKET_MIB * 1024**2 or len(group) >= 1024):
                            groups.append(group)
                            group, compressed, raw_bytes = [], 0, 0
                        group.append(segment)
                        compressed += cost
                        raw_bytes += len(content)
            if group:
                groups.append(group)
            if not groups:
                return None
            plan = {"schema_version": "georag-checkpoint-upload-plan-1.0", "ids": self.ids,
                "base_sequence": self.sequence, "base_parent": self.parent, "cycle_id": secrets.token_hex(8),
                "at_created": utc_now(), "reason": reason, "observed": observed, "groups": groups,
                "resume_config": copy.deepcopy(self.config), "versions": copy.deepcopy(VERSIONS)}
            atomic_json(self.output / f"cycle_plan_{self.sequence + 1:08d}_{plan['cycle_id']}.json", plan)
            return plan

        def backup(self, reason, download):
            started = time.perf_counter()
            # Finish pending deliveries before considering any new files/cycle.
            self.deliver_pending(download)
            if self.active_plan is None:
                self.active_plan = self._find_plan() or self._make_plan(reason)
            plan = self.active_plan
            if plan is None:
                self.last_cycle = time.monotonic()
                return []
            for name, saved in plan["observed"].items():
                stat = (self.root / name).stat()
                if (stat.st_size, stat.st_mtime_ns) != tuple(saved[:2]):
                    raise RuntimeError("Pending checkpoint source file changed: " + name)
            parent, created = plan["base_parent"], []
            for number, segments in enumerate(plan["groups"]):
                sequence = plan["base_sequence"] + number + 1
                path = self.output / f"{RUN_NAME}_cp_{sequence:06d}_{plan['cycle_id']}.zip"
                manifest = {"schema_version": PACKET_SCHEMA, **self.ids, "sequence": sequence,
                    "parent_sha256": parent, "cycle_id": plan["cycle_id"],
                    "cycle_complete": number == len(plan["groups"]) - 1, "reason": plan["reason"],
                    "resume_config": plan["resume_config"], "versions": plan["versions"], "at_created": plan["at_created"],
                    "segments": [{**s, "entry": f"segment_{i:06d}.bin"} for i, s in enumerate(segments)]}
                if path.exists():
                    with zipfile.ZipFile(path) as archive:
                        if json.loads(archive.read(PACKET_MANIFEST)) != manifest:
                            raise ValueError("Pending local ZIP differs from the immutable upload plan")
                        expected_entries = {PACKET_MANIFEST} | {s["entry"] for s in manifest["segments"]}
                        if len(archive.namelist()) != len(expected_entries) or set(archive.namelist()) != expected_entries:
                            raise ValueError("Unexpected or duplicate ZIP entries in pending checkpoint")
                        for segment in manifest["segments"]:
                            if archive.getinfo(segment["entry"]).file_size != segment["bytes"]:
                                raise ValueError("Pending ZIP segment length changed")
                            with archive.open(segment["entry"]) as src:
                                h = hashlib.sha256()
                                for block in iter(lambda: src.read(1024 * 1024), b""):
                                    h.update(block)
                            if h.hexdigest() != segment["segment_sha256"]:
                                raise ValueError("Pending ZIP segment content hash changed")
                else:
                    temp = path.with_name(path.name + ".tmp")
                    # An interrupted .tmp is never committed or uploaded. Rebuild it from the plan.
                    with zipfile.ZipFile(temp, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
                        for segment in manifest["segments"]:
                            h, remaining = hashlib.sha256(), segment["bytes"]
                            with (self.root / segment["path"]).open("rb") as src, archive.open(segment["entry"], "w") as dst:
                                src.seek(segment["offset"])
                                while remaining:
                                    block = src.read(min(1024 * 1024, remaining))
                                    if not block:
                                        raise IOError("Checkpoint source shortened while packing")
                                    h.update(block)
                                    dst.write(block)
                                    remaining -= len(block)
                            if h.hexdigest() != segment["segment_sha256"]:
                                raise ValueError("Source bytes changed since the upload plan")
                        archive.writestr(PACKET_MANIFEST, json.dumps(manifest, ensure_ascii=False, allow_nan=False))
                    with zipfile.ZipFile(temp) as archive:
                        if archive.testzip() is not None:
                            raise IOError("Checkpoint ZIP integrity check failed")
                    if temp.stat().st_size >= 25 * 1024**2:
                        raise ValueError("Packed checkpoint exceeds the GitHub transport size bound")
                    os.rename(temp, path)
                parent = sha_file_stream(path)
                receipt = {"path": str(path), "sha256": parent, "sequence": sequence,
                    "cycle_complete": manifest["cycle_complete"], "status": "server_file_created",
                    "bytes": path.stat().st_size, "reason": plan["reason"]}
                # Replay uses the same name/bytes; the transport checks remote SHA/size before writing.
                self.pending.append(receipt)
                self.receipts.append(receipt)
                print("CHECKPOINT PACKET VERIFIED:", path.name, "uploading now", flush=True)
                self.deliver_pending(download)
                created.append(receipt)
            self.covered.update({name: tuple(value) for name, value in plan["observed"].items()})
            self.sequence = plan["base_sequence"] + len(plan["groups"])
            self.parent = parent
            self.active_plan = None
            self.last_cycle = time.monotonic()
            GITHUB_HTTP_STATS["backup_seconds"] += time.perf_counter() - started
            GITHUB_HTTP_STATS["packets_delivered"] += len(created)
            print("Checkpoint cycle delivered:", len(created), "ZIP(s); recovery index can now be published", flush=True)
            return created

    """Single-pass staging and standard ZIP assembly from already deflated segments.

    Remote packet/index contracts remain v3.1/v1.0. All canonical files and raw data
    remain included; caches below are local, derived, and never required to restore.
    """
    import io
    import struct

    FRESH_CHECKPOINT_FILES = {}


    def backup_signature(path):
        stat = Path(path).stat()
        return stat.st_size, stat.st_mtime_ns, stat.st_ctime_ns


    CheckpointGitHubV13 = CheckpointGitHub


    class CheckpointGitHub(CheckpointGitHubV13):
        def publish_index(self, manager):
            super().publish_index(manager)
            if self.published_head == manager.parent and hasattr(manager, "release_confirmed_staging"):
                manager.release_confirmed_staging()

        def _local_identity(self, path):
            path = Path(path)
            signature = backup_signature(path)
            key = str(path.resolve())
            cache = self.__dict__.setdefault("_local_hash_cache", {})
            if key not in cache or cache[key][0] != signature:
                git = hashlib.sha1(f"blob {signature[0]}\0".encode())
                sha = hashlib.sha256()
                with path.open("rb") as src:
                    for block in iter(lambda: src.read(1024 * 1024), b""):
                        git.update(block)
                        sha.update(block)
                        GITHUB_HTTP_STATS["local_hash_bytes_read"] += len(block)
                if backup_signature(path) != signature:
                    raise RuntimeError("Checkpoint changed while its identity was calculated")
                cache[key] = signature, git.hexdigest(), sha.hexdigest()
                GITHUB_HTTP_STATS["local_hash_passes"] += 1
            else:
                GITHUB_HTTP_STATS["local_hash_cache_hits"] += 1
            return cache[key]

        def put(self, path, remote):
            path = Path(path)
            remote = self.remote_path(remote)
            signature, sha, sha256 = self._local_identity(path)
            size = signature[0]
            if not 0 < size < 25 * 1024**2:
                raise ValueError("Checkpoint ZIP/index must be smaller than 25 MiB")
            verified = self.__dict__.setdefault("_verified_uploads", {})
            key = self.branch, remote
            if key in verified and verified[key][0] == (sha, sha256, size):
                GITHUB_HTTP_STATS["verified_metadata_checks_avoided"] += 1
                GITHUB_HTTP_STATS["existing_files_reused"] += 1
                return dict(verified[key][1])
            waited = 0.0
            for attempt in range(GITHUB_RETRY_ATTEMPTS):
                fresh = attempt == 0 and FRESH_CHECKPOINT_FILES.pop(str(path.resolve()), None) == signature
                try:
                    # Verification caches never cross kernels.
                    # A newly built nonce-named packet can use create-only PUT (no
                    # update SHA): GitHub cannot overwrite an existing file this way.
                    # All resumed/uncertain attempts still check the server first.
                    old = None if fresh else self.metadata(remote, missing=True)
                    if old is not None:
                        if not isinstance(old, dict) or old.get("type") != "file" or old.get("sha") != sha or old.get("size") != size:
                            raise FileExistsError("Different immutable remote checkpoint exists: " + remote)
                        GITHUB_HTTP_STATS["existing_files_reused"] += 1
                        saved = old
                    else:
                        result = self.api("PUT", "/contents/" + urllib.parse.quote(remote, safe="/"), {
                            "branch": self.branch, "message": "Add immutable GeoRAG checkpoint " + path.name,
                            "content": base64.b64encode(path.read_bytes()).decode("ascii")})
                        if result["content"]["sha"] != sha:
                            raise RuntimeError("Uploaded checkpoint Git blob SHA differs from the local bytes")
                        saved = self.metadata(remote, ref=result["commit"]["sha"])
                        if fresh:
                            GITHUB_HTTP_STATS["fresh_create_metadata_checks_avoided"] += 1
                    if saved.get("sha") != sha or saved.get("size") != size or backup_signature(path) != signature:
                        raise RuntimeError("Checkpoint changed or remote SHA/size verification failed")
                    receipt = {"path": remote, "blob_sha": sha, "sha256": sha256, "bytes": size}
                    verified[key] = (sha, sha256, size), dict(receipt)
                    return receipt
                except GitHubDeliveryError as error:
                    if fresh and error.detail.get("status") == 422:
                        old = self.metadata(remote, missing=True)
                        if old is not None:
                            if old.get("type") != "file" or old.get("sha") != sha or old.get("size") != size:
                                raise FileExistsError("Create-only checkpoint path already contains different bytes: " + remote) from None
                            if backup_signature(path) != signature:
                                raise RuntimeError("Local checkpoint changed during create-only reconciliation") from None
                            receipt = {"path": remote, "blob_sha": sha, "sha256": sha256, "bytes": size}
                            verified[key] = (sha, sha256, size), dict(receipt)
                            GITHUB_HTTP_STATS["existing_files_reused"] += 1
                            return receipt
                    if not error.retryable or error.exhausted:
                        raise
                    delay = self._delay(error, attempt)
                    if attempt + 1 >= GITHUB_RETRY_ATTEMPTS or waited + delay > GITHUB_RETRY_MAX_WAIT_SECONDS:
                        raise
                    github_wait(delay, "uncertain write; checking remote SHA/size before another write")
                    waited += delay
            raise AssertionError("Unreachable upload retry state")


    class DeflatedCheckpointZIP:
        """Write the standard ZIP32 headers plus cached raw-DEFLATE payloads.

        Member names are generated ASCII, sizes are known, data is bounded below
        ZIP64 limits, and no descriptors/encryption/extensions are used. zipfile
        performs the independent CRC/length validation before a ZIP is committed.
        """
        def __init__(self, handle, at_created):
            stamp = datetime.fromisoformat(at_created.replace("Z", "+00:00"))
            if not 1980 <= stamp.year <= 2107:
                raise ValueError("Timestamp outside standard ZIP date range")
            self.dos_time = (stamp.hour << 11) | (stamp.minute << 5) | (stamp.second // 2)
            self.dos_date = ((stamp.year - 1980) << 9) | (stamp.month << 5) | stamp.day
            self.handle, self.offset, self.central = handle, 0, []
            self.sha = hashlib.sha256()

        def write(self, value):
            self.handle.write(value)
            self.sha.update(value)
            self.offset += len(value)

        def member(self, name, source, size, crc, compressed_size, compressed_sha256):
            name = name.encode("ascii")
            if not 0 <= size < 2**32 or not 0 <= compressed_size < 2**32 or self.offset >= 2**32:
                raise ValueError("Checkpoint exceeds ZIP32 limits")
            offset = self.offset
            self.write(struct.pack("<IHHHHHIIIHH", 0x04034B50, 20, 0, 8, self.dos_time,
                self.dos_date, crc, compressed_size, size, len(name), 0))
            self.write(name)
            remaining, digest_compressed = compressed_size, hashlib.sha256()
            while remaining:
                block = source.read(min(1024 * 1024, remaining))
                if not block:
                    raise IOError("Truncated staged compressed segment")
                self.write(block)
                digest_compressed.update(block)
                remaining -= len(block)
            if source.read(1) or digest_compressed.hexdigest() != compressed_sha256:
                raise ValueError("Staged compressed segment size/hash mismatch")
            self.central.append((name, size, crc, compressed_size, offset))

        def finish(self):
            start = self.offset
            if len(self.central) >= 65535:
                raise ValueError("Too many checkpoint ZIP members")
            for name, size, crc, compressed_size, offset in self.central:
                self.write(struct.pack("<IHHHHHHIIIHHHHHII", 0x02014B50, 20, 20, 0, 8,
                    self.dos_time, self.dos_date, crc, compressed_size, size,
                    len(name), 0, 0, 0, 0, 0, offset))
                self.write(name)
            length = self.offset - start
            self.write(struct.pack("<IHHHHIIH", 0x06054B50, 0, 0, len(self.central),
                len(self.central), length, start, 0))
            return self.sha.hexdigest()


    class CheckpointCompressedSlice:
        def __init__(self, handle, offset, length):
            self.handle, self.remaining = handle, length
            handle.seek(offset)

        def read(self, length=-1):
            length = self.remaining if length < 0 else min(length, self.remaining)
            value = self.handle.read(length)
            self.remaining -= len(value)
            return value


    CheckpointPacketsV13 = CheckpointPackets


    class CheckpointPackets(CheckpointPacketsV13):
        def release_confirmed_staging(self):
            if not RELEASE_CONFIRMED_COMPRESSION_CACHE or self.active_plan is not None:
                return
            paths = self.__dict__.pop("_completed_staging_paths", set())
            for relative in paths:
                if not re.fullmatch(r"compressed_segments/spool_[0-9a-f]{16}\.deflate", relative):
                    raise ValueError("Unexpected derived staging cleanup path")
                # Individual derived files only; never a source/ZIP/directory removal.
                path = prepared_path(self.output, relative)
                if path.exists():
                    size = path.stat().st_size
                    path.unlink()
                    GITHUB_HTTP_STATS["confirmed_compression_cache_bytes_released"] += size
            self.__dict__.pop("_compression_metadata", None)

        def _find_plan(self):
            # Older completed plans may be large: filter by next sequence before reading.
            paths = sorted(self.output.glob(f"cycle_plan_{self.sequence + 1:08d}_*.json"))
            matching = []
            for path in paths:
                plan = load_json(path)
                if plan["base_sequence"] == self.sequence and plan["base_parent"] == self.parent:
                    if plan["ids"] != self.ids:
                        raise ValueError("Pending backup plan belongs to a different run/source")
                    matching.append(plan)
            if len(matching) > 1:
                raise ValueError("Multiple pending backup cycles at the same chain position")
            return matching[0] if matching else None

        def _stage(self, content):
            digest_raw = hashlib.sha256(content).hexdigest()
            cache = self._compression_metadata
            if digest_raw in cache:
                GITHUB_HTTP_STATS["compression_cache_hits"] += 1
                return cache[digest_raw]
            if self._stage_handle is None:
                folder = self.output / "compressed_segments"
                folder.mkdir(exist_ok=True)
                self._stage_path = folder / f"spool_{secrets.token_hex(8)}.deflate"
                self._stage_handle = self._stage_path.open("xb")
            compressor = zlib.compressobj(1, zlib.DEFLATED, -15)
            compressed = compressor.compress(content) + compressor.flush()
            offset = self._stage_handle.tell()
            self._stage_handle.write(compressed)
            GITHUB_HTTP_STATS["raw_bytes_compressed"] += len(content)
            GITHUB_HTTP_STATS["segments_compressed"] += 1
            receipt = {"schema_version": "georag-deflate-stage-1.0", "raw_sha256": digest_raw,
                "raw_bytes": len(content), "crc32": zlib.crc32(content), "compressed_bytes": len(compressed),
                "compressed_sha256": hashlib.sha256(compressed).hexdigest(),
                "cache_path": self._stage_path.relative_to(self.output).as_posix(), "cache_offset": offset}
            cache[digest_raw] = receipt
            return receipt

        def _staged_path(self, segment):
            staged = segment["staged"]
            path = prepared_path(self.output, staged["cache_path"])
            known = self.__dict__.setdefault("_repaired_spools", {})
            if path.exists() and backup_signature(path) in (tuple(staged["cache_signature"]), known.get(staged["cache_path"])):
                return path
            # Rebuild the one derived spool from retained canonical segments. No NER.
            entries = {}
            for group in self.active_plan["groups"]:
                for item in group:
                    if item["staged"]["cache_path"] == staged["cache_path"]:
                        entries.setdefault(item["staged"]["cache_offset"], item)
            temp = path.with_name(path.name + ".repair.tmp")
            with temp.open("wb") as dst:
                for offset, item in sorted(entries.items()):
                    if dst.tell() != offset:
                        raise ValueError("Invalid compressed spool offsets in pending plan")
                    with prepared_path(self.root, item["path"]).open("rb") as src:
                        src.seek(item["offset"])
                        raw = src.read(item["bytes"])
                    if len(raw) != item["bytes"] or hashlib.sha256(raw).hexdigest() != item["segment_sha256"]:
                        raise ValueError("Canonical raw checkpoint differs; cannot rebuild spool")
                    compressor = zlib.compressobj(1, zlib.DEFLATED, -15)
                    compressed = compressor.compress(raw) + compressor.flush()
                    meta = item["staged"]
                    if len(compressed) != meta["compressed_bytes"] or hashlib.sha256(compressed).hexdigest() != meta["compressed_sha256"]:
                        raise ValueError("Rebuilt compressed bytes differ from the frozen plan")
                    dst.write(compressed)
                    GITHUB_HTTP_STATS["derived_cache_segments_rebuilt"] += 1
                dst.flush()
                os.fsync(dst.fileno())
            os.replace(temp, path)
            known[staged["cache_path"]] = backup_signature(path)
            print("Rebuilt derived compression spool from verified raw checkpoints", flush=True)
            return path

        def _make_plan(self, reason):
            self._compression_metadata = {}
            self._stage_handle = None
            try:
                plan = self._make_plan_core(reason)
            finally:
                if self._stage_handle is not None:
                    self._stage_handle.flush()
                    os.fsync(self._stage_handle.fileno())
                    self._stage_handle.close()
                    self._stage_handle = None
            if plan is None:
                return None
            signature = list(backup_signature(self._stage_path))
            for metadata in self._compression_metadata.values():
                metadata["cache_signature"] = signature
            atomic_json(self.output / f"cycle_plan_{self.sequence + 1:08d}_{plan['cycle_id']}.json", plan)
            return plan

        def _make_plan_core(self, reason):
            started = time.perf_counter()
            observed, groups, group = {}, [], []
            compressed_size = raw_size = 0
            for path in self.candidate_files():
                name = path.relative_to(self.root).as_posix()
                checkpoint_relative_path(name)
                stat = path.stat()
                prior = self.covered.get(name)
                if prior:
                    if (stat.st_size, stat.st_mtime_ns) != tuple(prior[:2]):
                        raise RuntimeError("Previously backed-up immutable checkpoint changed: " + name)
                    continue
                if not stat.st_size:
                    raise ValueError("Empty checkpoint file")
                hasher, file_segments = hashlib.sha256(), []
                with path.open("rb") as src:
                    offset = 0
                    while True:
                        content = src.read(self.segment_bytes)
                        if not content:
                            break
                        hasher.update(content)
                        GITHUB_HTTP_STATS["source_bytes_read_for_packing"] += len(content)
                        staged = self._stage(content)
                        file_segments.append({"path": name, "offset": offset, "bytes": len(content),
                            "file_bytes": stat.st_size, "segment_sha256": staged["raw_sha256"],
                            "staged": staged})
                        offset += len(content)
                if offset != stat.st_size or (path.stat().st_size, path.stat().st_mtime_ns) != (stat.st_size, stat.st_mtime_ns):
                    raise RuntimeError("Checkpoint source changed during staging")
                digest_file = hasher.hexdigest()
                observed[name] = [stat.st_size, stat.st_mtime_ns, digest_file]
                for segment in file_segments:
                    segment["file_sha256"] = digest_file
                    cost = segment["staged"]["compressed_bytes"] + len(encoded({k:v for k,v in segment.items() if k != "staged"})) + 256
                    if group and (compressed_size + cost > self.packet_bytes or
                            raw_size + segment["bytes"] > BACKUP_RAW_PACKET_MIB * 1024**2 or len(group) >= 1024):
                        groups.append(group)
                        group, compressed_size, raw_size = [], 0, 0
                    group.append(segment)
                    compressed_size += cost
                    raw_size += segment["bytes"]
            GITHUB_HTTP_STATS["staging_seconds"] += time.perf_counter() - started
            if group:
                groups.append(group)
            if not groups:
                GITHUB_HTTP_STATS["empty_backup_cycles_skipped"] += 1
                return None
            plan = {"schema_version": "georag-checkpoint-upload-plan-2.0", "ids": self.ids,
                "base_sequence": self.sequence, "base_parent": self.parent, "cycle_id": secrets.token_hex(8),
                "at_created": utc_now(), "reason": reason, "observed": observed, "groups": groups,
                "resume_config": copy.deepcopy(self.config), "versions": copy.deepcopy(VERSIONS)}
            return plan

        def _packet(self, plan, number, parent):
            started = time.perf_counter()
            sequence = plan["base_sequence"] + number + 1
            segments = plan["groups"][number]
            path = self.output / f"{RUN_NAME}_cp_{sequence:06d}_{plan['cycle_id']}.zip"
            manifest = {"schema_version": PACKET_SCHEMA, **self.ids, "sequence": sequence,
                "parent_sha256": parent, "cycle_id": plan["cycle_id"],
                "cycle_complete": number == len(plan["groups"]) - 1, "reason": plan["reason"],
                "resume_config": plan["resume_config"], "versions": plan["versions"], "at_created": plan["at_created"],
                "segments": [{**{k:v for k,v in item.items() if k != "staged"}, "entry": f"segment_{i:06d}.bin"}
                             for i, item in enumerate(segments)]}
            local_receipt = self.output / f"local_packet_{sequence:08d}_{plan['cycle_id']}.json"
            expected_manifest = hashlib.sha256(encoded(manifest)).hexdigest()
            if path.exists() and local_receipt.exists():
                saved = load_json(local_receipt)
                if saved["manifest_sha256"] == expected_manifest and tuple(saved["signature"]) == backup_signature(path):
                    GITHUB_HTTP_STATS["local_packet_revalidation_avoided"] += 1
                    return path, saved["sha256"], manifest
            if path.exists():
                with zipfile.ZipFile(path) as archive:
                    if json.loads(archive.read(PACKET_MANIFEST)) != manifest:
                        raise ValueError("Pending ZIP differs from its immutable plan")
                    entries = {PACKET_MANIFEST} | {s["entry"] for s in manifest["segments"]}
                    if set(archive.namelist()) != entries or len(archive.namelist()) != len(entries):
                        raise ValueError("Unexpected or duplicate pending ZIP entries")
                    for item in manifest["segments"]:
                        if archive.getinfo(item["entry"]).file_size != item["bytes"]:
                            raise ValueError("Pending ZIP segment size differs")
                        with archive.open(item["entry"]) as src:
                            h = hashlib.sha256()
                            for block in iter(lambda: src.read(1024 * 1024), b""):
                                h.update(block)
                        if h.hexdigest() != item["segment_sha256"]:
                            raise ValueError("Pending ZIP segment hash differs")
                sha = sha_file_stream(path)
            else:
                temp = path.with_name(path.name + ".tmp")
                with temp.open("wb") as dst:
                    writer = DeflatedCheckpointZIP(dst, plan["at_created"])
                    with ExitStack() as stack:
                        handles = {}
                        for descriptor, segment in zip(manifest["segments"], segments):
                            staged = segment["staged"]
                            cache = self._staged_path(segment)
                            if cache not in handles:
                                handles[cache] = stack.enter_context(cache.open("rb"))
                            src = CheckpointCompressedSlice(handles[cache], staged["cache_offset"], staged["compressed_bytes"])
                            writer.member(descriptor["entry"], src, staged["raw_bytes"], staged["crc32"],
                                          staged["compressed_bytes"], staged["compressed_sha256"])
                    content = encoded(manifest)
                    compressor = zlib.compressobj(1, zlib.DEFLATED, -15)
                    compressed = compressor.compress(content) + compressor.flush()
                    writer.member(PACKET_MANIFEST, io.BytesIO(compressed), len(content), zlib.crc32(content),
                                  len(compressed), hashlib.sha256(compressed).hexdigest())
                    sha = writer.finish()
                    dst.flush()
                    os.fsync(dst.fileno())
                if temp.stat().st_size >= 25 * 1024**2:
                    raise ValueError("Checkpoint ZIP exceeds transport size bound")
                with zipfile.ZipFile(temp) as archive:
                    if archive.testzip() is not None or json.loads(archive.read(PACKET_MANIFEST)) != manifest:
                        raise ValueError("Assembled ZIP failed independent zipfile validation")
                os.rename(temp, path)
                GITHUB_HTTP_STATS["zip_packets_built"] += 1
                FRESH_CHECKPOINT_FILES[str(path.resolve())] = backup_signature(path)
            receipt = {"schema_version": "georag-local-packet-verification-1.0", "sha256": sha,
                "manifest_sha256": expected_manifest, "signature": list(backup_signature(path))}
            if local_receipt.exists():
                if load_json(local_receipt) != receipt:
                    raise ValueError("Immutable local ZIP verification receipt differs")
            else:
                atomic_json(local_receipt, receipt)
            GITHUB_HTTP_STATS["zip_assembly_validation_seconds"] += time.perf_counter() - started
            return path, sha, manifest

        def backup(self, reason, download):
            started = time.perf_counter()
            self.deliver_pending(download)
            if self.active_plan is None:
                self.active_plan = self._find_plan() or self._make_plan(reason)
            plan = self.active_plan
            if plan is None:
                self.last_cycle = time.monotonic()
                return []
            if plan["schema_version"] == "georag-checkpoint-upload-plan-1.0":
                # Incomplete v13 local cycles retain their original names and bytes.
                return super().backup(reason, download)
            if plan["schema_version"] != "georag-checkpoint-upload-plan-2.0":
                raise ValueError("Unsupported local upload plan version")
            for name, saved in plan["observed"].items():
                stat = (self.root / name).stat()
                if (stat.st_size, stat.st_mtime_ns) != tuple(saved[:2]):
                    raise RuntimeError("Pending immutable backup source changed")
            parent, created = plan["base_parent"], []
            for number in range(len(plan["groups"])):
                path, parent, manifest = self._packet(plan, number, parent)
                print("CHECKPOINT PACKET VERIFIED:", path.name, "uploading/reconciling now", flush=True)
                result = download(path)
                receipt = {"path": str(path), "sha256": parent, "sequence": manifest["sequence"],
                    "cycle_complete": manifest["cycle_complete"], "bytes": path.stat().st_size,
                    "reason": plan["reason"], "status": result["status"], "delivery": result}
                receipt_path = self.output / f"delivery_{manifest['sequence']:08d}_{plan['cycle_id']}.json"
                if receipt_path.exists():
                    if load_json(receipt_path) != receipt:
                        raise ValueError("Immutable packet delivery receipt differs")
                else:
                    atomic_json(receipt_path, receipt)
                created.append(receipt)
            self.covered.update({name: tuple(value) for name, value in plan["observed"].items()})
            self.sequence = plan["base_sequence"] + len(plan["groups"])
            self.parent = parent
            self.active_plan = None
            self.last_cycle = time.monotonic()
            self.__dict__.setdefault("_completed_staging_paths", set()).update(
                s["staged"]["cache_path"] for group in plan["groups"] for s in group)
            seen = {r["sequence"] for r in self.receipts}
            self.receipts.extend(r for r in created if r["sequence"] not in seen)
            GITHUB_HTTP_STATS["backup_seconds"] += time.perf_counter() - started
            GITHUB_HTTP_STATS["packets_delivered"] += len(created)
            print("Checkpoint cycle delivered:", len(created), "ZIP(s); all data retained, ready to publish index", flush=True)
            return created

"""Lossless Parquet payload storage; no changes to NER decisions or model calls.

Large model JSON files become small, versioned pointers. Original JSON bytes are
stored in a Zstandard-compressed binary Parquet column, without float conversion.
This is a checkpoint container, not the nine-column consumer Parquet schema.
"""
from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import secrets
import shutil
import time
import urllib.parse

import pyarrow as pa
import pyarrow.parquet as pq

POINTER_SCHEMA = "georag-json-parquet-pointer-1.0"
PAYLOAD_SCHEMA = "georag-lossless-checkpoint-parquet-1.0"
PAYLOAD_ARROW = pa.schema([
    ("logical_path", pa.string()), ("payload_sha256", pa.string()),
    ("payload_bytes", pa.int64()), ("payload_json", pa.large_binary()),
], metadata={b"schema_version": PAYLOAD_SCHEMA.encode()})


def compact_sha(data):
    return hashlib.sha256(data).hexdigest()


def compact_safe_path(root, relative):
    p = PurePosixPath(relative)
    if (not isinstance(relative, str) or "\\" in relative or ":" in relative
            or p.is_absolute() or not p.parts or ".." in p.parts):
        raise ValueError("Unsafe compact payload path")
    root = Path(root).resolve()
    result = (root / relative).resolve()
    if not result.is_relative_to(root):
        raise ValueError("Compact payload escapes run directory")
    return result


class CompactJSONStore:
    def __init__(self, root):
        self.root = Path(root).resolve()

    @staticmethod
    def is_pointer(obj):
        return isinstance(obj, dict) and obj.get("storage_schema") == POINTER_SCHEMA

    def logical_bytes(self, path):
        path = Path(path)
        value = path.read_bytes()
        # Tiny pointers only; avoid parsing large original payloads twice.
        if len(value) < 4096:
            obj = json.loads(value)
            if self.is_pointer(obj):
                target = compact_safe_path(self.root, obj["payload_path"])
                logical = path.resolve().relative_to(self.root)
                expected_prefix = ("models/compact_storage/payloads/" if logical.parts[0] == "models"
                    else "/".join(logical.parts[:2]) + "/compact_payloads/")
                if not obj["payload_path"].startswith(expected_prefix):
                    raise ValueError("Unexpected compact payload namespace")
                with pq.ParquetFile(target) as pf:
                    if not pf.schema_arrow.equals(PAYLOAD_ARROW, check_metadata=True) or pf.metadata.num_rows != 1:
                        raise ValueError("Compact Parquet schema/row-count mismatch")
                    row = pf.read().to_pylist()[0]
                actual = path.resolve().relative_to(self.root).as_posix()
                if (row["logical_path"] != actual or row["payload_sha256"] != obj["payload_sha256"]
                        or row["payload_bytes"] != obj["payload_bytes"]):
                    raise ValueError("Compact pointer/Parquet identity mismatch")
                value = row["payload_json"]
                if len(value) != obj["payload_bytes"] or compact_sha(value) != obj["payload_sha256"]:
                    raise ValueError("Compact payload failed exact-byte SHA256 validation")
        return value

    def load(self, path):
        return json.loads(self.logical_bytes(path))

    def write(self, path, payload, *, replace_original=False):
        """Commit payload first, verify bytes, then atomically commit its pointer.

        At most one source checkpoint and its compressed replacement are held.
        Re-running after a crash verifies/reuses any already committed payload.
        """
        path = Path(path).resolve()
        relative = path.relative_to(self.root).as_posix()
        if not compact_payload_file(path, self.root):
            raise ValueError("Only model checkpoints or detailed export parts may be compacted")
        if path.is_symlink():
            raise ValueError("Refusing to replace a symlink")
        if path.exists() and not replace_original:
            raise FileExistsError(path)
        before = path.stat() if replace_original else None
        sha = compact_sha(payload)
        key = compact_sha(relative.encode())[:20] + "_" + sha
        logical = Path(relative)
        container = (self.root / "models/compact_storage/payloads" if logical.parts[0] == "models"
                     else self.root.joinpath(*logical.parts[:2]) / "compact_payloads")
        target = container / key[:2] / (key + ".parquet")
        target.parent.mkdir(parents=True, exist_ok=True)
        row = {"logical_path": relative, "payload_sha256": sha,
               "payload_bytes": len(payload), "payload_json": payload}
        if not target.exists():
            temp = target.with_name(target.name + "." + secrets.token_hex(6) + ".tmp")
            try:
                with temp.open("xb") as dst:
                    pq.write_table(pa.Table.from_pylist([row], schema=PAYLOAD_ARROW), dst,
                                   compression="zstd", compression_level=3,
                                   use_dictionary=False, write_statistics=False)
                    dst.flush()
                    os.fsync(dst.fileno())
                with pq.ParquetFile(temp) as pf:
                    if not pf.schema_arrow.equals(PAYLOAD_ARROW, check_metadata=True) or pf.read().to_pylist() != [row]:
                        raise ValueError("Parquet exact-byte round-trip failed")
                os.replace(temp, target)
            finally:
                if temp.exists():
                    temp.unlink()
        else:
            with pq.ParquetFile(target) as pf:
                if not pf.schema_arrow.equals(PAYLOAD_ARROW, check_metadata=True) or pf.read().to_pylist() != [row]:
                    raise ValueError("Existing immutable compact payload differs")
        pointer = {"storage_schema": POINTER_SCHEMA,
                   "payload_path": target.relative_to(self.root).as_posix(),
                   "payload_sha256": sha, "payload_bytes": len(payload)}
        temp = path.with_name(path.name + "." + secrets.token_hex(6) + ".pointer.tmp")
        path.parent.mkdir(parents=True, exist_ok=True)
        try:
            with temp.open("xb") as dst:
                dst.write(json.dumps(pointer, sort_keys=True).encode())
                dst.flush()
                os.fsync(dst.fileno())
            if replace_original:
                now = path.stat()
                if (now.st_size, now.st_mtime_ns) != (before.st_size, before.st_mtime_ns):
                    raise RuntimeError("Original checkpoint changed during migration")
            elif path.exists():
                raise FileExistsError(path)
            os.replace(temp, path)
        finally:
            if temp.exists():
                temp.unlink()
        return len(payload), target.stat().st_size + path.stat().st_size


def compact_model_checkpoint(path, root):
    try:
        relative = Path(path).resolve().relative_to(Path(root).resolve())
    except ValueError:
        return False
    return (relative.parts[0] == "models" and relative.suffix == ".json"
            and relative.name.startswith(("unique_part_", "part_")))


def compact_payload_file(path, root):
    if compact_model_checkpoint(path, root):
        return True
    try:
        p = Path(path).resolve().relative_to(Path(root).resolve())
    except ValueError:
        return False
    # Newly imported delta-cache payloads are compact from their first write.
    # Existing legacy imports are NOT migrated: their manifest hashes describe
    # the original physical files and must remain valid.
    if (p.parts[0] == "models" and p.suffix == ".json"
            and p.name.startswith("cache_import_")):
        return True
    return (len(p.parts) == 4 and p.parts[0] == "exports"
            and p.parts[2] in {"selected_parts", "candidates_parts", "conflicts_parts"}
            and p.name.startswith("part_") and p.suffix == ".json")


def migrate_compact_checkpoints(root, progress=print, free_disk=shutil.disk_usage, expected_logical_files=None):
    store = CompactJSONStore(root)
    paths = sorted(p for p in (Path(root)/"models").rglob("*.json") if compact_model_checkpoint(p, root))
    report = {"storage_schema": PAYLOAD_SCHEMA, "files": len(paths), "migrated": 0,
              "already_compact": 0, "original_bytes": 0, "physical_bytes": 0}
    started, last = time.perf_counter(), 0
    for position, path in enumerate(paths, 1):
        if free_disk(root).free < 256 * 1024**2:
            raise RuntimeError("Migration paused below 256 MiB free; existing files remain readable. Rerun after freeing unrelated files.")
        raw = path.read_bytes()
        pointer = json.loads(raw) if len(raw) < 4096 else None
        if store.is_pointer(pointer):
            original = store.logical_bytes(path)  # Validate before treating it as completed.
            report["already_compact"] += 1
            report["original_bytes"] += len(original)
            report["physical_bytes"] += path.stat().st_size + compact_safe_path(root, pointer["payload_path"]).stat().st_size
        else:
            original = raw
        expected = (expected_logical_files or {}).get(path.relative_to(root).as_posix())
        if expected and (len(original) != expected["bytes"] or compact_sha(original) != expected["sha256"]):
            raise ValueError("Local logical checkpoint differs from the legacy recovery index: " + str(path))
        if not store.is_pointer(pointer):
            json.loads(raw)  # Never replace an unreadable source checkpoint.
            old, new = store.write(path, raw, replace_original=True)
            report["migrated"] += 1
            report["original_bytes"] += old
            report["physical_bytes"] += new
        mark = position * 10 // max(1, len(paths))
        if mark > last:
            progress(f"Compact migration {mark*10}%: {position}/{len(paths)}; "
                     f"saved={(report['original_bytes']-report['physical_bytes'])/1024**3:.2f} GiB; "
                     f"elapsed={time.perf_counter()-started:.1f}s", flush=True)
            last = mark
    report["seconds"] = time.perf_counter() - started
    return report


def validate_compact_tree(root):
    """Read every logical checkpoint; verify both Parquet and original byte hashes."""
    store, count = CompactJSONStore(root), 0
    paths = sorted(p for p in (Path(root)/"models").rglob("*.json") if compact_payload_file(p, root))
    started, last = time.perf_counter(), 0
    for path in paths:
        store.load(path)
        count += 1
        mark = count * 10 // max(1, len(paths))
        if mark > last:
            print(f"Restored logical payloads {mark*10}%: verified={count}/{len(paths)}; "
                  f"elapsed={time.perf_counter()-started:.1f}s", flush=True)
            last = mark
    return count


def install_compact_transport(ns):
    """Install before source restore. Legacy backup root remains read-only until pruning."""
    if getattr(ns["CheckpointGitHub"], "_compact_transport_v16", False):
        return
    ns["_COMPACT_TRANSPORT_INSTALLED"] = True
    legacy_root = ns["GITHUB_CHECKPOINT_ROOT"].strip("/")
    compact_root = legacy_root + "_v16"
    ns["LEGACY_CHECKPOINT_ROOT"] = legacy_root
    ns["COMPACT_CHECKPOINT_ROOT"] = compact_root
    prior_class = ns["CheckpointGitHub"]
    prior_validate = ns["checkpoint_relative_path"]

    def validate_relative(name):
        if isinstance(name, str) and name.startswith("models/compact_storage/payloads/") and name.endswith(".parquet"):
            p = PurePosixPath(name)
            if ".." not in p.parts and "\\" not in name and ":" not in name:
                return p
        return prior_validate(name)

    ns["checkpoint_relative_path"] = validate_relative
    packet_class = ns["CheckpointPackets"]
    old_candidates = packet_class.candidate_files

    def candidates(self):
        return old_candidates(self) + sorted((self.root / "models/compact_storage/payloads").rglob("*.parquet"))

    packet_class.candidate_files = candidates

    class CompactCheckpointGitHub(prior_class):
        _compact_transport_v16 = True
        def __init__(self, *args, **kwargs):
            if len(args) < 3:
                kwargs.setdefault("root", compact_root)
            super().__init__(*args, **kwargs)

        def restore(self, run_name, output_root):
            # Query compact first. A deleted legacy chain is never needed after conversion.
            index = self.latest_index(run_name, missing=True)
            if index is None and self.root == compact_root:
                legacy = prior_class(repository=self.repository, branch=self.branch,
                                     root=legacy_root, token_provider=lambda: self.token)
                info = legacy.restore(run_name, output_root)
                info["storage_backend_root"] = legacy_root
                return info
            info = super().restore(run_name, output_root)
            info["storage_backend_root"] = self.root
            return info

        def publish_index(self, manager):
            super().publish_index(manager)
            if (ns.get("COMPACT_CLEAN_LOCAL_CONFIRMED", True)
                    and self.published_head == manager.parent and manager.active_plan is None):
                removed = 0
                # Delete only local ZIPs with independently confirmed remote identity.
                for receipt in manager.receipts:
                    descriptor = self.packets.get(receipt["sequence"])
                    path = Path(receipt["path"]).resolve()
                    if (descriptor and descriptor["sha256"] == receipt["sha256"]
                            and path.is_relative_to(manager.output.resolve()) and path.is_file()
                            and ns["sha_file_stream"](path) == descriptor["sha256"]):
                        removed += path.stat().st_size
                        path.unlink()
                if removed:
                    print(f"Confirmed local ZIP copies released: {removed/1024**3:.3f} GiB", flush=True)

    ns["CheckpointGitHub"] = CompactCheckpointGitHub


def install_compact_io(ns):
    if (getattr(ns["load_json"], "_compact_io_v16", False)
            and getattr(ns["atomic_json"], "_compact_io_v16", False)):
        return
    ns["_COMPACT_IO_INSTALLED"] = True
    old_load, old_atomic = ns["load_json"], ns["atomic_json"]

    def load(path):
        if compact_payload_file(path, ns["RUN_DIR"]):
            return CompactJSONStore(ns["RUN_DIR"]).load(path)
        # Delta imports may read a different compact run on the same filesystem.
        p = Path(path).resolve()
        if p.suffix == ".json" and p.name.startswith(("unique_part_", "part_", "cache_import_")):
            for parent in p.parents:
                if (parent / "run.json").is_file() and (parent / "source.parquet").is_file():
                    if compact_payload_file(p, parent):
                        return CompactJSONStore(parent).load(p)
                    break
        return old_load(path)

    def atomic(path, obj):
        if compact_payload_file(path, ns["RUN_DIR"]):
            return CompactJSONStore(ns["RUN_DIR"]).write(path, ns["encoded"](obj))
        return old_atomic(path, obj)

    load._compact_io_v16 = atomic._compact_io_v16 = True
    ns["load_json"], ns["atomic_json"] = load, atomic
    original_hook = ns["checkpoint_safety_hook"]

    def guarded_hook(*args, **kwargs):
        if not ns.get("COMPACT_BACKEND_READY"):
            raise RuntimeError("Compact migration not backed up yet. Rerun compact initialization; do not run a model cell.")
        return original_hook(*args, **kwargs)

    ns["checkpoint_safety_hook"] = guarded_hook


def compact_remote_prune(client, legacy_root, run_name, proof, pace):
    """One atomic Git commit, exact current-run legacy prefix only; never rewrite history."""
    if (proof.get("verified") is not True or proof.get("legacy_coverage_verified") is not True
            or proof.get("run_name") != run_name
            or proof.get("root") != client.root or proof.get("head_sha256") != client.published_head):
        raise ValueError("Remote pruning requires a successful independent restore of the current compact head")
    if not run_name or "/" in run_name or "\\" in run_name or run_name in {".", ".."}:
        raise ValueError("Unsafe run name")
    prefix = client.remote_path(legacy_root.strip("/") + "/" + run_name) + "/"
    if (client.root + "/").startswith(prefix) or prefix.startswith(client.root + "/"):
        raise ValueError("Old and new backup namespaces overlap")
    latest = client.latest_index(run_name)
    if latest["head_sha256"] != proof["head_sha256"]:
        raise RuntimeError("New backup published after restore proof; verify the latest recovery point first")
    head = client.api("GET", "/branches/" + urllib.parse.quote(client.branch, safe=""))["commit"]["sha"]
    commit = client.api("GET", "/git/commits/" + head)
    tree = client.api("GET", "/git/trees/" + commit["tree"]["sha"] + "?recursive=1")
    if tree.get("truncated"):
        raise RuntimeError("Git tree listing truncated; no deletion performed")
    doomed = [r for r in tree["tree"] if r["type"] == "blob" and (
        (r["path"].startswith(prefix + "packets/") and r["path"].endswith(".zip")) or
        (r["path"].startswith(prefix + "indexes/checkpoint_") and r["path"].endswith(".json")))]
    if not doomed:
        return {"status": "already_absent", "deleted_files": 0, "prefix": prefix}
    known_legacy = set(proof.get("legacy_packet_paths", []))
    if any(r["path"].endswith(".zip") and r["path"] not in known_legacy for r in doomed):
        raise RuntimeError("Legacy packet set changed or is unconfirmed; old backup retained")
    # Never delete any packet referenced by the new complete snapshot.
    protected = {p["path"] for p in latest["packets"]}
    if any(r["path"] in protected for r in doomed):
        raise ValueError("Legacy deletion includes a packet required by the new backup")
    pace()
    new_tree = client.api("POST", "/git/trees", {"base_tree": commit["tree"]["sha"], "tree": [
        {"path": r["path"], "mode": r["mode"], "type": "blob", "sha": None} for r in doomed]})
    pace()
    new_commit = client.api("POST", "/git/commits", {
        "message": "Retire verified legacy NER backup " + run_name,
        "tree": new_tree["sha"], "parents": [head]})
    pace()
    client.api("PATCH", "/git/refs/heads/" + urllib.parse.quote(client.branch, safe="/"),
               {"sha": new_commit["sha"], "force": False})
    # An interrupted/uncertain write is safe to retry: only this exact prefix is targeted.
    check = client.api("GET", "/git/trees/" + new_tree["sha"] + "?recursive=1")
    deleted_paths = {r["path"] for r in doomed}
    if check.get("truncated") or any(r["path"] in deleted_paths for r in check["tree"]):
        raise RuntimeError("Could not verify legacy deletion; compact backup remains intact")
    return {"status": "deleted_from_current_branch", "deleted_files": len(doomed),
            "prefix": prefix, "commit": new_commit["sha"], "history_rewritten": False}


def compact_initialize(ns):
    """Migrate with bounded disk, create independent backup, test restore, then prune."""
    started = time.perf_counter()
    root = Path(ns["RUN_DIR"]).resolve()
    # Recovery receipts are local and are deliberately absent from packet restores.
    # Create their directory even when this is an already compact restored run.
    (root / "recovery").mkdir(parents=True, exist_ok=True)
    install_compact_io(ns)
    ns["COMPACT_BACKEND_READY"] = False
    # Keep legacy point as evidence; never reuse its immutable covered-file map.
    previous_client = ns.get("CHECKPOINT_GITHUB")
    ns["CHECKPOINT_BACKUPS"] = None
    kwargs = {}
    if previous_client is not None:
        kwargs = dict(repository=previous_client.repository, branch=previous_client.branch,
                      token_provider=lambda: previous_client.token, transport=previous_client.open)
    legacy_client = ns["CheckpointGitHub"](root=ns["LEGACY_CHECKPOINT_ROOT"], **kwargs)
    legacy_index = legacy_client.latest_index(ns["RUN_NAME"], missing=True)
    legacy_files = (legacy_index or {}).get("files", {})
    # Prove that local migration contains EVERY canonical file in the old recovery point.
    # Unique/occurrence payloads are verified during the migration pass, avoiding another
    # full read of tens of GiB. Small source/prepared/metadata files are streamed here.
    for name, detail in legacy_files.items():
        p = compact_safe_path(root, name)
        if not p.is_file():
            raise ValueError("Local run is missing a legacy-backed file; restore before migration: " + name)
        if not compact_model_checkpoint(p, root):
            if p.stat().st_size != detail["bytes"] or ns["sha_file_stream"](p) != detail["sha256"]:
                raise ValueError("Local legacy-backed metadata differs: " + name)
    report = migrate_compact_checkpoints(root, expected_logical_files=legacy_files)
    ns["COMPACT_STORAGE_REPORT"] = report
    print("Lossless migration:", json.dumps(report), flush=True)
    if previous_client is not None:
        client = ns["CheckpointGitHub"](repository=previous_client.repository,
            branch=previous_client.branch, token_provider=lambda: previous_client.token,
            transport=previous_client.open)
    else:
        client = ns["CheckpointGitHub"]()
    ns["CHECKPOINT_GITHUB"] = client
    prior = client.latest_index(ns["RUN_NAME"], missing=True)
    if prior is not None:
        for key, expected in (("source_sha256", ns["SOURCE_META"]["sha256"]),
                              ("sample_sha256", ns["SAMPLE_HASH"]), ("inference_sha256", ns["INFERENCE_HASH"])):
            if prior[key] != expected:
                raise ValueError("Compact remote run identity differs")
        for name, detail in prior["files"].items():
            p = compact_safe_path(root, name)
            if not p.is_file() or p.stat().st_size != detail["bytes"] or ns["sha_file_stream"](p) != detail["sha256"]:
                raise ValueError("Local compact checkpoint differs from its confirmed remote snapshot: " + name)
        client.packets = {p["sequence"]: p for p in prior["packets"]}
        client.published_head = prior["head_sha256"]
    manager = ns["CheckpointPackets"](
        root, Path(ns["OUTPUT_ROOT"]) / "compact_backups" / ns["RUN_NAME"], ns["CONFIG"],
        ns["SOURCE_META"]["sha256"], ns["SAMPLE_HASH"], ns["INFERENCE_HASH"],
        packet_bytes=ns["BACKUP_PACKET_MIB"] * 1024**2,
        segment_bytes=ns["BACKUP_SEGMENT_MIB"] * 1024**2, restore_info=prior)
    ns["CHECKPOINT_BACKUPS"] = manager
    # Direct call bypasses resource guard only for this bounded migration/bootstrap.
    manager.backup("compact_storage_migration", client.upload_packet)
    client.publish_index(manager)
    proof_path = root / "recovery" / ("compact_restore_proof_" + manager.parent[:16] + ".json")
    if proof_path.exists():
        proof = json.loads(proof_path.read_bytes())
        if proof.get("head_sha256") != manager.parent or not proof.get("verified"):
            raise ValueError("Unexpected compact restore proof")
    else:
        # A real download+restore, not just existence checks. No cleanup before it passes.
        test_root = Path(ns["OUTPUT_ROOT"]) / ("compact_restore_test_" + secrets.token_hex(6))
        test_root.mkdir(exist_ok=False)
        try:
            info = client.restore(ns["RUN_NAME"], test_root)
            restored = Path(info["run_dir"])
            checked = validate_compact_tree(restored)
            proof = {"verified": True, "run_name": ns["RUN_NAME"], "root": client.root,
                     "head_sha256": info["head_sha256"], "logical_checkpoints_verified": checked,
                     "physical_files_verified": len(info["files"]),
                     "legacy_coverage_verified": True, "legacy_files_verified": len(legacy_files),
                     "legacy_head_sha256": (legacy_index or {}).get("head_sha256"),
                     "legacy_packet_paths": [p["path"] for p in (legacy_index or {}).get("packets", [])]}
            ns["atomic_json"](proof_path, proof)
        finally:
            # This random directory contains replicas only, never canonical run files.
            # Remove partial test downloads too, so failed verification cannot fill disk.
            if not test_root.resolve().is_relative_to(Path(ns["OUTPUT_ROOT"]).resolve()):
                raise ValueError("Unsafe restore-test cleanup directory")
            shutil.rmtree(test_root)
    ns["COMPACT_RESTORE_PROOF"] = proof
    cleanup = {"remote": {"status": "disabled"}, "local_bytes_removed": 0}
    if ns.get("COMPACT_PRUNE_LEGACY_REMOTE", True):
        try:
            now_legacy = legacy_client.latest_index(ns["RUN_NAME"], missing=True)
            if now_legacy and now_legacy["head_sha256"] != proof.get("legacy_head_sha256"):
                raise RuntimeError("Legacy recovery point changed after migration; no old backups deleted")
            cleanup["remote"] = compact_remote_prune(client, ns["LEGACY_CHECKPOINT_ROOT"],
                ns["RUN_NAME"], proof, ns["github_pace_write"])
        except Exception as exc:
            # Optional cleanup cannot invalidate an already durable extraction snapshot.
            cleanup["remote"] = {"status": "failed_or_unconfirmed", "error": str(exc)}
            print("WARNING legacy cleanup not confirmed; verified compact backup retained:", exc, flush=True)
    if ns.get("COMPACT_CLEAN_LOCAL_CONFIRMED", True):
        # Only directories belonging to this exact run. Packet hashes must be known.
        known = {p["sha256"] for p in client.packets.values()}
        known.update(p["sha256"] for p in (legacy_index or {}).get("packets", []))
        if previous_client is not None:
            known.update(p["sha256"] for p in previous_client.packets.values())
        old_info = ns.get("RESTORE_INFO") or {}
        known.update(p["sha256"] for p in old_info.get("remote_index", {}).get("packets", []))
        folders = [Path(ns["OUTPUT_ROOT"]) / "checkpoint_backups" / ns["RUN_NAME"],
                   Path(ns["OUTPUT_ROOT"]) / "restore_packets"]
        for folder in folders:
            if not folder.resolve().is_relative_to(Path(ns["OUTPUT_ROOT"]).resolve()):
                raise ValueError("Unsafe local cleanup directory")
            for p in folder.rglob("*.zip"):
                if p.is_file() and ns["sha_file_stream"](p) in known:
                    cleanup["local_bytes_removed"] += p.stat().st_size
                    p.unlink()
    ns["COMPACT_CLEANUP_REPORT"] = cleanup
    ns["STAGE_TIMINGS"]["compact_migration_backup_restore_seconds"] = time.perf_counter()-started
    ns["atomic_json"](root / "recovery" / ("compact_cleanup_" + secrets.token_hex(6) + ".json"), cleanup)
    ns["COMPACT_BACKEND_READY"] = True
    print("Compact backup/restore verified:",
          {**{k:v for k,v in proof.items() if k != "legacy_packet_paths"},
           "legacy_packet_count": len(proof.get("legacy_packet_paths", []))}, flush=True)
    print("Cleanup:", cleanup, flush=True)
    print(f"Free disk after migration: {shutil.disk_usage(root).free/1024**3:.2f} GiB", flush=True)


"""Specific rule-engine timeout classification for reconciled immutable PUTs.

403 alone is NOT retryable. Only GitHub's exact rule-validation timeout message
on Contents PUT is handed to the existing bounded put() retry/reconciliation loop.
POST/PATCH and genuine permission/ruleset rejection keep their existing behavior.
"""


def install_rule_timeout_retry(ns):
    cls = ns["GitHubDeliveryError"]
    if getattr(cls.__init__, "_rule_timeout_retry_v1", False):
        return
    previous = cls.__init__

    def initialize(self, detail, retryable=False):
        message = str(detail.get("response_message", "")).strip().lower().rstrip(".")
        if (detail.get("status") == 403 and message == "timed out validating the rule"
                and detail.get("method") == "PUT"
                and str(detail.get("endpoint", "")).startswith("/contents/")):
            retryable = True
            detail = {**detail, "transient_rule_validation_timeout": True,
                "explanation": "GitHub rule-validation timeout; immutable put() will wait and check "
                    "remote SHA/size before a bounded retry. This response does not establish "
                    "a permanent permission denial; repeated timeouts still stop safely."}
            ns["GITHUB_HTTP_STATS"]["rule_timeout_classification_corrections"] += 1
        previous(self, detail, retryable)

    initialize._rule_timeout_retry_v1 = True
    cls.__init__ = initialize


# In a notebook cell, call install_rule_timeout_retry(globals()).

COMPACT_READER_SOURCE = '"""Lossless Parquet payload storage; no changes to NER decisions or model calls.\n\nLarge model JSON files become small, versioned pointers. Original JSON bytes are\nstored in a Zstandard-compressed binary Parquet column, without float conversion.\nThis is a checkpoint container, not the nine-column consumer Parquet schema.\n"""\nfrom pathlib import Path, PurePosixPath\nimport hashlib\nimport json\nimport os\nimport secrets\nimport shutil\nimport time\nimport urllib.parse\n\nimport pyarrow as pa\nimport pyarrow.parquet as pq\n\nPOINTER_SCHEMA = "georag-json-parquet-pointer-1.0"\nPAYLOAD_SCHEMA = "georag-lossless-checkpoint-parquet-1.0"\nPAYLOAD_ARROW = pa.schema([\n    ("logical_path", pa.string()), ("payload_sha256", pa.string()),\n    ("payload_bytes", pa.int64()), ("payload_json", pa.large_binary()),\n], metadata={b"schema_version": PAYLOAD_SCHEMA.encode()})\n\n\ndef compact_sha(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef compact_safe_path(root, relative):\n    p = PurePosixPath(relative)\n    if (not isinstance(relative, str) or "\\\\" in relative or ":" in relative\n            or p.is_absolute() or not p.parts or ".." in p.parts):\n        raise ValueError("Unsafe compact payload path")\n    root = Path(root).resolve()\n    result = (root / relative).resolve()\n    if not result.is_relative_to(root):\n        raise ValueError("Compact payload escapes run directory")\n    return result\n\n\nclass CompactJSONStore:\n    def __init__(self, root):\n        self.root = Path(root).resolve()\n\n    @staticmethod\n    def is_pointer(obj):\n        return isinstance(obj, dict) and obj.get("storage_schema") == POINTER_SCHEMA\n\n    def logical_bytes(self, path):\n        path = Path(path)\n        value = path.read_bytes()\n        # Tiny pointers only; avoid parsing large original payloads twice.\n        if len(value) < 4096:\n            obj = json.loads(value)\n            if self.is_pointer(obj):\n                target = compact_safe_path(self.root, obj["payload_path"])\n                logical = path.resolve().relative_to(self.root)\n                expected_prefix = ("models/compact_storage/payloads/" if logical.parts[0] == "models"\n                    else "/".join(logical.parts[:2]) + "/compact_payloads/")\n                if not obj["payload_path"].startswith(expected_prefix):\n                    raise ValueError("Unexpected compact payload namespace")\n                with pq.ParquetFile(target) as pf:\n                    if not pf.schema_arrow.equals(PAYLOAD_ARROW, check_metadata=True) or pf.metadata.num_rows != 1:\n                        raise ValueError("Compact Parquet schema/row-count mismatch")\n                    row = pf.read().to_pylist()[0]\n                actual = path.resolve().relative_to(self.root).as_posix()\n                if (row["logical_path"] != actual or row["payload_sha256"] != obj["payload_sha256"]\n                        or row["payload_bytes"] != obj["payload_bytes"]):\n                    raise ValueError("Compact pointer/Parquet identity mismatch")\n                value = row["payload_json"]\n                if len(value) != obj["payload_bytes"] or compact_sha(value) != obj["payload_sha256"]:\n                    raise ValueError("Compact payload failed exact-byte SHA256 validation")\n        return value\n\n    def load(self, path):\n        return json.loads(self.logical_bytes(path))\n\n    def write(self, path, payload, *, replace_original=False):\n        """Commit payload first, verify bytes, then atomically commit its pointer.\n\n        At most one source checkpoint and its compressed replacement are held.\n        Re-running after a crash verifies/reuses any already committed payload.\n        """\n        path = Path(path).resolve()\n        relative = path.relative_to(self.root).as_posix()\n        if not compact_payload_file(path, self.root):\n            raise ValueError("Only model checkpoints or detailed export parts may be compacted")\n        if path.is_symlink():\n            raise ValueError("Refusing to replace a symlink")\n        if path.exists() and not replace_original:\n            raise FileExistsError(path)\n        before = path.stat() if replace_original else None\n        sha = compact_sha(payload)\n        key = compact_sha(relative.encode())[:20] + "_" + sha\n        logical = Path(relative)\n        container = (self.root / "models/compact_storage/payloads" if logical.parts[0] == "models"\n                     else self.root.joinpath(*logical.parts[:2]) / "compact_payloads")\n        target = container / key[:2] / (key + ".parquet")\n        target.parent.mkdir(parents=True, exist_ok=True)\n        row = {"logical_path": relative, "payload_sha256": sha,\n               "payload_bytes": len(payload), "payload_json": payload}\n        if not target.exists():\n            temp = target.with_name(target.name + "." + secrets.token_hex(6) + ".tmp")\n            try:\n                with temp.open("xb") as dst:\n                    pq.write_table(pa.Table.from_pylist([row], schema=PAYLOAD_ARROW), dst,\n                                   compression="zstd", compression_level=3,\n                                   use_dictionary=False, write_statistics=False)\n                    dst.flush()\n                    os.fsync(dst.fileno())\n                with pq.ParquetFile(temp) as pf:\n                    if not pf.schema_arrow.equals(PAYLOAD_ARROW, check_metadata=True) or pf.read().to_pylist() != [row]:\n                        raise ValueError("Parquet exact-byte round-trip failed")\n                os.replace(temp, target)\n            finally:\n                if temp.exists():\n                    temp.unlink()\n        else:\n            with pq.ParquetFile(target) as pf:\n                if not pf.schema_arrow.equals(PAYLOAD_ARROW, check_metadata=True) or pf.read().to_pylist() != [row]:\n                    raise ValueError("Existing immutable compact payload differs")\n        pointer = {"storage_schema": POINTER_SCHEMA,\n                   "payload_path": target.relative_to(self.root).as_posix(),\n                   "payload_sha256": sha, "payload_bytes": len(payload)}\n        temp = path.with_name(path.name + "." + secrets.token_hex(6) + ".pointer.tmp")\n        path.parent.mkdir(parents=True, exist_ok=True)\n        try:\n            with temp.open("xb") as dst:\n                dst.write(json.dumps(pointer, sort_keys=True).encode())\n                dst.flush()\n                os.fsync(dst.fileno())\n            if replace_original:\n                now = path.stat()\n                if (now.st_size, now.st_mtime_ns) != (before.st_size, before.st_mtime_ns):\n                    raise RuntimeError("Original checkpoint changed during migration")\n            elif path.exists():\n                raise FileExistsError(path)\n            os.replace(temp, path)\n        finally:\n            if temp.exists():\n                temp.unlink()\n        return len(payload), target.stat().st_size + path.stat().st_size\n\n\ndef compact_model_checkpoint(path, root):\n    try:\n        relative = Path(path).resolve().relative_to(Path(root).resolve())\n    except ValueError:\n        return False\n    return (relative.parts[0] == "models" and relative.suffix == ".json"\n            and relative.name.startswith(("unique_part_", "part_")))\n\n\ndef compact_payload_file(path, root):\n    if compact_model_checkpoint(path, root):\n        return True\n    try:\n        p = Path(path).resolve().relative_to(Path(root).resolve())\n    except ValueError:\n        return False\n    # Newly imported delta-cache payloads are compact from their first write.\n    # Existing legacy imports are NOT migrated: their manifest hashes describe\n    # the original physical files and must remain valid.\n    if (p.parts[0] == "models" and p.suffix == ".json"\n            and p.name.startswith("cache_import_")):\n        return True\n    return (len(p.parts) == 4 and p.parts[0] == "exports"\n            and p.parts[2] in {"selected_parts", "candidates_parts", "conflicts_parts"}\n            and p.name.startswith("part_") and p.suffix == ".json")\n\n\ndef migrate_compact_checkpoints(root, progress=print, free_disk=shutil.disk_usage, expected_logical_files=None):\n    store = CompactJSONStore(root)\n    paths = sorted(p for p in (Path(root)/"models").rglob("*.json") if compact_model_checkpoint(p, root))\n    report = {"storage_schema": PAYLOAD_SCHEMA, "files": len(paths), "migrated": 0,\n              "already_compact": 0, "original_bytes": 0, "physical_bytes": 0}\n    started, last = time.perf_counter(), 0\n    for position, path in enumerate(paths, 1):\n        if free_disk(root).free < 256 * 1024**2:\n            raise RuntimeError("Migration paused below 256 MiB free; existing files remain readable. Rerun after freeing unrelated files.")\n        raw = path.read_bytes()\n        pointer = json.loads(raw) if len(raw) < 4096 else None\n        if store.is_pointer(pointer):\n            original = store.logical_bytes(path)  # Validate before treating it as completed.\n            report["already_compact"] += 1\n            report["original_bytes"] += len(original)\n            report["physical_bytes"] += path.stat().st_size + compact_safe_path(root, pointer["payload_path"]).stat().st_size\n        else:\n            original = raw\n        expected = (expected_logical_files or {}).get(path.relative_to(root).as_posix())\n        if expected and (len(original) != expected["bytes"] or compact_sha(original) != expected["sha256"]):\n            raise ValueError("Local logical checkpoint differs from the legacy recovery index: " + str(path))\n        if not store.is_pointer(pointer):\n            json.loads(raw)  # Never replace an unreadable source checkpoint.\n            old, new = store.write(path, raw, replace_original=True)\n            report["migrated"] += 1\n            report["original_bytes"] += old\n            report["physical_bytes"] += new\n        mark = position * 10 // max(1, len(paths))\n        if mark > last:\n            progress(f"Compact migration {mark*10}%: {position}/{len(paths)}; "\n                     f"saved={(report[\'original_bytes\']-report[\'physical_bytes\'])/1024**3:.2f} GiB; "\n                     f"elapsed={time.perf_counter()-started:.1f}s", flush=True)\n            last = mark\n    report["seconds"] = time.perf_counter() - started\n    return report\n\n\ndef validate_compact_tree(root):\n    """Read every logical checkpoint; verify both Parquet and original byte hashes."""\n    store, count = CompactJSONStore(root), 0\n    paths = sorted(p for p in (Path(root)/"models").rglob("*.json") if compact_payload_file(p, root))\n    started, last = time.perf_counter(), 0\n    for path in paths:\n        store.load(path)\n        count += 1\n        mark = count * 10 // max(1, len(paths))\n        if mark > last:\n            print(f"Restored logical payloads {mark*10}%: verified={count}/{len(paths)}; "\n                  f"elapsed={time.perf_counter()-started:.1f}s", flush=True)\n            last = mark\n    return count\n\n\n'

install_compact_transport(globals())
install_rule_timeout_retry(globals())

"""Publish a binary Parquet + provenance atomically with Git, not REST blob JSON.

Only shallow commit/tree metadata are fetched. No checkout of corpus/checkpoints,
token in URLs/arguments/files, force push or modification of the user's checkout.
"""
from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import time


def publication_path(value):
    p=PurePosixPath(value)
    if not value or '\\' in value or ':' in value or p.is_absolute() or '..' in p.parts:
        raise ValueError('Unsafe repository-relative publication path')
    return p.as_posix()


def publication_bytes(value):
    return json.dumps(value,ensure_ascii=False,sort_keys=True,allow_nan=False).encode('utf-8')


def publication_blob_sha(value):
    return hashlib.sha1(f'blob {len(value)}\0'.encode()+value).hexdigest()


def publish_git_pair(path, pointer, pointer_path, repository, branch, token, output_root,
                     *, remote_url=None, allow_local_test=False, wait=None, attempts=3):
    path=Path(path).resolve(strict=True)
    if not re.fullmatch(r'[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+',repository):
        raise ValueError('Invalid GitHub repository')
    if not branch or branch.startswith('-') or any(c in branch for c in '\n\r\0'):
        raise ValueError('Invalid publication branch')
    if not isinstance(token,str) or not token.strip():
        raise RuntimeError('An automatically accessible GitHub credential is required')
    if not 0<path.stat().st_size<100_000_000:
        raise ValueError('This publisher requires a nonempty file smaller than 100 MB')
    git=shutil.which('git')
    if not git:raise RuntimeError('Git executable not found; source/output files are unchanged')
    target=publication_path(pointer['output']['path'])
    pointer_path=publication_path(pointer_path)
    if target==pointer_path:raise ValueError('Parquet and provenance destinations must differ')
    remote_url=remote_url or f'https://github.com/{repository}.git'
    if remote_url!=f'https://github.com/{repository}.git' and not allow_local_test:
        raise ValueError('Unexpected Git remote; credentials are restricted to the configured GitHub repository')
    pointer_data=publication_bytes(pointer)
    pointer_sha=publication_blob_sha(pointer_data)
    size=path.stat().st_size
    h=hashlib.sha1(f'blob {size}\0'.encode());sha=hashlib.sha256()
    with path.open('rb') as src:
        for block in iter(lambda:src.read(1024**2),b''):h.update(block);sha.update(block)
    blob=h.hexdigest()
    if (pointer['output']['git_blob_sha']!=blob or pointer['output']['sha256']!=sha.hexdigest()
            or pointer['output']['bytes']!=size):
        raise ValueError('Provenance does not match the exact Parquet bytes')
    key=hashlib.sha256(publication_bytes([repository,branch,target,blob,pointer_path,pointer_sha])).hexdigest()[:24]
    parent=Path(output_root).resolve()/'publication_git'
    parent.mkdir(parents=True,exist_ok=True)
    folder=parent/key
    folder.mkdir(exist_ok=True)
    if not folder.resolve().is_relative_to(parent):raise ValueError('Unsafe publication staging directory')
    # Native mkdir inherits the workspace ACL on Windows; no private-mode temp dirs.
    bare=folder/'objects.git';bare.mkdir(exist_ok=True)
    hook_dir=folder/'empty_hooks';hook_dir.mkdir(exist_ok=True)
    credential=folder/'askpass.py'
    credential.write_text("import os,sys\nprint('x-access-token' if 'username' in ' '.join(sys.argv[1:]).lower() else os.environ['_GEORAG_GIT_TOKEN'])\n",encoding='utf-8')
    if os.name=='nt':
        launcher=folder/'askpass.cmd'
        executable=str(sys.executable)
        if any(c in executable+str(credential) for c in '%\n\r"'):
            raise ValueError('Unsupported characters in Git askpass launcher paths')
        launcher.write_text(f'@echo off\n"{executable}" "{credential}" "%~1"\n',encoding='utf-8')
    else:
        launcher=folder/'askpass.sh'
        launcher.write_text('#!/bin/sh\nexec '+shlex.quote(sys.executable)+' '+shlex.quote(str(credential))+' "$@"\n',encoding='utf-8')
        launcher.chmod(0o700)
    env=os.environ.copy()
    for name in list(env):
        if name.startswith('GIT_CONFIG_'):env.pop(name,None)
    env.update(_GEORAG_GIT_TOKEN=token,GIT_ASKPASS=str(launcher),GIT_TERMINAL_PROMPT='0',
        GCM_INTERACTIVE='never',GH_PROMPT_DISABLED='1',GIT_CONFIG_GLOBAL=os.devnull,GIT_CONFIG_NOSYSTEM='1',
        GIT_AUTHOR_NAME='GeoRAG NER',GIT_AUTHOR_EMAIL='georag-ner@users.noreply.github.com',
        GIT_COMMITTER_NAME='GeoRAG NER',GIT_COMMITTER_EMAIL='georag-ner@users.noreply.github.com',
        GIT_INDEX_FILE=str(folder/'publication.index'))
    for name in ('GIT_TRACE','GIT_TRACE_CURL','GIT_TRACE_PACKET','GIT_CURL_VERBOSE','GIT_TRACE2',
                 'GIT_TRACE2_EVENT','GIT_TRACE2_PERF','GCM_TRACE'):env.pop(name,None)
    def native(value):return str(value).removeprefix('\\\\?\\')
    env['GIT_INDEX_FILE']=native(folder/'publication.index')
    # ASKPASS is an executable path, not a shell command. Git handles its argv;
    # embedding quote characters makes Windows treat them as part of the filename.
    env['GIT_ASKPASS']=native(launcher)
    prefix=[git,'--git-dir='+native(bare),'-c','credential.helper=',
            '-c','core.hooksPath='+native(hook_dir),'-c','http.version=HTTP/1.1',
            '-c','http.lowSpeedLimit=1024','-c','http.lowSpeedTime=120']
    def safe(text):
        return re.sub(r'(?i)Bearer\s+\S+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9_]+',
                      '[REDACTED]',text.replace(token,'[REDACTED]'))[-6000:]
    def run(args, data=None, check=True):
        try:
            result=subprocess.run(prefix+args,input=data,capture_output=True,env=env,timeout=600)
        except subprocess.TimeoutExpired:
            raise RuntimeError('Git publication command timed out; local staging and original files retained') from None
        if check and result.returncode:
            raise RuntimeError('Git '+args[0]+' failed: '+safe(result.stderr.decode('utf-8','replace')))
        return result
    if not (bare/'HEAD').is_file():
        run(['init','--bare'])
        run(['remote','add','origin',remote_url])
    elif run(['remote','get-url','origin']).stdout.decode().strip()!=remote_url:
        raise ValueError('Existing publication staging remote differs')
    print('Git publication: fetching shallow tree metadata; no dataset checkout',flush=True)
    def fetch():
        run(['-c','protocol.version=2','fetch','--depth=1','--filter=blob:none','--no-tags','origin','refs/heads/'+branch])
        return run(['rev-parse','FETCH_HEAD']).stdout.decode().strip()
    def pair(commit):
        result={}
        for item in run(['ls-tree','-z',commit,'--',target,pointer_path]).stdout.split(b'\0'):
            if not item:continue
            metadata,name=item.split(b'\t',1)
            mode,kind,oid=metadata.split()
            if kind!=b'blob':raise ValueError('Publication destination is not a regular file')
            result[name.decode()]=oid.decode()
        return result
    desired={target:blob,pointer_path:pointer_sha}
    base=fetch();initial=pair(base)
    plan=folder/'plan.json'
    if plan.exists():
        saved=json.loads(plan.read_bytes())
        if saved['desired']!=desired:raise ValueError('Staged publication identity changed')
        initial=saved['initial']
    else:
        plan.write_bytes(publication_bytes({'desired':desired,'initial':initial,'base':base}))
    local_blob=run(['hash-object','-w','--no-filters',native(path)]).stdout.decode().strip()
    local_pointer=run(['hash-object','-w','--stdin'],pointer_data).stdout.decode().strip()
    if (local_blob,local_pointer)!=(blob,pointer_sha):raise ValueError('Git local object hashes differ')
    def receipt(commit,status):
        return {'status':status,'path':target,'pointer_path':pointer_path,'commit_sha':commit,
            'git_blob_sha':blob,'sha256':sha.hexdigest(),'bytes':size,'transport':'git',
            'url':f'https://github.com/{repository}/blob/{branch}/{target}'}
    for attempt in range(attempts):
        actual=pair(base)
        if actual==desired:return receipt(base,'already_present_identical')
        if actual!=initial:
            raise RuntimeError('Concurrent publication changed the target pair; refusing to overwrite it or force-push')
        run(['read-tree',base])
        for name,oid in desired.items():
            run(['update-index','--add','--cacheinfo','100644,'+oid+','+name])
        # Unchanged parent blobs deliberately remain on the promisor remote.
        # Both new blobs were hashed and verified locally above.
        tree=run(['write-tree','--missing-ok']).stdout.decode().strip()
        commit=run(['commit-tree',tree,'-p',base],('Publish GeoRAG final Parquet: '+str(pointer['run_name'])+'\n').encode()).stdout.decode().strip()
        print(f'Git publication: uploading exact Parquet and provenance atomically; attempt {attempt+1}/{attempts}',flush=True)
        result=run(['push','--porcelain','origin',commit+':refs/heads/'+branch],check=False)
        # Check remote state even after a lost/error response. Never repeat a blind write.
        base=fetch()
        if pair(base)==desired:
            return receipt(base,'updated' if target in initial else 'created')
        if not result.returncode:
            raise RuntimeError('Published branch no longer contains the expected pair; original local files retained')
        message=safe(result.stderr.decode('utf-8','replace'))
        if any(word in message.lower() for word in ('authentication failed','permission denied','gh013',
                'repository rule','protected branch','push declined','exceeds github','repository not found')):
            raise RuntimeError('Git publication rejected: '+message)
        if attempt+1>=attempts:raise RuntimeError('Git publication exhausted retries: '+message)
        seconds=60*(2**attempt)
        if wait is not None:wait(seconds,'Git push retry after checking remote object identities')
        else:
            while seconds>0:
                step=min(30,seconds);print(f'Waiting {seconds}s before Git retry',flush=True);time.sleep(step);seconds-=step
    raise AssertionError('Unreachable Git publication state')

def publish_current_parquet(path, expected_rows, client=None):
        path = Path(path)
        if not path.is_file() or not 0 < path.stat().st_size < 100_000_000:
            raise ValueError("GitHub publication requires a nonempty Parquet smaller than 100 MB")
        with pq.ParquetFile(path) as parquet:
            if parquet.schema_arrow != ARROW_SCHEMA or parquet.metadata.num_rows != expected_rows:
                raise ValueError("Current Parquet schema/count differs from validated export")
        if not PROCESS_ALL and not ALLOW_SAMPLE_PUBLICATION:
            return {"status": "skipped_sample", "reason": "Test samples do not replace the full current output"}
        client = client or CheckpointGitHub(branch=GITHUB_UPLOAD_BRANCH)
        if PurePosixPath(GITHUB_FINAL_FILENAME).name != GITHUB_FINAL_FILENAME or not GITHUB_FINAL_FILENAME.endswith(".parquet"):
            raise ValueError("GITHUB_FINAL_FILENAME must be a Parquet filename without a folder")
        target = client.remote_path(str(PurePosixPath(GITHUB_OUTPUT_DIR) / GITHUB_FINAL_FILENAME))
        pointer_path = client.remote_path(current_pointer_path())
        local_sha, local_hash = client.blob_sha(path), digest_file(path)
        manager = globals().get("CHECKPOINT_BACKUPS")
        checkpoint_client = globals().get("CHECKPOINT_GITHUB")
        checkpoint = None
        if manager is not None and checkpoint_client is not None:
            checkpoint_safety_hook(force=True, reason="before_fixed_publication")
            if checkpoint_client.published_head != manager.parent or manager.pending:
                raise RuntimeError("Cannot publish a delta pointer before the recovery point is confirmed")
            checkpoint = {"repository": checkpoint_client.repository, "branch": checkpoint_client.branch,
                "root": checkpoint_client.root, "run_name": RUN_NAME, "head_sha256": manager.parent,
                "sequence": manager.sequence}
        pointer = {"schema_version": "georag-current-output-1.0", "run_name": RUN_NAME,
            "ner_schema_version": SCHEMA_VERSION, "source": SOURCE_META, "sample_sha256": SAMPLE_HASH,
            "inference_sha256": INFERENCE_HASH, "prediction_fingerprints": {n: prediction_cache_namespace(n) for n in MODELS},
            "output": {"path": target, "git_blob_sha": local_sha, "sha256": local_hash,
                       "bytes": path.stat().st_size, "rows": expected_rows},
            "checkpoint": checkpoint, "extraction_status": RUN_STATUS,
            "delta_reuse": DELTA_IMPORT_MANIFEST}
        pointer_bytes = encoded(pointer)
        pointer_sha = hashlib.sha1(f"blob {len(pointer_bytes)}\0".encode() + pointer_bytes).hexdigest()
        result = publish_git_pair(path, pointer, pointer_path, client.repository, client.branch,
            client.token, OUTPUT_ROOT, wait=github_wait)
        return {"schema_version":"github-current-publication-1.0", **result,
            "rows":expected_rows,"source":SOURCE_META,"at_published":utc_now()}


### 4b — חידוש מ־GitHub, אם התבקש

כאשר הוגדר שם ריצה: מוריד ומאמת את הגיבוי המאושר האחרון. אחרת מדלג.


In [ ]:
with cell("4b-local — prefer verified configuration of the same existing local run"):
    if PREFER_MATCHING_LOCAL_RESUME and GITHUB_RESUME_RUN_NAME and not RESUME_RUN_DIR and not RESTORE_FROM_BACKUPS:
        local_candidates = []
        if globals().get("RUN_DIR"):
            local_candidates.append(Path(globals()["RUN_DIR"]))
        local_candidates.append(OUTPUT_ROOT / GITHUB_RESUME_RUN_NAME)
        local_candidates.extend(sorted(OUTPUT_ROOT.glob("restored_*"), key=lambda p:p.stat().st_mtime, reverse=True))
        for local_candidate in local_candidates:
            run_file = local_candidate / "run.json"
            if not run_file.is_file():
                continue
            local_run = load_json(run_file)
            if local_run.get("run_name") != GITHUB_RESUME_RUN_NAME:
                continue
            config_file = local_candidate / "models/delta_source_config.json"
            if not config_file.is_file():
                raise RuntimeError("Matching local run has no saved extraction configuration; set an explicit resume source before proceeding")
            local_saved = load_json(config_file)
            restore_saved_parameters({"run_dir":str(local_candidate),"resume_config":local_saved["configuration"],
                "versions":local_saved["versions"],"head_sha256":None})
            GITHUB_RESUME_RUN_NAME = None
            print("Continuing the matching local run; committed unuploaded predictions are retained:", RESUME_RUN_DIR)
            break
        else:
            print("No matching local run found at the active/standard path; restoring confirmed GitHub checkpoint")

with cell("4b — restore the last confirmed GitHub checkpoint, if requested"):
    RESTORE_INFO = None
    if GITHUB_RESUME_RUN_NAME:
        if RESUME_RUN_DIR or RESTORE_FROM_BACKUPS:
            raise ValueError("Choose one resume method: GitHub run name, local run directory, or local packets")
        CHECKPOINT_GITHUB = CheckpointGitHub()
        RESTORE_INFO = CHECKPOINT_GITHUB.restore(GITHUB_RESUME_RUN_NAME, OUTPUT_ROOT)
        restore_saved_parameters(RESTORE_INFO)
        print("Restored run:", GITHUB_RESUME_RUN_NAME, "packet:", RESTORE_INFO["sequence"])
        print("Only missing committed shards will be inferred; incomplete/unpublished work may be repeated.")
    elif RESTORE_FROM_BACKUPS:
        RESTORE_INFO = restore_checkpoint_packets(sorted(Path(RESTORE_BACKUP_DIR).glob("*_cp_*.zip")), OUTPUT_ROOT)
        restore_saved_parameters(RESTORE_INFO)
    else:
        print("Remote restore skipped; starting a new run or continuing RESUME_RUN_DIR")


### 4a — בדיקת הזדהות לגיבוי

מאתר הזדהות זמינה בלי להדפיס טוקן. כשל עוצר לפני ההסקה המלאה.


In [ ]:
with cell("4a — GitHub checkpoint authentication before inference"):
    if ENABLE_CHECKPOINT_BACKUPS:
        if CHECKPOINT_GITHUB is None:
            CHECKPOINT_GITHUB = CheckpointGitHub()
        print("Checkpoint connection ready:", GITHUB_REPOSITORY, "branch:", CHECKPOINT_GITHUB.branch)
        print("A verified initial backup is required before full model inference.")
    else:
        print("WARNING: remote checkpoint protection explicitly disabled")


### 5 — מקור עדכני בשם קבוע והקפאת גרסת הריצה

פלט: ה־commit שנפתר אוטומטית, hashes, ספירת פסקאות ושם ריצה לחידוש.


In [ ]:
with cell("5 — GitHub Parquet input, full coverage and resume"):
    LEGACY = None
    if RESELECT_V1_ZIP:
        raise ValueError("Legacy JSON replay remains available in v8; it is not migrated into this Parquet run.")
    if RESUME_RUN_DIR:
        RUN_DIR = Path(RESUME_RUN_DIR)
        frozen = load_json(RUN_DIR / "run.json")
        SOURCE_META = frozen["source"]
        source_to_copy = RUN_DIR / "source.parquet"
        if SOURCE_META.get("format") != "parquet" or SOURCE_META["path"] != SOURCE_RELATIVE_PATH:
            raise ValueError("Resume source format/path mismatch; use the notebook that created the checkpoint")
        if SOURCE_META["repo_url"] != REPO_URL:
            raise ValueError("Resume GitHub repository differs")
        SOURCE_COMMIT = SOURCE_META["commit_sha"]  # keep the original run snapshot after a crash
    else:
        class InputNoRedirect(urllib.request.HTTPRedirectHandler):
            def redirect_request(self, req, fp, code, msg, headers, newurl):
                return None
        input_open = urllib.request.build_opener(InputNoRedirect()).open
        input_base = f"https://api.github.com/repos/{GITHUB_REPOSITORY}"
        request = urllib.request.Request(input_base + "/commits/" + urllib.parse.quote(SOURCE_REF, safe=""),
            headers={"Accept": "application/vnd.github+json", "User-Agent": "GeoRAG-v11"})
        with input_open(request, timeout=120) as response:
            SOURCE_COMMIT = json.load(response)["sha"]
        if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT):
            raise ValueError("GitHub returned an invalid resolved commit")
        print("Input ref resolved once:", SOURCE_REF, "->", SOURCE_COMMIT)
        path_quoted = urllib.parse.quote(SOURCE_RELATIVE_PATH, safe="/")
        request = urllib.request.Request(input_base + "/contents/" + path_quoted + "?" +
            urllib.parse.urlencode({"ref": SOURCE_COMMIT}),
            headers={"Accept": "application/vnd.github.object+json", "User-Agent": "GeoRAG-v9"})
        with input_open(request, timeout=120) as response:
            input_meta = json.load(response)
        if input_meta.get("type") != "file" or input_meta.get("size", 0) <= 0:
            raise ValueError("GitHub source is not a nonempty regular file")
        cache = OUTPUT_ROOT / "input_cache" / secrets.token_hex(12)
        cache.mkdir(parents=True, exist_ok=False)
        source_to_copy = cache / "corpus_paragraphs.parquet"
        source_url = f"https://raw.githubusercontent.com/{GITHUB_REPOSITORY}/{SOURCE_COMMIT}/{path_quoted}"
        request = urllib.request.Request(source_url, headers={"User-Agent": "GeoRAG-v9"})
        downloaded = 0
        with input_open(request, timeout=120) as src, source_to_copy.open("xb") as dst:
            while True:
                block = src.read(1024 * 1024)
                if not block:
                    break
                downloaded += len(block)
                if downloaded > input_meta["size"]:
                    raise ValueError("GitHub input exceeds its declared size")
                dst.write(block)
                print("Input download:", downloaded, "/", input_meta["size"], "bytes", flush=True)
            dst.flush()
            os.fsync(dst.fileno())
        if downloaded != input_meta["size"] or CheckpointGitHub.blob_sha(source_to_copy) != input_meta["sha"]:
            raise ValueError("Downloaded GitHub input failed Git blob hash/size verification")
        sha = digest_file(source_to_copy)
        SOURCE_META = {"kind": "github", "format": "parquet", "repo_url": REPO_URL,
            "path": SOURCE_RELATIVE_PATH, "commit_sha": SOURCE_COMMIT,
            "sha256": sha, "git_blob_sha": input_meta["sha"], "bytes": downloaded,
            "version_id": "git:" + SOURCE_COMMIT, "raw_url": source_url,
            "full_article_slice_validation": "not_available_in_paragraph_only_input"}
    CONFIG["source_commit"] = SOURCE_COMMIT
    if digest_file(source_to_copy) != SOURCE_META["sha256"]:
        raise ValueError("Frozen source hash mismatch")
    with pq.ParquetFile(source_to_copy) as source_parquet:
        origin_hash = (source_parquet.schema_arrow.metadata or {}).get(b"source_sha256")
    if origin_hash:
        origin_hash = origin_hash.decode("ascii")
        if not re.fullmatch(r"[0-9a-f]{64}", origin_hash):
            raise ValueError("Invalid original JSON hash in Parquet metadata")
        SOURCE_META["origin_json_sha256"] = origin_hash
    INPUT_ROWS = ParquetParagraphs(source_to_copy)
    INPUT_VALIDATION = validate_paragraph_source(INPUT_ROWS)
    TOTAL_ELIGIBLE = len(INPUT_ROWS)
    mode = "full" if PROCESS_ALL else "sample"
    if not PROCESS_ALL and not 0 < SAMPLE_SIZE <= TOTAL_ELIGIBLE:
        raise ValueError(f"Requested {SAMPLE_SIZE}; available {TOTAL_ELIGIBLE}")
    positions = None if PROCESS_ALL else random.Random(SEED).sample(range(TOTAL_ELIGIBLE), SAMPLE_SIZE)
    SAMPLE_PAYLOAD = {"schema_version": "ner-parquet-sample-index-1.0", "ner_schema_version": SCHEMA_VERSION,
        "source": SOURCE_META, "split_version": SPLIT_VERSION, "mode": mode,
        "total_eligible": TOTAL_ELIGIBLE, "actual_sample_size": TOTAL_ELIGIBLE if PROCESS_ALL else SAMPLE_SIZE,
        "positions": positions, "position_meaning": "corpus_position in frozen source.parquet"}
    SAMPLE_HASH = digest(encoded(SAMPLE_PAYLOAD))
    INFERENCE_CONFIG = {"schema_version": SCHEMA_VERSION, "source": SOURCE_META,
        "sample_sha256": SAMPLE_HASH, "models": MODELS, "windows": WINDOWS,
        "labels": GLINER_LABELS, "label_map": LABEL_MAP, "threshold_raw": GLINER_THRESHOLD,
        "text_preprocessing": TEXT_PREPROCESSING, "prediction_cache": CONFIG["prediction_cache"],
        "delta_reuse_enabled": DELTA_REUSE_ENABLED,
        "batch_execution": CONFIG["batch_execution"], "aggregation": "average", "versions": VERSIONS,
        "device": DEVICE, "checkpoint_paragraphs": CHECKPOINT_PARAGRAPHS, "legacy_sha256": None}
    INFERENCE_HASH = digest(encoded(INFERENCE_CONFIG))
    CONFIG.update(mode=mode, actual_sample_size=SAMPLE_PAYLOAD["actual_sample_size"], total_eligible=TOTAL_ELIGIBLE)
    CONFIG_HASH = digest(encoded(CONFIG))
    if RESUME_RUN_DIR:
        if frozen["inference_sha256"] != INFERENCE_HASH or load_json(RUN_DIR / "sample.json") != SAMPLE_PAYLOAD:
            raise ValueError("Resume mismatch: source/model/revision/config/library versions")
        RUN_NAME = frozen["run_name"]
    else:
        date = datetime.now(ZoneInfo("Asia/Jerusalem")).strftime("%Y%m%d")
        size_part = f"full_n{TOTAL_ELIGIBLE}" if PROCESS_ALL else f"sample_seed{SEED}_n{SAMPLE_SIZE}"
        RUN_NAME = f"{FINAL_OUTPUT_PREFIX}_{size_part}_r{RUN_RANDOM_ID}_{date}"
        RUN_DIR = OUTPUT_ROOT / RUN_NAME
        RUN_DIR.mkdir(parents=True, exist_ok=False)
        with source_to_copy.open("rb") as src, (RUN_DIR / "source.parquet").open("xb") as dst:
            shutil.copyfileobj(src, dst, length=1024 * 1024)
            dst.flush()
            os.fsync(dst.fileno())
        if digest_file(RUN_DIR / "source.parquet") != SOURCE_META["sha256"]:
            raise ValueError("Frozen source copy hash mismatch")
        save_json(RUN_DIR / "sample.json", SAMPLE_PAYLOAD)
        save_json(RUN_DIR / "run.json", {"schema_version": SCHEMA_VERSION, "run_name": RUN_NAME,
            "source": SOURCE_META, "sample_sha256": SAMPLE_HASH, "inference_sha256": INFERENCE_HASH,
            "inference_configuration": INFERENCE_CONFIG, "legacy": None})
    INPUT_ROWS = ParquetParagraphs(RUN_DIR / "source.parquet")
    SAMPLE = INPUT_ROWS if PROCESS_ALL else ParquetParagraphs(RUN_DIR / "source.parquet", positions)
    SESSION_ID = secrets.token_hex(8)
    for row in CELL_TIMINGS:
        row["run_name"] = RUN_NAME
    ACTIVE_TIMING_RUN_NAME = RUN_NAME
    PATHS = {"selftests": RUN_DIR / f"selftests_{SESSION_ID}.json"}
    EXPORT_DIR = RUN_DIR / "exports" / SESSION_ID
    EXPORT_DIR.mkdir(parents=True, exist_ok=False)
    ZIP_PATH = OUTPUT_ROOT / f"{RUN_NAME}_export_{SESSION_ID}.zip"
    with (EXPORT_DIR / "environment.txt").open("x", encoding="utf-8") as f:
        f.write(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
    with (EXPORT_DIR / "installation_log.txt").open("x", encoding="utf-8") as f:
        f.write(INSTALLATION_LOG)
    print("Mode:", mode, "eligible paragraphs:", TOTAL_ELIGIBLE, "planned:", len(SAMPLE))
    print("Source:", SOURCE_META)
    print("Input validation scope:", INPUT_VALIDATION)
    print("Run directory:", RUN_DIR)
    print("To resume after VM loss, set GITHUB_RESUME_RUN_NAME =", repr(RUN_NAME))


### 5a — checkpoints, דלתאות ומטמון

פלט: תאימות התחזיות הקודמות, ספירת טקסטים חדשים/שהוסרו ויצירת גיבוי בסיס.


In [ ]:
with cell("5a — checkpoint engine"):
    from contextlib import ExitStack

    def stream_digest(handle):
        h=hashlib.sha256()
        for block in iter(lambda:handle.read(1024*1024),b""):
            h.update(block)
        return h.hexdigest()


    @contextmanager
    def contextmanager_export_writers():
        with ExitStack() as stack:
            writers = {}
            for kind, cols, schema in [
                ("selected", COLUMNS, ARROW_SCHEMA),
                ("candidates", CANDIDATE_COLUMNS, CANDIDATE_SCHEMA),
            ]:
                handle = stack.enter_context(OUTPUT_FILES[kind+"_parquet"].open("xb"))
                writer = stack.enter_context(pq.ParquetWriter(handle, schema))
                writers[kind] = (cols, schema, writer)
            yield writers
    
    def verify_parquet_roundtrip(kind):
        cols = COLUMNS if kind == "selected" else CANDIDATE_COLUMNS
        def expected_rows():
            index = load_json(OUTPUT_FILES[kind+"_json"])
            for part in index["parts"]:
                yield from load_json(EXPORT_DIR/part["path"])["records"]
        expected = iter(expected_rows())
        count = 0
        with OUTPUT_FILES[kind+"_parquet"].open("rb") as handle:
            parquet = pq.ParquetFile(handle)
            assert parquet.schema_arrow == (ARROW_SCHEMA if kind == "selected" else CANDIDATE_SCHEMA)
            for batch in parquet.iter_batches(batch_size=1024):
                for row in batch.to_pylist():
                    detail = next(expected)
                    assert row == {key: detail.get(key) for key in cols}
                    count += 1
            assert next(expected, None) is None
            assert parquet.metadata.num_rows == count
        return count
    def atomic_json(path, obj):
        """Single writer only. A .tmp file is never a committed checkpoint."""
        path = Path(path)
        if path.exists():
            raise FileExistsError(path)
        temp = path.with_name(path.name+"."+secrets.token_hex(6)+".tmp")
        with temp.open("xb") as f:
            f.write(encoded(obj)); f.flush(); os.fsync(f.fileno())
        if path.exists():
            raise FileExistsError(path)
        os.rename(temp, path)

    install_compact_io(globals())  # must precede the first restored-shard scan

    def shard_path(name, start):
        return RUN_DIR/"models"/name/f"part_{start:08d}.json"

    def load_shard(name, start):
        path = shard_path(name, start)
        data = load_json(path)
        assert data["inference_sha256"] == INFERENCE_HASH
        assert data["sample_sha256"] == SAMPLE_HASH and data["model"] == name
        expected = SAMPLE[start:start+CHECKPOINT_PARAGRAPHS]
        assert [r["paragraph"] for r in data["results"]] == expected
        assert data["payload_sha256"] == digest(encoded(data["results"]))
        return data

    def scan_model(name):
        covered, totals = set(), Counter()
        for start in range(0, len(SAMPLE), CHECKPOINT_PARAGRAPHS):
            if not shard_path(name, start).exists():
                continue
            data = load_shard(name, start)
            for r in data["results"]:
                pos = r["paragraph"]["sample_position"]
                assert pos not in covered
                covered.add(pos)
                totals[r["status_processing"]] += 1
        return covered, totals

    def model_complete(name):
        return len(scan_model(name)[0]) == len(SAMPLE)

    def run_model(name, predict, measure, limit, metadata):
        global ACTIVE_RAW_FILE, ACTIVE_RAW_INDEX
        if CACHE_ENABLED:
            prediction_cache(name).reset_for_run()
        folder = RUN_DIR/"models"/name
        folder.mkdir(parents=True, exist_ok=True)
        metadata = {**metadata, "device": DEVICE, "window": WINDOWS[name],
            "effective_encoded_limit": limit, "threshold": GLINER_THRESHOLD if name == "gliner" else None,
            "model_id": MODELS[name]["id"], "versions": VERSIONS,
            "prompt": None, "version_prompt": None}
        metadata_path = folder/"metadata.json"
        if not metadata_path.exists():
            atomic_json(metadata_path, metadata)
        else:
            prior = load_json(metadata_path)
            for k in ["revision_model", "revision_tokenizer", "labels", "effective_encoded_limit", "model_id", "versions"]:
                assert prior[k] == metadata[k], f"Model metadata differs: {k}"
        covered, totals = scan_model(name)
        started = time.perf_counter()
        failures = totals["error"]+totals["partial_invalid"]
        last = progress(name, len(covered), len(SAMPLE), failures, started, 0)
        try:
            for start in range(0, len(SAMPLE), CHECKPOINT_PARAGRAPHS):
                path = shard_path(name, start)
                if path.exists():
                    continue
                ACTIVE_RAW_FILE = path.relative_to(RUN_DIR).as_posix()
                records = []
                for local, p in enumerate(SAMPLE[start:start+CHECKPOINT_PARAGRAPHS]):
                    ACTIVE_RAW_INDEX = local
                    result = process_paragraph(p, name, predict, measure, limit)
                    records.append(result)
                    totals[result["status_processing"]] += 1
                    covered.add(p["sample_position"])
                    failures += result["status_processing"] != "success"
                    last = progress(name, len(covered), len(SAMPLE), failures, started, last)
                atomic_json(path, {"schema_version": SCHEMA_VERSION, "model": name,
                    "sample_sha256": SAMPLE_HASH, "inference_sha256": INFERENCE_HASH,
                    "payload_sha256": digest(encoded(records)), "results": records})
                if CACHE_ENABLED:
                    prediction_cache(name).register(ACTIVE_RAW_FILE, records)
                del records
                records = []
                checkpoint_safety_hook(reason=name+"_checkpoint_committed")
        finally:
            active_error = sys.exc_info()[1]
            if active_error and 'records' in locals() and records:
                diagnostic=folder/f"incomplete_{SESSION_ID}_{secrets.token_hex(6)}.json"
                saved_records=copy.deepcopy(records)
                for saved in saved_records:
                    if saved.get("raw_shared_ref"):
                        saved["raw_shared_ref"] = saved["raw_shared_ref"].replace(ACTIVE_RAW_FILE+"#", diagnostic.relative_to(RUN_DIR).as_posix()+"#", 1)
                    for m in saved["mentions"]:
                        m["ref_output_raw"]=[ref.replace(ACTIVE_RAW_FILE+"#",diagnostic.relative_to(RUN_DIR).as_posix()+"#",1)
                                             for ref in m["ref_output_raw"]]
                atomic_json(diagnostic,{"schema_version":"ner-incomplete-checkpoint-1.0",
                    "original_target":ACTIVE_RAW_FILE,"committed":False,
                    "sample_sha256":SAMPLE_HASH,"results":saved_records,
                    "error":f"{type(active_error).__name__}: {active_error}"})
            attempt = {"model": name, "session_id": SESSION_ID, "load_seconds": metadata["load_seconds"],
                "inference_session_seconds": time.perf_counter()-started,
                "committed_paragraphs": len(scan_model(name)[0]), "at_processed": utc_now(),
                "status_processing": "interrupted_or_failed" if active_error else "success",
                "error": f"{type(active_error).__name__}: {active_error}" if active_error else None}
            atomic_json(folder/f"attempt_{SESSION_ID}_{secrets.token_hex(4)}.json", attempt)
        MODEL_OUTPUTS[name] = {"processing_seconds": sum(
            r["processing_seconds"] for start in range(0,len(SAMPLE),CHECKPOINT_PARAGRAPHS)
            for r in load_shard(name,start)["results"])}

    def import_legacy(name):
        folder = RUN_DIR/"models"/name
        folder.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(LEGACY["path"]) as z:
            old = json.loads(z.read(LEGACY["members"][name]))
        assert old["source"] == SOURCE_META
        assert old["model_id"] == MODELS[name]["id"]
        assert old["metadata"]["revision_model"] == MODELS[name]["revision"]
        assert old["metadata"]["window"] == WINDOWS[name]
        if name == "gliner":
            assert old["metadata"]["threshold"] == GLINER_THRESHOLD
            assert old["metadata"]["labels"] == GLINER_LABELS
        assert len(old["results"]) == len(SAMPLE)
        if not (folder/"metadata.json").exists():
            atomic_json(folder/"metadata.json", {**old["metadata"], "model_id": old["model_id"],
                "imported_legacy": True, "legacy_sha256": LEGACY["sha256"]})
        for start in range(0,len(SAMPLE),CHECKPOINT_PARAGRAPHS):
            if shard_path(name,start).exists():
                load_shard(name,start)
                continue
            results = old["results"][start:start+CHECKPOINT_PARAGRAPHS]
            for local,r in enumerate(results):
                assert r["paragraph"] == SAMPLE[start+local]
                for m in r["mentions"]:
                    m["legacy_ref_output_raw"] = list(m["ref_output_raw"])
                    m["ref_output_raw"] = [
                        shard_path(name,start).relative_to(RUN_DIR).as_posix()+f"#/results/{local}/raw/"+ref.split("/raw/",1)[1]
                        for ref in m["ref_output_raw"]]
            atomic_json(shard_path(name,start), {"schema_version": SCHEMA_VERSION, "model": name,
                "sample_sha256": SAMPLE_HASH, "inference_sha256": INFERENCE_HASH,
                "payload_sha256": digest(encoded(results)), "results": results})
        del old

    if LEGACY:
        for name in MODELS:
            if not model_complete(name):
                import_legacy(name)
    for name in MODELS:
        print(name, "committed:", len(scan_model(name)[0]), "/", len(SAMPLE))

    """Batch adapters and gates, embedded in the notebook; no service or new model."""
    KNOWN_LONG_UNIT_CASES = [
        ("Arthur Blackburn",261),("Japanese war crimes",196),("Taiwan under Japanese rule",164),
        ("Type 100 submachine gun",48),("USS LST-578",80),
        *[("USS Wolverine (IX-64)",i) for i in [123,124,126,127,130,131,133,137]],
    ]


    class BatchPredictionFailure:
        def __init__(self, error):
            self.error = error


    class AdaptiveBatchRunner:
        def __init__(self, name, single, batch, batch_size, event_sink=None):
            self.name,self.single,self.batch=name,single,batch
            self.size=batch_size
            self.event_sink=event_sink
            self.stats=Counter()

        def event(self, exc, tasks):
            self.stats['recovered_batch_exceptions']+=1
            event={'schema_version':'ner-batch-event-1.0','model':self.name,
                'error':f'{type(exc).__name__}: {exc}','batch_size':len(tasks),
                'tasks':[{k:t.get(k) for k in ['paragraph_position','window_id']} for t in tasks],
                'at_processed':utc_now()}
            if self.event_sink:self.event_sink(event)
            print(f"{self.name}: batch of {len(tasks)} failed ({type(exc).__name__}); reducing/isolation fallback",flush=True)

        def execute(self,tasks):
            if not tasks:return []
            try:
                with torch.inference_mode():
                    if len(tasks)==1:
                        result=[self.single(tasks[0]['text'])]
                    else:
                        result=self.batch([t['text'] for t in tasks],len(tasks))
                if not isinstance(result,list) or len(result)!=len(tasks):
                    raise ValueError('Batch API must return exactly one ordered result per input window')
                self.stats['successful_windows']+=len(tasks)
                return result
            except Exception as exc:
                self.event(exc,tasks)
                oom=isinstance(exc,torch.cuda.OutOfMemoryError)
                if oom:
                    gc.collect()
                    if torch.cuda.is_available():torch.cuda.empty_cache()
                if len(tasks)==1:
                    if oom:
                        raise FatalWindowGPUOOM("GPU memory exhausted even for one window. Committed checkpoints are retained; restore on a fresh GPU runtime.") from exc
                    self.stats['failed_windows']+=1
                    return [BatchPredictionFailure(f'{type(exc).__name__}: {exc}')]
                if oom:
                    self.size=min(self.size,max(1,len(tasks)//2))
                    middle=max(1,len(tasks)//2)
                    return self.execute(tasks[:middle])+self.execute(tasks[middle:])
                # An input-specific API error must not lose other inputs in the batch.
                self.size=1
                return [self.execute([task])[0] for task in tasks]

        def predict(self,tasks):
            result=[]
            cursor=0
            while cursor<len(tasks):
                part=tasks[cursor:cursor+self.size]
                result.extend(self.execute(part))
                cursor+=len(part)
            return result


    def batch_event_sink(name):
        folder=RUN_DIR/'models'/name
        folder.mkdir(parents=True,exist_ok=True)
        return lambda event:atomic_json(folder/f'batch_event_{SESSION_ID}_{secrets.token_hex(6)}.json',event)


    @contextmanager
    def load_ner_adapter(name):
        """The same pinned models, FP32 defaults, labels and decoding options as v6."""
        started=time.perf_counter()
        adapter={}
        model=tokenizer=ner=schema=single=batch=measure=None
        try:
            revision=hf_revision(name)
            if name=='bert':
                tokenizer=AutoTokenizer.from_pretrained(MODELS[name]['id'],revision=revision,use_fast=True)
                assert tokenizer.is_fast,'A fast tokenizer is required'
                model=AutoModelForTokenClassification.from_pretrained(MODELS[name]['id'],revision=revision)
                ner=pipeline('ner',model=model,tokenizer=tokenizer,aggregation_strategy='average',
                    device=0 if DEVICE.startswith('cuda') else -1)
                limit=min(int(tokenizer.model_max_length),int(model.config.max_position_embeddings))
                single=lambda text:ner(text)
                batch=lambda texts,size:ner(texts,batch_size=size,num_workers=0)
                measure=tokenizer_measure(tokenizer)
                metadata={'revision_model':revision,'revision_tokenizer':revision,
                    'labels':sorted(set(model.config.id2label.values())),
                    'unsupported':['DATE','TIME','EVENT'],'aggregation_strategy':'average'}
                print('BERT device:',ner.device,'revision:',revision,'token limit:',limit)
            elif name=='gliner':
                directory=snapshot_download(MODELS[name]['id'],revision=revision)
                model=AutoExtractor.from_pretrained(directory).to(DEVICE)
                model.eval()
                model.strict_extraction=True
                schema=model.create_schema().entities(GLINER_LABELS)
                limit=int(model.encoder.config.max_position_embeddings)
                measure=lambda text:len(model.processor.transform_record(text,schema,max_len=None).input_ids)
                single=lambda text:model.extract_entities(text,GLINER_LABELS,threshold=GLINER_THRESHOLD,
                    include_spans=True,include_confidence=True,max_len=None,overlap_policy='allow')
                if not callable(getattr(model,'batch_extract_entities',None)):
                    raise RuntimeError('This GLiNER adapter requires batch_extract_entities; choose sequential mode or supply an adapter')
                batch=lambda texts,size:model.batch_extract_entities(texts,GLINER_LABELS,batch_size=size,
                    threshold=GLINER_THRESHOLD,include_spans=True,include_confidence=True,
                    max_len=None,overlap_policy='allow')
                metadata={'revision_model':revision,'revision_tokenizer':revision,
                    'model_files_sha256':{str(p.relative_to(directory)):digest_file(p)
                        for p in sorted(Path(directory).rglob('*')) if p.is_file()},
                    'labels':GLINER_LABELS,'overlap_policy':'allow','max_len':None,'unsupported':[]}
                print('GLiNER device:',next(model.parameters()).device,'revision:',revision,'token limit:',limit)
            else:raise ValueError(f'No batch adapter registered for {name!r}')
            metadata['load_seconds']=time.perf_counter()-started
            adapter.update(single=single,batch=batch,measure=measure,limit=limit,metadata=metadata)
            print(name,'model load seconds:',metadata['load_seconds'])
            yield adapter
        finally:
            adapter.clear()
            single=batch=measure=ner=model=tokenizer=schema=None
            release()


    def build_preflight_paragraphs():
        wanted = {}
        known = set(KNOWN_LONG_UNIT_CASES)
        for p in INPUT_ROWS:
            key = p["Source"], p["Index"]
            if key in known and not prepare_inference_text(p['Text'])[1]['skipped_no_content']:
                wanted[key] = p
                if len(wanted) == len(known):
                    break
        positions = random.Random(SEED).sample(range(len(SAMPLE)), min(PREFLIGHT_PARAGRAPHS, len(SAMPLE)))
        for pos in positions:
            p = SAMPLE[pos]
            if len(p["Text"]) <= PREFLIGHT_MAX_PARAGRAPH_CHARS and not prepare_inference_text(p['Text'])[1]['skipped_no_content']:
                wanted.setdefault((p["Source"], p["Index"]), p)
        if not wanted:
            raise ValueError("No preflight paragraphs available")
        return [{**p, "sample_position": i} for i, p in enumerate(wanted.values())]


    def validate_preflight_windows(text,settings,measure,limit):
        result=list(windows(text,settings,measure,limit))
        through=0
        for w in result:
            assert w['Text']==text[w['start']:w['end']]
            assert w['start']<=through,'Gap in window character coverage'
            assert measure(w['Text'])==w['encoded_tokens']<=min(settings['max_encoded_tokens'],limit)
            through=max(through,w['end'])
        assert through==len(text),'Incomplete window character coverage'
        return result


    def replay_preflight_results(name,paragraphs,plans,outputs,path,measure,limit):
        global ACTIVE_RAW_FILE,ACTIVE_RAW_INDEX
        ACTIVE_RAW_FILE=path.relative_to(RUN_DIR).as_posix()
        cursor=0;results=[]
        for i,p in enumerate(paragraphs):
            ACTIVE_RAW_INDEX=i
            local_outputs=outputs[cursor:cursor+len(plans[i])]
            local_inputs=[w['Text'] for w in plans[i]]
            cursor+=len(local_inputs)
            queue=iter(zip(local_inputs,local_outputs))
            def cached(text):
                expected,value=next(queue)
                assert expected==text,'Batch output order mismatch'
                if isinstance(value,BatchPredictionFailure):raise RuntimeError(value.error)
                return value
            record=process_paragraph(p,name,cached,measure,limit)
            assert next(queue,None) is None
            results.append(record)
        assert cursor==len(outputs)
        atomic_json(path,{'schema_version':'ner-batch-preflight-1.0','model':name,
            'source':SOURCE_META,'mode':'batch' if '_batch_' in path.name else 'single',
            'results':results})
        return results


    def comparison_signature(rows):
        # Include selection order/priority, boundaries and source identity; scores compared separately.
        return [(r['Source'],r['Index'],r['entity_group'],r['start'],r['end'],r['word'],
            r.get('selected_model'),r.get('selection_status'),r.get('selection_reason'),
            tuple(r.get('supporting_models',[])),r.get('subtype')) for r in rows]


    def compare_model_predictions(name,paragraphs,single_results,batch_results):
        max_delta=0.0
        for p,ref,actual in zip(paragraphs,single_results,batch_results):
            assert ref['status_processing']==actual['status_processing'],f'{name}: processing status changed'
            assert ref['status_processing']!='error',f'{name}: preflight extraction failed'
            assert ref['errors']==actual['errors'],f'{name}: validation errors changed'
            def key(m):return (m['label_original'],str(m['start']),str(m['end']),str(m['word']),m['status_validation'])
            a=sorted(ref['mentions'],key=key);b=sorted(actual['mentions'],key=key)
            assert [key(m) for m in a]==[key(m) for m in b],f'{name}: mention labels/spans changed in {p["Source"]}'
            # Also compare every raw window prediction, not only window-deduplicated mentions.
            assert len(ref['raw'])==len(actual['raw'])
            for wr,wb in zip(ref['raw'],actual['raw']):
                ar=adapt_output(name,wr['output'],wr['Text']);ab=adapt_output(name,wb['output'],wb['Text'])
                k=lambda x:(x['label'],str(x['start']),str(x['end']),str(x.get('text')))
                ar=sorted(ar,key=k);ab=sorted(ab,key=k)
                assert [k(x) for x in ar]==[k(x) for x in ab],f'{name}: raw prediction spans changed'
                for x,y in zip(ar,ab):
                    if x['score'] is None or y['score'] is None:
                        assert x['score'] is y['score']
                    else:
                        delta=abs(float(x['score'])-float(y['score']))
                        assert math.isfinite(delta) and delta<=BATCH_SCORE_ATOL,f'{name}: score drift {delta} exceeds tolerance'
                        max_delta=max(max_delta,delta)
        return max_delta


    def make_shard_batch_predictor(name,adapter,batch_size):
        """Keep at most one look-ahead batch; do not infer an entire shard up front."""
        runner=AdaptiveBatchRunner(name,adapter['single'],adapter['batch'],batch_size,batch_event_sink(name))
        state={'file':None,'iterator':None,'counts':OrderedDict(),'queues':{},'peak_buffered_windows':0}
        def reset():
            active=globals().get('ACTIVE_RAW_FILE')
            if state['file']!=active:
                if any(state['queues'].values()):raise RuntimeError('Unconsumed batch outputs at checkpoint boundary')
                state.update(file=active,iterator=None,counts=OrderedDict(),queues={})
        def measure(text):
            reset()
            if text in state['counts']:
                state['counts'].move_to_end(text)
                return state['counts'][text]
            count=adapter['measure'](text)
            if len(text)<=20000:
                state['counts'][text]=count
                while len(state['counts'])>TOKEN_COUNT_CACHE_SIZE:state['counts'].popitem(last=False)
            return count
        def task_iterator():
            match=re.fullmatch(r'part_(\d+)\.json',Path(state['file']).name)
            if not match:raise ValueError('Expected a checkpoint part filename')
            start=int(match.group(1))
            seen = set()
            for local,p in enumerate(SAMPLE[start:start+CHECKPOINT_PARAGRAPHS]):
                cache_key = digest(p['Text'].encode('utf-8'))
                if CACHE_ENABLED and (cache_key in seen or prediction_cache(name).contains(p['Text'])):
                    continue
                seen.add(cache_key)
                cleaned, normalization = prepare_inference_text(p['Text'])
                if normalization['skipped_no_content']:
                    continue
                try:
                    for wi,w in enumerate(windows(cleaned,WINDOWS[name],measure,adapter['limit'])):
                        yield {'paragraph_position':p['sample_position'],'local':local,'window_id':wi,'text':w['Text']}
                except Exception:
                    # The original processor replays this splitter and retains its exception.
                    pass
        def predict(text):
            reset()
            if state['iterator'] is None:state['iterator']=iter(task_iterator())
            queue=state['queues'].get(ACTIVE_RAW_INDEX)
            if not queue:
                tasks=list(islice(state['iterator'],runner.size))
                if not tasks:raise RuntimeError('Missing batch task for current paragraph/window')
                outputs=runner.predict(tasks)
                for task,value in zip(tasks,outputs):
                    state['queues'].setdefault(task['local'],deque()).append((task['text'],value))
                buffered=sum(len(q) for q in state['queues'].values())
                state['peak_buffered_windows']=max(state['peak_buffered_windows'],buffered)
                queue=state['queues'].get(ACTIVE_RAW_INDEX)
            if not queue:raise RuntimeError('Batch task order differs from source occurrence order')
            expected,value=queue.popleft()
            if not queue:state['queues'].pop(ACTIVE_RAW_INDEX,None)
            assert expected==text,'Batch mapping mismatch'
            if isinstance(value,BatchPredictionFailure):raise RuntimeError(value.error)
            return value
        return predict,measure,runner,state


    def run_model_with_batch_adapter(name):
        assert PREFLIGHT_STATUS['status']=='passed','Run the real-model preflight before full inference'
        if model_complete(name):
            print(name,': checkpoint coverage complete; model loading skipped')
            return
        with load_ner_adapter(name) as adapter:
            metadata={**adapter['metadata'],
                'execution_mode':'checkpoint_window_batches' if BATCH_ENABLED else 'sequential',
                'batch_size_requested':BATCH_SIZES[name] if BATCH_ENABLED else 1,
                'preflight_report':f'models/batch_preflight_{SESSION_ID}.json',
                'paragraph_timing_scope':'Measured paragraph function wall time; with batching, the first prediction '
                    'in a shard includes prefetch of other paragraph windows. Use shard/model totals, not individual paragraph timings, for comparisons.'}
            if not BATCH_ENABLED:
                run_model(name,adapter['single'],adapter['measure'],adapter['limit'],metadata)
            else:
                size=PREFLIGHT_EFFECTIVE_SIZES[name]
                predict,measure,runner,state=make_shard_batch_predictor(name,adapter,size)
                try:
                    run_model(name,predict,measure,adapter['limit'],metadata)
                finally:
                    atomic_json(RUN_DIR/'models'/name/f'batch_runtime_{SESSION_ID}_{secrets.token_hex(4)}.json',
                        {'schema_version':'ner-batch-runtime-1.0','model':name,
                         'requested_batch_size':BATCH_SIZES[name],'preflight_batch_size':size,
                         'last_effective_batch_size':runner.size,'statistics':dict(runner.stats),
                         'at_processed':utc_now()})
                    pending=[{"paragraph_slot":local,"text":text,
                        "output":None if isinstance(value,BatchPredictionFailure) else value,
                        "error":value.error if isinstance(value,BatchPredictionFailure) else None}
                        for local,queue in state.get('queues',{}).items() for text,value in queue]
                    if pending:
                        atomic_json(RUN_DIR/'models'/name/f'prefetch_unconsumed_{SESSION_ID}_{secrets.token_hex(4)}.json',
                            {"schema_version":"ner-unconsumed-prefetch-1.0","model":name,
                             "checkpoint_target":state.get('file'),"windows":pending})
                    state.clear()
                    del predict,measure,runner,state
            print(name,'paragraph function total seconds:',MODEL_OUTPUTS[name]['processing_seconds'])

    with cell("5a-compact — lossless migration, independent backup, restore verification and cleanup"):
        compact_initialize(globals())
        reader_file = EXPORT_DIR / "compact_checkpoint_reader.py"
        if not reader_file.exists():
            reader_file.write_text(COMPACT_READER_SOURCE, encoding="utf-8")

    RUN_MODEL_BATCH_V9 = run_model_with_batch_adapter
    def run_model_with_batch_adapter(name):
        try:
            return RUN_MODEL_BATCH_V9(name)
        finally:
            cache = MODEL_CACHES.pop(name, None)
            if cache is not None:
                cache.close()
            RAW_REFERENCE_CACHE.clear()

    """Model-specific window batches over prepared uniques, then occurrence reconstruction."""
    import zlib


    def unique_shard_path(name, start):
        return RUN_DIR / "models" / name / f"unique_part_{start:08d}.json"


    def load_unique_shard(name, start):
        data = load_json(unique_shard_path(name, start))
        expected = list(range(start, min(start + CHECKPOINT_PARAGRAPHS, len(PREPARED))))
        if (data.get("schema_version") != "ner-prepared-unique-output-1.0" or data["model"] != name
                or data["prepared_key"] != PREPARED.key or data["inference_sha256"] != INFERENCE_HASH
                or data["payload_sha256"] != digest(encoded(data["results"]))
                or [r["prepared_unique_id"] for r in data["results"]] != expected):
            raise ValueError("Unique model checkpoint identity/count/hash mismatch")
        for result in data["results"]:
            row = PREPARED[result["prepared_unique_id"]]
            if result["paragraph"] != SAMPLE[row["representative_sample_position"]]:
                raise ValueError("Unique checkpoint representative differs from current source")
        return data


    def load_prediction_owner(filename):
        match = re.fullmatch(r"models/(bert|gliner)/(part|unique_part|cache_import)_(\d+)\.json", filename)
        if not match:
            raise ValueError("Unexpected prediction owner path")
        name, kind, number = match.group(1), match.group(2), int(match.group(3))
        if kind == "part":
            return load_shard(name, number)
        if kind == "unique_part":
            return load_unique_shard(name, number)
        chunk = load_json(RUN_DIR / filename)
        if (chunk.get("schema_version") != "ner-delta-cache-import-1.0" or chunk["model"] != name
                or chunk["prediction_fingerprint"] != prediction_cache_namespace(name)
                or chunk["payload_sha256"] != digest(encoded(chunk["results"]))):
            raise ValueError("Imported prediction owner hash/config mismatch")
        return chunk


    def unique_model_complete(name):
        return PREPARED is not None and all(unique_shard_path(name, start).is_file()
            for start in range(0, len(PREPARED), CHECKPOINT_PARAGRAPHS))


    def make_prepared_batch_predictor(name, adapter, batch_size):
        state = {"file": None, "iterator": None, "queues": {}, "computed": {}, "peak_buffered_windows": 0}
        class RecordedBatchRunner(AdaptiveBatchRunner):
            def execute(self, tasks):
                values = super().execute(tasks)
                # Successful smaller sub-batches remain inspectable even if a later
                # fallback sub-batch aborts the surrounding paragraph/checkpoint.
                for task, value in zip(tasks, values):
                    state["computed"][task["prepared_unique_id"], task["window_id"]] = {
                        **task, "output": None if isinstance(value, BatchPredictionFailure) else value,
                        "error": value.error if isinstance(value, BatchPredictionFailure) else None}
                return values
        runner = RecordedBatchRunner(name, adapter["single"], adapter["batch"], batch_size, batch_event_sink(name))

        def task_iterator(start):
            for local, row in enumerate(PREPARED[start:start + CHECKPOINT_PARAGRAPHS]):
                if not row["eligible"] or prediction_cache(name).contains(row["Text"]):
                    continue
                try:
                    for wi, window in enumerate(windows(row["Text_inference"], WINDOWS[name], adapter["measure"], adapter["limit"])):
                        yield {"local": local, "prepared_unique_id": row["unique_id"], "window_id": wi,
                            "paragraph_position": row["representative_sample_position"], "text": window["Text"],
                            "start": window["start"], "end": window["end"]}
                except Exception:
                    # process_uncached_occurrence records the actual splitter error for this unique text.
                    continue

        def predict(text):
            if state["file"] != ACTIVE_RAW_FILE:
                if any(state["queues"].values()):
                    raise RuntimeError("Unconsumed unique batch output at checkpoint boundary")
                match = re.fullmatch(r"unique_part_(\d+)\.json", Path(ACTIVE_RAW_FILE).name)
                if not match:
                    raise ValueError("Expected a prepared unique checkpoint path")
                state.update(file=ACTIVE_RAW_FILE, iterator=iter(task_iterator(int(match.group(1)))), queues={}, computed={})
            queue = state["queues"].get(ACTIVE_RAW_INDEX)
            if not queue:
                tasks = list(islice(state["iterator"], runner.size))
                if not tasks:
                    raise RuntimeError("Missing prepared unique window task")
                outputs = runner.predict(tasks)
                for task, value in zip(tasks, outputs):
                    state["queues"].setdefault(task["local"], deque()).append((task["text"], value))
                buffered = sum(len(q) for q in state["queues"].values())
                state["peak_buffered_windows"] = max(state["peak_buffered_windows"], buffered)
                queue = state["queues"].get(ACTIVE_RAW_INDEX)
            if not queue:
                raise RuntimeError("Unique batch task order differs from the prepared source")
            expected, value = queue.popleft()
            if not queue:
                state["queues"].pop(ACTIVE_RAW_INDEX, None)
            if expected != text:
                raise ValueError("Prepared window/input mapping mismatch")
            if isinstance(value, BatchPredictionFailure):
                raise RuntimeError(value.error)
            return value

        return predict, runner, state


    def run_prepared_uniques(name, adapter, timing):
        global ACTIVE_RAW_FILE, ACTIVE_RAW_INDEX
        folder = RUN_DIR / "models" / name
        folder.mkdir(parents=True, exist_ok=True)
        cache = prediction_cache(name)
        cache.reset_for_run()
        metadata = {**adapter["metadata"], "device": DEVICE, "window": WINDOWS[name], "prepared_key": PREPARED.key,
            "model_id": MODELS[name]["id"], "effective_encoded_limit": adapter["limit"], "versions": VERSIONS,
            "threshold": GLINER_THRESHOLD if name == "gliner" else None,
            "execution_mode": "prepared_unique_window_batches", "prompt": None, "version_prompt": None}
        path = folder / "metadata.json"
        if path.exists():
            prior = load_json(path)
            for key in ["revision_model", "revision_tokenizer", "labels", "effective_encoded_limit", "model_id", "versions", "prepared_key"]:
                if prior[key] != metadata[key]:
                    raise ValueError("Resumed model metadata differs: " + key)
        else:
            atomic_json(path, metadata)
        size = PREFLIGHT_EFFECTIVE_SIZES[name] if BATCH_ENABLED and DEVICE.startswith("cuda") else 1
        predict, runner, state = make_prepared_batch_predictor(name, adapter, size)
        started = time.perf_counter()
        done = failures = last = 0
        records = []
        try:
            for start in range(0, len(PREPARED), CHECKPOINT_PARAGRAPHS):
                target = unique_shard_path(name, start)
                if target.exists():
                    completed = load_unique_shard(name, start)["results"]
                    done += len(completed)
                    failures += sum(r["status_processing"] != "success" for r in completed)
                    continue
                ACTIVE_RAW_FILE = target.relative_to(RUN_DIR).as_posix()
                records = []
                for local, row in enumerate(PREPARED[start:start + CHECKPOINT_PARAGRAPHS]):
                    ACTIVE_RAW_INDEX = local
                    p = SAMPLE[row["representative_sample_position"]]
                    point = time.perf_counter()
                    hit = cache.lookup(row["Text"])
                    if hit is not None:
                        filename, slot, owner = hit
                        result = clone_occurrence(owner, p, filename, slot)
                        result["processing_seconds"] = time.perf_counter() - point
                        timing["unique_cache_reuse_seconds"] += result["processing_seconds"]
                    else:
                        result = process_uncached_occurrence(p, name, predict, adapter["measure"], adapter["limit"], prepared=row)
                        if row["eligible"]:
                            timing["inference_seconds"] += time.perf_counter() - point
                        else:
                            timing["skip_no_content_seconds"] += time.perf_counter() - point
                    result["prepared_unique_id"] = row["unique_id"]
                    result["cache_key"] = cache.key(row["Text"])
                    records.append(result)
                    done += 1
                    failures += result["status_processing"] != "success"
                saving = time.perf_counter()
                atomic_json(target, {"schema_version": "ner-prepared-unique-output-1.0", "ner_schema_version": SCHEMA_VERSION,
                    "prepared_key": PREPARED.key, "sample_sha256": SAMPLE_HASH, "inference_sha256": INFERENCE_HASH,
                    "model": name, "payload_sha256": digest(encoded(records)), "results": records})
                cache.register(ACTIVE_RAW_FILE, records)
                timing["save_unique_seconds"] += time.perf_counter() - saving
                records = []
                state["computed"].clear()
                checkpoint_safety_hook(reason=name + "_unique_checkpoint_committed")
                last = progress(name + "_unique", done, len(PREPARED), failures, started, last)
        finally:
            active = sys.exc_info()[1]
            if active and (records or state["computed"]):
                diagnostic = folder / f"incomplete_unique_{SESSION_ID}_{secrets.token_hex(4)}.json"
                saved = copy.deepcopy(records)
                for record in saved:
                    if record.get("raw_shared_ref"):
                        record["raw_shared_ref"] = record["raw_shared_ref"].replace(ACTIVE_RAW_FILE + "#", diagnostic.relative_to(RUN_DIR).as_posix() + "#", 1)
                    for m in record["mentions"]:
                        m["ref_output_raw"] = [r.replace(ACTIVE_RAW_FILE + "#", diagnostic.relative_to(RUN_DIR).as_posix() + "#", 1) for r in m["ref_output_raw"]]
                atomic_json(diagnostic, {"schema_version": "ner-incomplete-prepared-unique-1.0", "committed": False,
                    "prepared_key": PREPARED.key, "results": saved, "computed_windows": list(state["computed"].values()),
                    "error": f"{type(active).__name__}: {active}"})
            timing["unique_phase_wall_seconds"] += time.perf_counter() - started
            atomic_json(folder / f"batch_runtime_{SESSION_ID}_{secrets.token_hex(4)}.json", {
                "schema_version": "ner-prepared-batch-runtime-1.0", "model": name,
                "statistics": dict(runner.stats), "effective_batch_size": runner.size,
                "peak_buffered_windows": state["peak_buffered_windows"], "at_processed": utc_now()})


    class OccurrenceProjection:
        """Bounded lookup of small record projections; immutable unique raw remains authoritative."""
        def __init__(self, name):
            self.name = name
            self.db = sqlite3.connect(RUN_DIR / "runtime_cache" / f"projection_{name}.sqlite")
            self.db.execute("PRAGMA cache_size=-32768")
            self.db.execute("CREATE TABLE IF NOT EXISTS rows(uid INTEGER PRIMARY KEY, payload BLOB NOT NULL)")
            self.db.execute("DELETE FROM rows")
            self.cache = OrderedDict()
            self.counts = Counter()
            self.inference_processing_seconds = 0.0
            for start in range(0, len(PREPARED), CHECKPOINT_PARAGRAPHS):
                filename = unique_shard_path(name, start).relative_to(RUN_DIR).as_posix()
                chunk = load_unique_shard(name, start)
                for slot, r in enumerate(chunk["results"]):
                    projection = {k: v for k, v in r.items() if k != "raw"}
                    projection["raw"] = []
                    owner_file, owner_slot = filename, slot
                    if r.get("raw_shared_ref"):
                        owner_file, pointer = r["raw_shared_ref"].split("#", 1)
                        match = re.fullmatch(r"/results/(\d+)/raw", pointer)
                        if not match:
                            raise ValueError("Invalid unique raw owner reference")
                        owner_slot = int(match.group(1))
                    projection["_projection_owner_file"] = owner_file
                    projection["_projection_owner_slot"] = owner_slot
                    uid = r["prepared_unique_id"]
                    self.db.execute("INSERT INTO rows VALUES (?,?)", (uid, zlib.compress(encoded(projection), level=1)))
                    self.counts["unique_total"] += 1
                    self.counts["unique_inferred"] += bool(r["inference_evaluated"])
                    self.counts["unique_reused_previous"] += bool(r.get("reused_from_run"))
                    self.counts["unique_skipped_no_content"] += bool(r["normalization"]["skipped_no_content"])
                    self.counts["unique_error"] += r["status_processing"] == "error"
                    self.counts["unique_partial_invalid"] += r["status_processing"] == "partial_invalid"
                    if r["inference_evaluated"]:
                        self.inference_processing_seconds += r["processing_seconds"]
            if self.counts["unique_total"] != len(PREPARED):
                raise ValueError("Unique predictions do not cover the prepared corpus")
            self.db.commit()

        def get(self, uid):
            if uid not in self.cache:
                while len(self.cache) >= RECONSTRUCTION_CACHE_ROWS:
                    self.cache.popitem(last=False)
                row = self.db.execute("SELECT payload FROM rows WHERE uid=?", (uid,)).fetchone()
                if row is None:
                    raise ValueError("Missing prepared unique prediction")
                self.cache[uid] = json.loads(zlib.decompress(row[0]))
            self.cache.move_to_end(uid)
            return self.cache[uid]

        def close(self):
            self.cache.clear()
            self.db.close()


    def reconstruct_model_occurrences(name, timing):
        started = time.perf_counter()
        index_started = time.perf_counter()
        projection = OccurrenceProjection(name)
        timing["reconstruction_index_seconds"] += time.perf_counter() - index_started
        done = errors = last = 0
        records = []
        record_counts = Counter()
        try:
            for mapping in PREPARED.iter_occurrences():
                pos = mapping["sample_position"]
                start = (pos // CHECKPOINT_PARAGRAPHS) * CHECKPOINT_PARAGRAPHS
                target = shard_path(name, start)
                if target.exists():
                    # Immutable parts are validated by the standard source-linked reader.
                    if pos == start:
                        saved = load_shard(name, start)
                        for r in saved["results"]:
                            done += 1
                            errors += r["status_processing"] != "success"
                            record_counts["prediction_reused_occurrences"] += r["occurrence_reused_prediction"]
                            record_counts["duplicate_occurrences"] += r["is_repeated_text_occurrence"]
                    continue
                point = time.perf_counter()
                p = SAMPLE[pos]
                if any(p[k] != mapping[k] for k in ["Source", "Index", "start_document", "end_document", "corpus_position"]):
                    raise ValueError("Prepared occurrence map differs from current original source")
                owner = projection.get(mapping["unique_id"])
                result = clone_occurrence(owner, p, owner["_projection_owner_file"], owner["_projection_owner_slot"])
                result.pop("_projection_owner_file", None)
                result.pop("_projection_owner_slot", None)
                first = owner["paragraph"]["sample_position"]
                repeated = pos != first
                result.update(prepared_unique_id=mapping["unique_id"], is_repeated_text_occurrence=repeated,
                    occurrence_reused_prediction=bool(not owner["normalization"]["skipped_no_content"] and (repeated or owner.get("reused_from_run"))),
                    processing_seconds=time.perf_counter() - point, inference_evaluated=False)
                timing["reconstruction_seconds"] += time.perf_counter() - point
                records.append(result)
                done += 1
                errors += result["status_processing"] != "success"
                record_counts["prediction_reused_occurrences"] += result["occurrence_reused_prediction"]
                record_counts["duplicate_occurrences"] += repeated
                if len(records) == CHECKPOINT_PARAGRAPHS or pos == len(SAMPLE) - 1:
                    saving = time.perf_counter()
                    atomic_json(target, {"schema_version": SCHEMA_VERSION, "model": name, "prepared_key": PREPARED.key,
                        "sample_sha256": SAMPLE_HASH, "inference_sha256": INFERENCE_HASH,
                        "payload_sha256": digest(encoded(records)), "results": records})
                    timing["save_occurrences_seconds"] += time.perf_counter() - saving
                    records = []
                    checkpoint_safety_hook(reason=name + "_occurrences_committed")
                    last = progress(name + "_reconstruction", done, len(SAMPLE), errors, started, last)
            if done != len(SAMPLE):
                raise ValueError("Reconstruction coverage mismatch")
            return {**dict(projection.counts), **dict(record_counts), "occurrences": done,
                    "inference_processing_seconds_all_checkpoints": projection.inference_processing_seconds}
        finally:
            timing["reconstruction_phase_wall_seconds"] += time.perf_counter() - started
            projection.close()


    def run_model_with_batch_adapter(name):
        if PREPARED is None or PREFLIGHT_STATUS["status"] != "passed":
            raise RuntimeError("Run preparation and the equivalence/batch preflight first")
        timing = Counter({key:0.0 for key in ["model_load_seconds", "inference_seconds", "unique_cache_reuse_seconds",
            "skip_no_content_seconds", "save_unique_seconds", "unique_phase_wall_seconds", "reconstruction_index_seconds",
            "reconstruction_seconds", "save_occurrences_seconds", "reconstruction_phase_wall_seconds"]})
        started = time.perf_counter()
        folder = RUN_DIR / "models" / name
        folder.mkdir(parents=True, exist_ok=True)
        try:
            if not unique_model_complete(name):
                with load_ner_adapter(name) as adapter:
                    timing["model_load_seconds"] += adapter["metadata"]["load_seconds"]
                    run_prepared_uniques(name, adapter, timing)
            else:
                print(name, "unique coverage complete; model loading/inference skipped")
            counts = reconstruct_model_occurrences(name, timing)
            MODEL_OUTPUTS[name] = {"processing_seconds": timing["inference_seconds"], "prepared_counts": counts,
                                   "stage_timings_current_session": dict(timing)}
            print(name, "prepared unique/reuse counts:", json.dumps(counts, indent=2))
            print(name, "stage timings:", json.dumps(dict(timing), indent=2))
        finally:
            active = sys.exc_info()[1]
            timing["model_pipeline_wall_seconds"] = time.perf_counter() - started
            STAGE_TIMINGS["models"][name] = dict(timing)
            atomic_json(folder / f"stage_attempt_{SESSION_ID}_{secrets.token_hex(4)}.json", {
                "schema_version": "ner-prepared-stage-timings-1.0", "model": name, "session_id": SESSION_ID,
                "status_processing": "interrupted_or_failed" if active else "success", "timings": dict(timing),
                "error": f"{type(active).__name__}: {active}" if active else None, "at_processed": utc_now(),
                "scope": "Inference includes tokenization/windowing and batch prediction consumption; saving and reconstruction are measured separately. Phase wall times also include checkpoint uploads and IO."})
            cache = MODEL_CACHES.pop(name, None)
            if cache is not None:
                cache.close()
            RAW_REFERENCE_CACHE.clear()


    def build_preflight_paragraphs():
        eligible = []
        wanted = set(KNOWN_LONG_UNIT_CASES)
        chosen = {}
        for row in PREPARED.iter_unique():
            if row["eligible"] and len(row["Text"]) <= PREFLIGHT_MAX_PARAGRAPH_CHARS:
                eligible.append(row["unique_id"])
            p = SAMPLE[row["representative_sample_position"]]
            if (p["Source"], p["Index"]) in wanted and row["eligible"]:
                chosen[row["unique_id"]] = p
        positions = random.Random(EQUIVALENCE_SEED).sample(eligible, min(PREFLIGHT_PARAGRAPHS, len(eligible)))
        for uid in positions:
            chosen.setdefault(uid, SAMPLE[PREPARED[uid]["representative_sample_position"]])
        if not chosen:
            raise ValueError("No eligible prepared paragraphs for the fixed preflight")
        return [{**p, "sample_position": i} for i, p in enumerate(chosen.values())]


    def replay_preflight_results(name, paragraphs, plans, outputs, path, measure, limit):
        global ACTIVE_RAW_FILE, ACTIVE_RAW_INDEX, prepare_inference_text
        ACTIVE_RAW_FILE = path.relative_to(RUN_DIR).as_posix()
        results, legacy_results = [], []
        cursor = 0
        for i, p in enumerate(paragraphs):
            ACTIVE_RAW_INDEX = i
            inputs = [w["Text"] for w in plans[i]]
            local_outputs = outputs[cursor:cursor + len(inputs)]
            cursor += len(inputs)

            def replay_call(processor):
                queue = iter(zip(inputs, local_outputs))
                def cached(text):
                    expected, value = next(queue)
                    if expected != text:
                        raise ValueError("v11/prepared inference text or window plan differs")
                    if isinstance(value, BatchPredictionFailure):
                        raise RuntimeError(value.error)
                    return value
                result = processor(p, name, cached, measure, limit)
                if next(queue, None) is not None:
                    raise ValueError("v11/prepared window consumption differs")
                return result

            results.append(replay_call(process_uncached_occurrence))
            # Only the fixed equivalence audit invokes the v11 cleaner again.
            stored_reader = prepare_inference_text
            try:
                prepare_inference_text = LEGACY_PREPARE_TEXT
                legacy_results.append(replay_call(LEGACY_PROCESS_UNCACHED))
            finally:
                prepare_inference_text = stored_reader
        if cursor != len(outputs):
            raise ValueError("Preflight output count mismatch")
        delta = compare_model_predictions(name, paragraphs, legacy_results, results)
        for a, b in zip(legacy_results, results):
            if a["normalization"] != b["normalization"]:
                raise ValueError("Stored normalization metadata differs from v11")
            if a["raw"] != b["raw"]:
                # Runtime timestamps are intentionally different; compare model input/output instead.
                def raw_signature(items):
                    return [{k:v for k,v in r.items() if k != "at_processed"} for r in items]
                if raw_signature(a["raw"]) != raw_signature(b["raw"]):
                    raise ValueError("Prepared raw model output differs from v11 replay")
        atomic_json(path, {"schema_version": "ner-prepared-equivalence-1.0", "model": name,
            "source": SOURCE_META, "prepared_key": PREPARED.key, "equivalence_seed": EQUIVALENCE_SEED,
            "v11_equivalent": True, "max_score_delta": delta, "results": results,
            "sample_text_hashes": [digest(p["Text"].encode("utf-8")) for p in paragraphs]})
        return results

    """Exact tail-dot alignment plus immutable saved-raw replay views. No inference."""
    ALIGNMENT_POLICY = copy.deepcopy(CONFIG["alignment_validation"])
    ALIGNMENT_POLICY_HASH = digest(encoded(ALIGNMENT_POLICY))
    ALIGNMENT_REPORT = {}
    ADAPT_OUTPUT_BEFORE_ALIGNMENT = adapt_output
    PROCESS_UNCACHED_BEFORE_ALIGNMENT = process_uncached_occurrence
    LOAD_UNIQUE_BEFORE_ALIGNMENT = load_unique_shard
    LOAD_OWNER_BEFORE_ALIGNMENT = load_prediction_owner
    SHARD_PATH_BEFORE_ALIGNMENT = shard_path
    RUN_MODEL_BEFORE_ALIGNMENT = run_model_with_batch_adapter


    def alignment_folder():
        return RUN_DIR / "models" / "gliner" / ("alignment_" + ALIGNMENT_POLICY_HASH[:12])


    def adapt_output(name, output, text):
        predictions = ADAPT_OUTPUT_BEFORE_ALIGNMENT(name, output, text)
        if name != "gliner" or not ENABLE_GLINER_SAFE_DOT_REPAIR:
            return predictions
        repaired = []
        for original in predictions:
            p = dict(original)
            a, b, word = p.get("start"), p.get("end"), p.get("text")
            if (type(a) is int and type(b) is int and 0 <= a < len(text)
                    and b == len(text) + 1 and isinstance(word, str) and word.endswith(".")
                    and word[:-1] == text[a:]):
                p.update(end=len(text), text=text[a:], surface_checked=True,
                    alignment_repair={"schema_version": "georag-span-alignment-1.0",
                        "policy_sha256": ALIGNMENT_POLICY_HASH, "status": "corrected",
                        "reason": "one_extra_terminal_dot_exact_source_slice",
                        "start_original": a, "end_original": b, "text_original": word,
                        "start_corrected": a, "end_corrected": len(text)})
            repaired.append(p)
        return repaired


    def process_uncached_occurrence(p, name, predict, measure, limit, prepared=None):
        result = PROCESS_UNCACHED_BEFORE_ALIGNMENT(p, name, predict, measure, limit, prepared=prepared)
        if name == "gliner":
            result["alignment_policy_sha256"] = ALIGNMENT_POLICY_HASH
        return result


    def shard_path(name, start):
        if name == "gliner":
            return alignment_folder() / f"part_{start:08d}.json"
        return SHARD_PATH_BEFORE_ALIGNMENT(name, start)


    def raw_windows_for_replay(record):
        if record.get("raw"):
            return record["raw"]
        if record.get("raw_shared_ref"):
            value = resolve_raw_reference(record["raw_shared_ref"])
            if not isinstance(value, list):
                raise ValueError("Saved raw reference is not a window list")
            return value
        return []


    def replay_gliner_record(record, filename, slot):
        started = time.perf_counter()
        p = record["paragraph"]
        row = PREPARED[record["prepared_unique_id"]]
        if row["Text"] != p["Text"]:
            raise ValueError("Saved prediction differs from prepared original text")
        text = row["Text_inference"]
        windows_raw = raw_windows_for_replay(record)
        if row["eligible"] and not windows_raw:
            result = copy.deepcopy(record)
            result["alignment_validation"] = {"schema_version": "georag-span-alignment-1.0",
                "policy_sha256": ALIGNMENT_POLICY_HASH, "source_status": record["status_processing"],
                "status": "raw_unavailable_retained", "corrected_mentions": 0}
            result["alignment_policy_sha256"] = ALIGNMENT_POLICY_HASH
            return result
        found, errors, window_checks = [], [], []
        known_errors = Counter(str(m["error"]) for m in record.get("mentions", []) if m.get("error"))
        for window in windows_raw:
            if window.get("error"):
                known_errors[str(window["error"])] += 1
            output = window.get("output")
            if isinstance(output, dict):
                for error in output.get("errors", []):
                    known_errors[str(error)] += 1
        inherited_errors = []
        for error in record.get("errors", []):
            if known_errors[str(error)]:
                known_errors[str(error)] -= 1
            else:
                inherited_errors.append(str(error))
        raw_file, raw_slot = filename, slot
        if record.get("raw_shared_ref"):
            raw_file, pointer = record["raw_shared_ref"].split("#", 1)
            match = re.fullmatch(r"/results/(\d+)/raw", pointer)
            if not match:
                raise ValueError("Unexpected saved raw owner pointer")
            raw_slot = int(match.group(1))
        corrected = 0
        for wi, window in enumerate(windows_raw):
            before = len(found)
            check = {"window_id": window.get("window_id", wi), "status": "valid", "error": None}
            try:
                a, b, window_text = window["start"], window["end"], window["Text"]
                if not (type(a) is int and type(b) is int and 0 <= a < b <= len(text)
                        and text[a:b] == window_text):
                    raise ValueError("Saved raw window does not exactly match prepared inference text")
                output = window.get("output")
                if output is None:
                    raise ValueError(window.get("error") or "No saved model output for this window")
                output_errors = output.get("errors", []) if isinstance(output, dict) else []
                errors.extend(str(e) for e in output_errors)
                ref = f"{raw_file}#/results/{raw_slot}/raw/{wi}/output"
                for ordinal, item in enumerate(adapt_output("gliner", output, window_text)):
                    prediction = dict(item)
                    pa, pb = prediction.get("start"), prediction.get("end")
                    if type(pa) is int and type(pb) is int and not 0 <= pa < pb <= len(window_text):
                        synthetic = pb == len(window_text) + 1 and prediction.get("text", "").endswith(".")
                        prediction["alignment_error"] = ("GLiNER synthetic trailing punctuation outside source window"
                            if synthetic else "Span outside source window")
                    m = mention({**p, "Text": text}, "gliner", prediction, a, ref)
                    m["raw_ordinal"] = ordinal
                    found.append(m)
                    corrected += prediction.get("alignment_repair", {}).get("status") == "corrected"
                if output_errors or any(m["status_validation"] != "valid" for m in found[before:]):
                    check["status"] = "invalid"
            except Exception as exc:
                check.update(status="invalid", error=f"{type(exc).__name__}: {exc}")
                errors.append(check["error"])
            window_checks.append(check)
        count_before = len(found)
        found = deduplicate(found)
        for m in found:
            m["word_inference"] = m["word"]
            m["Text"] = p["Text"]
            m["text_sha256"] = row["text_sha256"]
            m["inference_text_sha256"] = json.loads(row["normalization_json"])["inference_sha256"]
            a, b = m["start"], m["end"]
            if type(a) is int and type(b) is int and 0 <= a < b <= len(p["Text"]):
                m["word"] = p["Text"][a:b]
                m["start_document"], m["end_document"] = p["start_document"] + a, p["start_document"] + b
                m["source_span_contains_masked_characters"] = p["Text"][a:b] != text[a:b]
                if m["status_validation"] == "valid":
                    assert_document_span(p, m["start_document"], m["end_document"], m["word"])
        invalid = [m for m in found if m["status_validation"] != "valid"]
        valid_corrected = sum(m["status_validation"] == "valid" and
            m.get("prediction",{}).get("alignment_repair",{}).get("status") == "corrected" for m in found)
        errors.extend(inherited_errors)
        result = {k:copy.deepcopy(v) for k,v in record.items() if k not in {"raw", "mentions", "errors"}}
        result.update(raw=[], raw_shared_ref=f"{raw_file}#/results/{raw_slot}/raw", mentions=found,
            errors=errors + [m["error"] for m in invalid], duplicates_resolved=count_before-len(found),
            status_processing="error" if errors else "partial_invalid" if invalid else "success",
            alignment_policy_sha256=ALIGNMENT_POLICY_HASH,
            alignment_validation={"schema_version": "georag-span-alignment-1.0",
                "policy_sha256": ALIGNMENT_POLICY_HASH, "source_status": record["status_processing"],
                "status": "replayed_from_saved_raw", "corrected_mentions": valid_corrected,
                "span_repairs_attempted": corrected,
                "remaining_invalid_mentions": len(invalid), "window_validation": window_checks,
                "replay_seconds": time.perf_counter()-started, "at_revalidated": utc_now()})
        return result


    def aligned_unique_view(name, start, original):
        if name != "gliner" or all(r.get("alignment_policy_sha256") == ALIGNMENT_POLICY_HASH for r in original["results"]):
            return original
        folder = alignment_folder()
        folder.mkdir(parents=True, exist_ok=True)
        path = folder / f"unique_part_{start:08d}.json"
        if not path.exists():
            filename = unique_shard_path(name, start).relative_to(RUN_DIR).as_posix()
            records = [replay_gliner_record(r, filename, slot) for slot,r in enumerate(original["results"])]
            atomic_json(path, {"schema_version": "georag-alignment-view-1.0",
                "alignment_policy_sha256": ALIGNMENT_POLICY_HASH, "source_payload_sha256": original["payload_sha256"],
                "source_file": filename, "model": name, "prepared_key": PREPARED.key,
                "sample_sha256": SAMPLE_HASH, "inference_sha256": INFERENCE_HASH,
                "payload_sha256": digest(encoded(records)), "results": records})
        view = load_json(path)
        if (view.get("schema_version") != "georag-alignment-view-1.0"
                or view.get("alignment_policy_sha256") != ALIGNMENT_POLICY_HASH
                or view.get("source_payload_sha256") != original["payload_sha256"]
                or view.get("prepared_key") != PREPARED.key or view.get("sample_sha256") != SAMPLE_HASH
                or view.get("inference_sha256") != INFERENCE_HASH
                or view["payload_sha256"] != digest(encoded(view["results"]))
                or [(r["prepared_unique_id"],r["paragraph"]) for r in view["results"]] !=
                   [(r["prepared_unique_id"],r["paragraph"]) for r in original["results"]]):
            raise ValueError("Alignment sidecar source/policy/payload identity mismatch")
        return view


    def load_unique_shard(name, start):
        return aligned_unique_view(name, start, LOAD_UNIQUE_BEFORE_ALIGNMENT(name, start))


    def load_prediction_owner(filename):
        match = re.fullmatch(r"models/gliner/unique_part_(\d+)\.json", filename)
        if match:
            start = int(match.group(1))
            original = LOAD_UNIQUE_BEFORE_ALIGNMENT("gliner", start)
            view = aligned_unique_view("gliner", start, original)
            records = []
            for actual, normalized in zip(original["results"], view["results"]):
                records.append({**normalized, "raw": actual["raw"]})
            return {**original, "results": records, "payload_sha256": digest(encoded(records))}
        if re.fullmatch(r"models/gliner/part_\d+\.json", filename):
            original = load_json(RUN_DIR / filename)
            if original["payload_sha256"] != digest(encoded(original["results"])):
                raise ValueError("Original occurrence raw owner payload hash mismatch")
            return original
        return LOAD_OWNER_BEFORE_ALIGNMENT(filename)


    def replay_saved_gliner_alignment():
        global ALIGNMENT_REPORT
        started = time.perf_counter()
        starts = [start for start in range(0,len(PREPARED),CHECKPOINT_PARAGRAPHS)
                  if unique_shard_path("gliner",start).is_file()]
        counts = Counter()
        last = 0
        for number,start in enumerate(starts,1):
            chunk = load_unique_shard("gliner",start)
            for record in chunk["results"]:
                metadata = record.get("alignment_validation",{})
                counts["saved_unique_predictions"] += 1
                counts["corrected_mentions"] += metadata.get("corrected_mentions",0)
                counts["source_"+metadata.get("source_status",record["status_processing"])] += 1
                counts["current_"+record["status_processing"]] += 1
                counts["raw_unavailable_retained"] += metadata.get("status")=="raw_unavailable_retained"
            last = progress("gliner_saved_raw_replay",number,len(starts),0,started,last)
            checkpoint_safety_hook(reason="gliner_alignment_replay_committed")
        ALIGNMENT_REPORT = {"schema_version":"georag-alignment-replay-report-1.0",
            "policy":ALIGNMENT_POLICY,"policy_sha256":ALIGNMENT_POLICY_HASH,"counts":dict(counts),
            "seconds":time.perf_counter()-started,"model_inference_calls":0,
            "scope":"Saved raw normalization only; missing unique checkpoints still require model inference"}
        folder=alignment_folder();folder.mkdir(parents=True,exist_ok=True)
        atomic_json(folder/f"replay_report_{SESSION_ID}_{secrets.token_hex(4)}.json",ALIGNMENT_REPORT)
        STAGE_TIMINGS["gliner_saved_raw_alignment_seconds"] = ALIGNMENT_REPORT["seconds"]
        print("Saved GLiNER raw replay:",json.dumps(ALIGNMENT_REPORT,indent=2))
        checkpoint_safety_hook(force=True,reason="gliner_alignment_replay_ready")


    def alignment_selftest():
        def output(word,start,end):
            return {"entities":{"person":[{"text":word,"start":start,"end":end,"confidence":0.8}]}}
        raw=output("Paris.",0,6)
        saved=copy.deepcopy(raw)
        fixed=adapt_output("gliner",raw,"Paris")[0]
        assert raw==saved
        if ENABLE_GLINER_SAFE_DOT_REPAIR:
            assert (fixed["text"],fixed["start"],fixed["end"],fixed["score"])==("Paris",0,5,0.8)
            assert adapt_output("gliner",output("London.",0,6),"Paris")[0]["end"]==6
            assert adapt_output("gliner",output("Paris.",0,7),"Paris")[0]["end"]==7
            assert adapt_output("gliner",output("Mr.",0,3),"Mr.")[0]["text"]=="Mr."
            assert adapt_output("gliner",output("Paris.",2,8),"😀 Paris")[0]["end"]==7
        print("Exact-dot alignment selftests passed; unsafe cases remain unchanged")


    def run_model_with_batch_adapter(name):
        if name == "gliner":
            replay_saved_gliner_alignment()
        return RUN_MODEL_BEFORE_ALIGNMENT(name)


### 5b — הכנת קלט ייחודי פעם אחת

פלט: מפתח מטמון, ספירות ייחודיים/הופעות, זמני הכנה ואישור גיבוי. לא נטענים מודלים בתא זה.


In [ ]:
with cell("5b — prepare unique input once for both models"):
    prepare_input_once()
    initialize_delta_reuse()
    checkpoint_safety_hook(force=True, reason="preparation_and_delta_ready")


### 6 — בחירה, מועמדים ומחלוקות

פלט: אישור כללי הבחירה והעברת מחלוקות באותו טווח לתוצר נפרד.


In [ ]:
with cell("6 — selection and table utilities"):
    COLUMNS = ["entity_group", "score", "word", "start", "end", "Source", "Index", "Text", "subtype"]
    ARROW_SCHEMA = pa.schema([(k, pa.int64() if k in {"start", "end", "Index"}
        else pa.float64() if k == "score" else pa.string()) for k in COLUMNS])
    def table(rows):
        return pa.Table.from_pylist([{k: r.get(k) for k in COLUMNS} for r in rows], schema=ARROW_SCHEMA)
    def write_parquet(rows, path):
        arrow = table(rows)
        with Path(path).open("xb") as handle:
            pq.write_table(arrow, handle)
        with Path(path).open("rb") as handle:
            restored = pq.read_table(handle)
        assert restored.schema == ARROW_SCHEMA
        assert restored.to_pylist() == arrow.to_pylist()

    def span_key(m):
        return (m["Source"], m["Index"], m["entity_group"], m["start"], m["end"])
    def overlaps(a, b):
        return (a["Source"], a["Index"]) == (b["Source"], b["Index"]) and max(a["start"], b["start"]) < min(a["end"], b["end"])
    def select_mentions(mentions):
        valid = [m for m in mentions if m["status_validation"] == "valid"]
        support = defaultdict(list)
        for m in valid:
            support[span_key(m)].append(m)
        primary = [m for m in valid if m["entity_group"] in SELECTION_THRESHOLDS
            and PRIMARY[m["entity_group"]] == m["model"] and m["score"] is not None
            and m["score"] >= SELECTION_THRESHOLDS[m["entity_group"]]]
        gliner_per = [m for m in primary if m["entity_group"] == "PER"]
        additions = [m for m in valid if ADD_BERT_PER and m["model"] == "bert"
            and m["entity_group"] == "PER" and m["score"] is not None
            and m["score"] >= BERT_PER_MIN_SCORE and not any(overlaps(m, g) for g in gliner_per)]
        accepted = list(primary)
        filtered_additions = []
        for m in sorted(additions, key=lambda x: (-x["score"], x["start"], x["end"])):
            if not any(overlaps(m, x) for x in accepted if x["entity_group"] == "PER"):
                filtered_additions.append(m)
                accepted.append(m)
        additions = filtered_additions
        selected, seen = [], set()
        for m in primary + additions:
            key = span_key(m)
            if key in seen:
                continue
            seen.add(key)
            candidate = m["entity_group"] in {"TIME", "EVENT"}
            high = m["entity_group"] == "EVENT" and m["score"] is not None and m["score"] >= EVENT_HIGH_PRIORITY_SCORE
            selected.append({**m, "selected_model": m["model"], "selected_model_id": m["model_id"],
                "selection_score": m["score"], "selection_policy": "v2",
                "selection_reason": "bert_per_supplement" if m in additions else "primary_v2",
                "selection_status": "candidate_high_priority" if high else "candidate" if candidate else "selected_mention",
                "semantic_validation": "not_assessed", "verified_event_fact": False,
                "supporting_models": sorted({x["model"] for x in support[key]}),
                "support": [{k: x[k] for k in ["model", "model_id", "label_original", "subtype", "score", "ref_output_raw"]}
                            for x in support[key]]})
        return sorted(selected, key=lambda m: (m["sample_position"], m["start"], m["end"], m["entity_group"]))
    print("v2 ready: MISC=BERT; TIME/EVENT candidates; score is the chosen model's score.")

def select_candidates(mentions):
    answer, seen = [], set()
    for m in mentions:
        if m["model"] != "gliner" or m["entity_group"] not in {"TIME", "EVENT"} or m["status_validation"] != "valid":
            continue
        if span_key(m) in seen:
            continue
        seen.add(span_key(m))
        high = m["entity_group"] == "EVENT" and m["score"] is not None and m["score"] >= EVENT_HIGH_PRIORITY_SCORE
        answer.append({**m, "selection_status": "candidate_high_priority" if high else "candidate",
            "selected_model": "gliner", "semantic_validation": "not_assessed", "verified_event_fact": False})
    return answer

with cell("6b — exact-span conflict policy"):
    SELECT_MENTIONS_V9 = select_mentions
    """Apply existing per-label selection, then quarantine exact-span type conflicts."""


    def arbitrate_mentions(mentions):
        chosen = SELECT_MENTIONS_V9(mentions)
        chosen_signatures = {(span_key(m), m["model"]) for m in chosen}
        chosen_by_span = defaultdict(list)
        for m in chosen:
            chosen_by_span[m["Source"], m["Index"], m["start"], m["end"]].append(m)
        by_span = defaultdict(list)
        # Always include every chosen prediction. Low-confidence unchosen alternatives
        # cannot quarantine it; their role is diagnostic only above the configured floor.
        for m in mentions:
            if (m["status_validation"] == "valid" and m["entity_group"] not in {"TIME", "EVENT"}
                    and m["score"] is not None and (m["score"] >= CONFLICT_DIAGNOSTIC_MIN_SCORE
                    or (span_key(m), m["model"]) in chosen_signatures)):
                by_span[m["Source"], m["Index"], m["start"], m["end"]].append(m)
        selected, conflicts = [], []
        quarantined = set()
        for span, alternatives in by_span.items():
            labels = sorted({m["entity_group"] for m in alternatives})
            if len(labels) <= 1:
                continue
            local = chosen_by_span[span]
            unresolved = len({m["entity_group"] for m in local}) > 1
            if unresolved:
                quarantined.add(span)
            status = "unresolved" if unresolved else "resolved_by_primary" if local else "no_selected_candidate"
            conflict_key = digest(encoded({"source": SOURCE_META["sha256"], "span": span, "labels": labels}))
            seen = set()
            for m in alternatives:
                key = m["model"], m["label_original"], m["start"], m["end"]
                if key in seen:
                    continue
                seen.add(key)
                accepted = any(c["model"] == m["model"] and c["entity_group"] == m["entity_group"] for c in local)
                conflicts.append({**m, "conflict_key": conflict_key, "conflict_status": status,
                    "selection_status": "quarantined_conflict" if unresolved else "selected_mention" if accepted else "alternative_not_selected",
                    "is_selected": accepted and not unresolved, "selected_model": m["model"],
                    "competing_labels": json.dumps(labels),
                    "competing_models": json.dumps(sorted({a["model"] for a in alternatives})),
                    "conflict_reason": "Different selected labels at the exact same source span" if unresolved else
                        "Existing per-label model/supplement/threshold policy decides; raw alternative retained",
                    "semantic_validation": "not_assessed"})
        for m in chosen:
            span = m["Source"], m["Index"], m["start"], m["end"]
            if CONFLICT_POLICY == "quarantine_exact_span" and span in quarantined:
                continue
            selected.append({**m, "conflict_status": "unresolved_retained" if span in quarantined else
                             "resolved_by_primary" if len({a["entity_group"] for a in by_span[span]}) > 1 else "none"})
        return selected, conflicts


    def select_mentions(mentions):
        return arbitrate_mentions(mentions)[0]


    @contextmanager
    def contextmanager_export_writers():
        with ExitStack() as stack:
            writers = {}
            for kind, cols, schema in [("selected", COLUMNS, ARROW_SCHEMA),
                    ("candidates", CANDIDATE_COLUMNS, CANDIDATE_SCHEMA),
                    ("conflicts", CONFLICT_COLUMNS, CONFLICT_SCHEMA)]:
                handle = stack.enter_context(OUTPUT_FILES[kind + "_parquet"].open("xb"))
                writer = stack.enter_context(pq.ParquetWriter(handle, schema, compression="zstd"))
                writers[kind] = cols, schema, writer
            yield writers


    def verify_parquet_roundtrip(kind):
        cols = COLUMNS if kind == "selected" else CANDIDATE_COLUMNS if kind == "candidates" else CONFLICT_COLUMNS
        def expected_rows():
            index = load_json(OUTPUT_FILES[kind + "_json"])
            for part in index["parts"]:
                yield from load_json(EXPORT_DIR / part["path"])["records"]
        expected = iter(expected_rows())
        count = 0
        with pq.ParquetFile(OUTPUT_FILES[kind + "_parquet"]) as parquet:
            for batch in parquet.iter_batches(batch_size=1024):
                for row in batch.to_pylist():
                    detail = next(expected)
                    assert row == {key: detail.get(key) for key in cols}
                    count += 1
            assert next(expected, None) is None and parquet.metadata.num_rows == count
        return count


### 7 — בדיקות תקינות

בודק היסטים, חזרות, חלונות, ספים, מועמדים וקריאה חוזרת של Parquet.


In [ ]:
with cell("7 — Parquet selftests"):
    original_meta = SOURCE_META
    saved_thresholds, saved_add = SELECTION_THRESHOLDS, ADD_BERT_PER
    SELECTION_THRESHOLDS = {"PER":.5,"DATE":.5,"LOC":0.,"ORG":0.,"MISC":0.}
    ADD_BERT_PER = True
    try:
        t = "Paris Paris John Smith battle 1941"
        SOURCE_META = {"sha256":"test", "commit_sha":"test", "path":"test.parquet"}
        p = {"Source":"test", "Index":0, "Text":t, "start_document":7,
            "end_document":7+len(t), "sample_position":0}
        def m(name,label,a,b,score):
            return mention(p,name,{"label":label,"start":a,"end":b,"text":t[a:b],"score":score},0,"test")
        items=[m("bert","LOC",0,5,.8),m("bert","LOC",6,11,.8),
            m("gliner","person",12,22,.7),m("bert","PER",12,16,.99),
            m("gliner","person",0,5,.1),m("gliner","date",30,34,.1),
            m("gliner","event",23,29,.95)]
        chosen=select_mentions(items)
        assert len([x for x in chosen if x["entity_group"]=="LOC"])==2
        assert all(x["entity_group"] not in {"TIME","EVENT"} for x in chosen)
        assert not any(x["entity_group"]=="DATE" for x in chosen)
        assert not any(x["word"]=="John" for x in chosen)
        candidates=select_candidates(items)
        assert len(candidates)==1 and not candidates[0]["verified_event_fact"]
        bad=mention(p,"gliner",{"label":"person","text":t+".","start":0,"end":len(t)+1,"score":.99},0,"test")
        assert bad["status_validation"]=="invalid"
        text="a b c New York d e f g"
        ws=list(windows(text,{"max_words":4,"overlap_words":2,"max_encoded_tokens":999},len,999))
        assert any("New York" in w["Text"] for w in ws)
        with tempfile.TemporaryDirectory(dir=RUN_DIR) as tmp:
            write_parquet(chosen,Path(tmp)/"test.parquet")
            write_parquet([],Path(tmp)/"empty.parquet")
        checks={"threshold_separation":True,"repeated_mentions":True,"per_overlap":True,
            "event_separate":True,"synthetic_punctuation_invalid":True,"boundary_window":True,
            "parquet_roundtrip":True}
    finally:
        SOURCE_META=original_meta
        SELECTION_THRESHOLDS, ADD_BERT_PER = saved_thresholds, saved_add
    save_json(PATHS["selftests"],checks)
    print(checks)

with cell("7a — strict GLiNER alignment selftests; no model inference"):
    alignment_selftest()


### 7b — השוואת הסקה יחידנית ובאצוות

טוען כל מודל בנפרד ומשווה את הבאצוות לחיזוי יחידני. עוצר אם הבחירה משתנה בבדיקה.


In [ ]:
def selection_probe_hash():
    return digest(encoded({"thresholds":SELECTION_THRESHOLDS,"add_bert_per":ADD_BERT_PER,
        "bert_per_min":BERT_PER_MIN_SCORE,"event_high":EVENT_HIGH_PRIORITY_SCORE,
        "primary":PRIMARY,"conflicts":CONFLICT_POLICY,"diagnostic_floor":CONFLICT_DIAGNOSTIC_MIN_SCORE}))

saved_probes = [load_json(p) for p in sorted((RUN_DIR/"models").glob("batch_preflight_*.json"))]
reusable_probe = next((r for r in reversed(saved_probes) if r.get("status")=="passed"
    and r.get("v11_equivalent_on_fixed_probe") and r.get("prepared_key")==PREPARED.key
    and r.get("inference_sha256")==INFERENCE_HASH and r.get("equivalence_seed")==EQUIVALENCE_SEED
    and r.get("selection_policy_sha256")==selection_probe_hash()), None)
if reusable_probe is not None:
    with cell("7b — reuse verified equivalence/batch preflight after resume"):
        PREFLIGHT_STATUS = reusable_probe
        PREFLIGHT_EFFECTIVE_SIZES = {n:reusable_probe["models"][n]["effective_batch_size"] for n in MODELS}
        print("Saved fixed-sample equivalence and batch checks match this exact run; probe model loading skipped")
else:
    try:
        with cell("7b — real-model window and batch comparison preflight"):
            paragraphs=build_preflight_paragraphs()
            reference_results={};batched_results={};model_reports={}
            PREFLIGHT_STATUS={'status':'running'}
            PREFLIGHT_EFFECTIVE_SIZES={}
            report_path=RUN_DIR/'models'/f'batch_preflight_{SESSION_ID}.json'
            report_path.parent.mkdir(parents=True,exist_ok=True)
            try:
                for name in MODELS:
                    folder=RUN_DIR/'models'/name
                    folder.mkdir(parents=True,exist_ok=True)
                    with load_ner_adapter(name) as adapter:
                        length_started=time.perf_counter()
                        validate_preflight_windows(
                            'URL https://example.org/?q='+'%7B%26quot%3B'*120+' tail',
                            WINDOWS[name],adapter['measure'],adapter['limit'])
                        plans=[validate_preflight_windows(prepare_inference_text(p['Text'])[0],WINDOWS[name],adapter['measure'],adapter['limit'])
                               for p in paragraphs]
                        tasks=[{'paragraph_position':i,'window_id':wi,'text':w['Text']}
                               for i,plan in enumerate(plans) for wi,w in enumerate(plan)]
                        length_seconds=time.perf_counter()-length_started
                        print(name,'length guard passed:',len(paragraphs),'paragraphs;',len(tasks),'windows')
                        if torch.cuda.is_available():torch.cuda.synchronize()
                        started=time.perf_counter()
                        with torch.inference_mode():
                            ref_outputs=[adapter['single'](task['text']) for task in tasks]
                        if torch.cuda.is_available():torch.cuda.synchronize()
                        sequential_seconds=time.perf_counter()-started
                        requested=BATCH_SIZES[name] if DEVICE.startswith('cuda') and BATCH_ENABLED else 1
                        runner=AdaptiveBatchRunner(name,adapter['single'],adapter['batch'],requested,batch_event_sink(name))
                        started=time.perf_counter()
                        if BATCH_ENABLED:
                            batch_outputs=runner.predict(tasks)
                        else:
                            batch_outputs=ref_outputs
                        if torch.cuda.is_available():torch.cuda.synchronize()
                        batch_seconds=time.perf_counter()-started if BATCH_ENABLED else None
                        reference_results[name]=replay_preflight_results(name,paragraphs,plans,ref_outputs,
                            folder/f'preflight_single_{SESSION_ID}.json',adapter['measure'],adapter['limit'])
                        batched_results[name]=replay_preflight_results(name,paragraphs,plans,batch_outputs,
                            folder/f'preflight_batch_{SESSION_ID}.json',adapter['measure'],adapter['limit'])
                        max_delta=compare_model_predictions(name,paragraphs,reference_results[name],batched_results[name])
                        PREFLIGHT_EFFECTIVE_SIZES[name]=runner.size
                        model_reports[name]={'paragraphs':len(paragraphs),'windows':len(tasks),'synthetic_long_unit_guard_passed':True,
                            'window_guard_seconds':length_seconds,'model_load_seconds':adapter['metadata']['load_seconds'],
                            'sequential_prediction_seconds':sequential_seconds,'batch_prediction_seconds':batch_seconds,
                            'sample_speedup':sequential_seconds/batch_seconds if batch_seconds else None,
                            'requested_batch_size':requested,'effective_batch_size':runner.size,
                            'max_score_delta':max_delta,'batch_statistics':dict(runner.stats)}
                        print(name,'preflight comparison passed:',json.dumps(model_reports[name]))
                        del ref_outputs,batch_outputs,runner
                # Compare the actual combined selection, including BERT PER ranking and overlaps.
                for i,p in enumerate(paragraphs):
                    a=[m for n in MODELS for m in reference_results[n][i]['mentions']]
                    b=[m for n in MODELS for m in batched_results[n][i]['mentions']]
                    for selector in [select_mentions,select_candidates]:
                        x,y=selector(a),selector(b)
                        assert comparison_signature(x)==comparison_signature(y),'Combined selected/candidate mentions changed'
                        for r,s in zip(x,y):
                            if r['score'] is None or s['score'] is None:assert r['score'] is s['score']
                            else:assert abs(r['score']-s['score'])<=BATCH_SCORE_ATOL
                PREFLIGHT_STATUS={'schema_version':'ner-batch-gate-1.0','status':'passed',
                    'at_processed':utc_now(),'source_sha256':SOURCE_META['sha256'],
                    'mode':'batched' if BATCH_ENABLED else 'sequential_window_guard',
                    'score_atol':BATCH_SCORE_ATOL,'models':model_reports,
                    'combined_selection_identical_on_probe':True,'v11_equivalent_on_fixed_probe':True,
                    'equivalence_seed':EQUIVALENCE_SEED,'prepared_key':PREPARED.key,
                    'inference_sha256':INFERENCE_HASH,'selection_policy_sha256':selection_probe_hash(),
                    'scope':'This probe is not a guarantee of identical predictions over the full corpus.'}
            except Exception as exc:
                PREFLIGHT_STATUS={'schema_version':'ner-batch-gate-1.0','status':'failed',
                    'at_processed':utc_now(),'error':f'{type(exc).__name__}: {exc}',
                    'models':model_reports,'source_sha256':SOURCE_META['sha256']}
                atomic_json(report_path,PREFLIGHT_STATUS)
                print('Preflight failed; full inference has not started. Report:',report_path)
                raise
            else:
                atomic_json(report_path,PREFLIGHT_STATUS)
                print('Both models and the combined selection passed. Preflight report:',report_path)
            finally:
                # Probe raw outputs are persisted; keep only summaries in runtime memory.
                reference_results.clear();batched_results.clear()
                release()
    finally:
        checkpoint_safety_hook(force=True,reason="preflight_finished_or_interrupted")

# Model batching/window evidence is separate from the new normalization policy.
PREFLIGHT_STATUS = copy.deepcopy(PREFLIGHT_STATUS)
PREFLIGHT_STATUS["alignment_policy_sha256"] = ALIGNMENT_POLICY_HASH
PREFLIGHT_STATUS["alignment_scope"] = "Strict CPU alignment tests plus saved-raw replay; legacy saved selection equality is historical evidence"
if reusable_probe is not None:
    PREFLIGHT_STATUS["historical_combined_selection_identical_on_probe"] = PREFLIGHT_STATUS.pop("combined_selection_identical_on_probe", None)
    PREFLIGHT_STATUS["historical_v11_equivalent_on_fixed_probe"] = PREFLIGHT_STATUS.pop("v11_equivalent_on_fixed_probe", None)
else:
    PREFLIGHT_STATUS["reference_comparison_scope"] = "Both comparison paths use the current explicit alignment adapter"


### 8 — BERT: הסקה ייחודית ושחזור הופעות

פלט: התקדמות ייחודיים, התקדמות שחזור, מספר טקסטים שהוסקו/מוחזרו וזמנים נפרדים.


In [ ]:
with cell("8 — bert"):
    try:
        checkpoint_safety_hook(reason="before_bert")
        run_model_with_batch_adapter("bert")
    finally:
        checkpoint_safety_hook(force=True,reason="bert_finished_or_interrupted")


### 9 — GLiNER: הסקה ייחודית ושחזור הופעות

אותו קלט מוכן, עם tokenizer וחלונות של GLiNER.


In [ ]:
with cell("9 — gliner"):
    try:
        checkpoint_safety_hook(reason="before_gliner")
        run_model_with_batch_adapter("gliner")
    finally:
        checkpoint_safety_hook(force=True,reason="gliner_finished_or_interrupted")


### 10 — ייצוא אזכורים, מועמדים ומחלוקות

פלט: מספר שורות בכל תוצר ומספר קבוצות המחלוקת שלא הוכרעו.


In [ ]:
with cell("10 — streaming selection, conflicts and export"):
    checkpoint_safety_hook(reason="before_export")
    assert all(model_complete(n) for n in MODELS), "Incomplete inference coverage"
    CANDIDATE_COLUMNS = COLUMNS + ["selection_status", "selected_model", "semantic_validation", "verified_event_fact"]
    CANDIDATE_SCHEMA = pa.schema(list(ARROW_SCHEMA) + [pa.field("selection_status", pa.string()),
        pa.field("selected_model", pa.string()), pa.field("semantic_validation", pa.string()),
        pa.field("verified_event_fact", pa.bool_())])
    CONFLICT_COLUMNS = COLUMNS + ["conflict_key", "conflict_status", "selection_status", "selected_model",
        "is_selected", "label_original", "competing_labels", "competing_models", "conflict_reason",
        "sample_position", "corpus_position", "start_document", "end_document", "status_validation"]
    CONFLICT_SCHEMA = pa.schema(list(ARROW_SCHEMA) + [pa.field(k,
        pa.bool_() if k == "is_selected" else pa.int64() if k in {"sample_position", "corpus_position", "start_document", "end_document"}
        else pa.string()) for k in CONFLICT_COLUMNS[len(COLUMNS):]])
    kinds = ["selected", "candidates", "conflicts"]
    OUTPUT_FILES = {}
    for kind in kinds:
        basename = RUN_NAME if kind == "selected" else f"{RUN_NAME}_{kind}"
        for ext in ["parquet", "json"]:
            filename = GITHUB_FINAL_FILENAME if kind == "selected" and ext == "parquet" else f"{basename}.{ext}"
            OUTPUT_FILES[kind + "_" + ext] = EXPORT_DIR / filename
        (EXPORT_DIR / (kind + "_parts")).mkdir()
    indexes = {kind: [] for kind in kinds}
    EXPORT_COUNTS = Counter()
    EXPORT_LABELS = {kind: Counter() for kind in kinds}
    started, last = time.perf_counter(), 0
    with contextmanager_export_writers() as writers:
        for start in range(0, len(SAMPLE), CHECKPOINT_PARAGRAPHS):
            chunks = {n: load_shard(n, start) for n in MODELS}
            rows = {kind: [] for kind in kinds}
            for local, p in enumerate(SAMPLE[start:start + CHECKPOINT_PARAGRAPHS]):
                mentions = [m for n in MODELS for m in chunks[n]["results"][local]["mentions"]]
                selected, conflicts = arbitrate_mentions(mentions)
                candidates = select_candidates(mentions)
                for m in selected + candidates + conflicts:
                    assert m["Text"] == p["Text"] and m["Text"][m["start"]:m["end"]] == m["word"]
                    assert_document_span(p, m["start_document"], m["end_document"], m["word"])
                rows["selected"].extend(selected)
                rows["candidates"].extend(candidates)
                rows["conflicts"].extend(conflicts)
                EXPORT_COUNTS["bert_per_additions"] += sum(m["selection_reason"] == "bert_per_supplement" for m in selected)
                EXPORT_COUNTS["unresolved_conflict_groups"] += len({m["conflict_key"] for m in conflicts if m["conflict_status"] == "unresolved"})
            for kind, values in rows.items():
                cols, schema, writer = writers[kind]
                writer.write_table(pa.Table.from_pylist([{k: m.get(k) for k in cols} for m in values], schema=schema))
                part = EXPORT_DIR / (kind + "_parts") / f"part_{start:08d}.json"
                atomic_json(part, {"schema_version": SCHEMA_VERSION, "sample_sha256": SAMPLE_HASH,
                    "kind": kind, "records": values})
                indexes[kind].append({"path": part.relative_to(EXPORT_DIR).as_posix(),
                    "record_count": len(values), "sha256": digest_file(part)})
                EXPORT_COUNTS[kind] += len(values)
                EXPORT_LABELS[kind].update(m["entity_group"] for m in values)
            last = progress("selection/export", min(start + CHECKPOINT_PARAGRAPHS, len(SAMPLE)), len(SAMPLE), 0, started, last)
    for kind in kinds:
        atomic_json(OUTPUT_FILES[kind + "_json"], {"schema_version": SCHEMA_VERSION,
            "run_name": RUN_NAME, "sample_sha256": SAMPLE_HASH, "source": SOURCE_META,
            "configuration_sha256": CONFIG_HASH, "selection_configuration": CONFIG,
            "record_count": EXPORT_COUNTS[kind], "parts": indexes[kind], "semantic_validation": "not_assessed"})
    print("Export counts:", dict(EXPORT_COUNTS))


### 11 — ביקורת כיסוי ותוצרים

מאמת כיסוי, הפניות לפלט גולמי, כל האזכורים שנבחרו וקריאה חוזרת של Parquet.


In [ ]:
"""Bounded, persistent catalog for audit-only raw reference existence checks.

Original payloads and resolver are unchanged. Each immutable raw owner is loaded
and validated once; only non-null pointer names are cached, never model payloads.
"""
from collections import OrderedDict, Counter
from pathlib import Path, PurePosixPath
import json
import sqlite3


class RawReferenceAudit:
    def __init__(self, ns, max_owners=128, max_pointers=50000):
        self.ns = ns
        self.root = Path(ns['RUN_DIR']).resolve()
        folder = self.root/'runtime_cache'
        folder.mkdir(exist_ok=True)
        self.path = folder/f"raw_reference_audit_{ns['SESSION_ID']}.sqlite"
        self.db = sqlite3.connect(self.path)
        self.db.execute('PRAGMA journal_mode=WAL')
        self.db.execute('PRAGMA synchronous=NORMAL')
        self.db.execute('PRAGMA cache_size=-16384')
        self.db.execute('CREATE TABLE IF NOT EXISTS owners (name TEXT PRIMARY KEY, signature TEXT NOT NULL)')
        self.db.execute('CREATE TABLE IF NOT EXISTS refs (name TEXT, pointer TEXT, PRIMARY KEY(name,pointer)) WITHOUT ROWID')
        self.cache = OrderedDict()
        self.max_owners, self.max_pointers = max_owners, max_pointers
        self.cached_pointers = 0
        self.signatures = {}
        self.stats = Counter()

    def __enter__(self): return self

    def __exit__(self, *args):
        self.db.close()
        self.cache.clear()

    def begin_chunk(self):
        # Check file identities again for each occurrence shard, not per mention.
        self.signatures.clear()

    def safe_path(self, name):
        if not isinstance(name,str) or '\\' in name or ':' in name:
            raise ValueError('Unsafe raw owner path')
        p = PurePosixPath(name)
        path = (self.root/name).resolve()
        if (p.is_absolute() or '..' in p.parts or not p.parts or p.parts[0]!='models'
                or not path.is_relative_to(self.root)):
            raise ValueError('Raw owner escapes model directory')
        return path

    def signature(self, name):
        if name not in self.signatures:
            path = self.safe_path(name)
            stat = path.stat()
            identity = [stat.st_size,stat.st_mtime_ns,stat.st_ctime_ns]
            if stat.st_size < 4096:
                obj = json.loads(path.read_bytes())
                if isinstance(obj,dict) and obj.get('storage_schema')=='georag-json-parquet-pointer-1.0':
                    target = self.safe_path(obj['payload_path'])
                    other = target.stat()
                    identity.extend([obj['payload_path'],obj['payload_sha256'],obj['payload_bytes'],
                                     other.st_size,other.st_mtime_ns,other.st_ctime_ns])
            self.signatures[name] = json.dumps(identity)
        return self.signatures[name]

    def remember(self, key, pointers):
        if len(pointers)>self.max_pointers:
            return
        while self.cache and (len(self.cache)>=self.max_owners or
                              self.cached_pointers+len(pointers)>self.max_pointers):
            _, old = self.cache.popitem(last=False)
            self.cached_pointers -= len(old)
        self.cache[key] = pointers
        self.cached_pointers += len(pointers)

    def pointers(self, name, chunk, current_file):
        signature = self.signature(name)
        key = name, signature
        if key in self.cache:
            self.cache.move_to_end(key)
            return self.cache[key]
        row = self.db.execute('SELECT signature FROM owners WHERE name=?',(name,)).fetchone()
        if row is not None and row[0] == signature:
            pointers = {r[0] for r in self.db.execute('SELECT pointer FROM refs WHERE name=?',(name,))}
            self.stats['catalog_reuses'] += 1
        else:
            if name==current_file and chunk is not None:
                owner = chunk  # load_shard already validated this current payload.
            else:
                owner = self.ns['load_prediction_owner'](name)
                self.stats['owner_loads'] += 1
            pointers = set()
            for slot, record in enumerate(owner['results']):
                if record.get('raw') is not None:
                    base = f'/results/{slot}/raw'
                    pointers.add(base)
                    if isinstance(record['raw'],list):
                        for wi, window in enumerate(record['raw']):
                            if isinstance(window,dict) and window.get('output') is not None:
                                pointers.add(base+f'/{wi}/output')
            if self.signature_after_load(name) != signature:
                raise RuntimeError('Raw owner changed during validation: '+name)
            with self.db:
                self.db.execute('DELETE FROM refs WHERE name=?',(name,))
                self.db.executemany('INSERT INTO refs VALUES (?,?)',((name,p) for p in pointers))
                self.db.execute('INSERT OR REPLACE INTO owners VALUES (?,?)',(name,signature))
            self.stats['owners_catalogued'] += 1
        self.remember(key,pointers)
        return pointers

    def signature_after_load(self, name):
        self.signatures.pop(name,None)
        return self.signature(name)

    def check(self, ref, chunk=None, current_file=None):
        name,pointer = ref.split('#',1)
        pointers = self.pointers(name,chunk,current_file)
        if pointer not in pointers:
            # Preserve compatibility with unusual legacy pointer shapes. A missing,
            # null or broken target must fail just as the original resolver did.
            value = self.ns['resolve_raw_reference'](ref,chunk,current_file)
            if value is None:
                raise AssertionError('Raw reference resolves to null: '+ref)
            with self.db:
                self.db.execute('INSERT OR IGNORE INTO refs VALUES (?,?)',(name,pointer))
            # Do not enlarge the bounded in-memory cache with arbitrary fallbacks.
            self.stats['fallback_resolutions'] += 1
        else:
            self.stats['catalog_checks'] += 1
        return True

with cell("11 — coverage, offsets and persisted outputs"), RawReferenceAudit(globals()) as raw_audit:
    summary={}
    for name in MODELS:
        # Each raw checkpoint is read, hash-checked and audited once in this cell.
        covered,totals=set(),Counter()
        inference_seconds=0.0
        counts=Counter();labels=Counter();started,last,failures=time.perf_counter(),0,0
        for start in range(0,len(SAMPLE),CHECKPOINT_PARAGRAPHS):
            raw_audit.begin_chunk()
            chunk=load_shard(name,start)
            raw_file=shard_path(name,start).relative_to(RUN_DIR).as_posix()
            for local,r in enumerate(chunk["results"]):
                p=r["paragraph"]
                pos=p["sample_position"]
                assert pos not in covered
                covered.add(pos)
                totals[r["status_processing"]]+=1
                inference_seconds+=r["processing_seconds"]
                failures+=r["status_processing"]!="success"
                counts["duplicates_resolved"]+=r.get("duplicates_resolved",0)
                counts["prediction_cache_hits"] += bool(r.get("occurrence_reused_prediction"))
                counts["duplicate_occurrences"] += bool(r.get("is_repeated_text_occurrence"))
                counts["delta_reused_occurrences"] += bool(r.get("reused_from_run"))
                counts["unique_text_owners"] += not bool(r.get("cache_hit"))
                counts["inference_paragraphs_evaluated"] += bool(r.get("inference_evaluated"))
                counts["skipped_no_content_occurrences"] += bool(r.get("normalization", {}).get("skipped_no_content"))
                if r.get("cache_hit"):
                    assert r["raw"] == []
                    assert raw_audit.check(r["raw_shared_ref"], chunk, raw_file)
                counts["error_messages"]+=len(r["errors"])
                for m in r["mentions"]:
                    counts[m["status_validation"]]+=1
                    labels[m["entity_group"] or "UNMAPPED"]+=1
                    if m["status_validation"]=="valid":
                        assert p["Text"][m["start"]:m["end"]]==m["word"]
                        assert m["start_document"]==p["start_document"]+m["start"]
                        assert m["end_document"]==p["start_document"]+m["end"]
                        assert_document_span(p, m["start_document"], m["end_document"], m["word"])
                    else:
                        assert m["error"]
                    for ref in m["ref_output_raw"]:
                        assert raw_audit.check(ref, chunk, raw_file)
                last=progress("audit_"+name,p["sample_position"]+1,len(SAMPLE),failures,started,last)
            del chunk
        assert covered==set(range(len(SAMPLE)))
        assert sum(totals[s] for s in ["success","partial_invalid","error"])==len(SAMPLE)
        for key in ["valid","pending","invalid","error_messages","duplicates_resolved"]:
            counts.setdefault(key,0)
        meta=load_json(RUN_DIR/"models"/name/"metadata.json")
        summary[name]={"planned":len(SAMPLE),"covered":len(covered),"coverage":len(covered)/len(SAMPLE),
            "success":totals["success"],"partial_invalid":totals["partial_invalid"],"error":totals["error"],
            "mentions_by_label":dict(labels),"counts":dict(counts),"inference_seconds":inference_seconds,
            "initial_load_seconds":meta.get("load_seconds"),"metadata":meta}
        summary[name]["prepared_counts"] = MODEL_OUTPUTS.get(name, {}).get("prepared_counts", {})
        summary[name]["inference_seconds"] = summary[name]["prepared_counts"].get("inference_processing_seconds_all_checkpoints", 0.0)
        counts["unique_text_owners"] = summary[name]["prepared_counts"].get("unique_total", 0)
        counts["inference_paragraphs_evaluated"] = summary[name]["prepared_counts"].get("unique_inferred", 0)
        summary[name]["counts"] = dict(counts)
        summary[name]["stage_timings_current_session"] = STAGE_TIMINGS["models"].get(name, {})
        summary[name]["stage_attempts"] = [load_json(p) for p in sorted((RUN_DIR/"models"/name).glob("stage_attempt_*.json"))]
        summary[name]["attempts"]=[load_json(p) for p in sorted((RUN_DIR/"models"/name).glob("attempt_*.json"))]
    for kind in ["selected","candidates","conflicts"]:
        index=load_json(OUTPUT_FILES[kind+"_json"])
        assert index["sample_sha256"]==SAMPLE_HASH and index["source"]==SOURCE_META
        actual=0
        for part in index["parts"]:
            path=EXPORT_DIR/part["path"]
            assert digest_file(path)==part["sha256"]
            saved=load_json(path)
            assert saved["sample_sha256"]==SAMPLE_HASH
            assert len(saved["records"])==part["record_count"]
            keys=set()
            for m in saved["records"]:
                key = span_key(m) + (m['model'], m['label_original']) if kind == 'conflicts' else span_key(m)
                assert key not in keys
                keys.add(key)
                assert m["status_validation"]=="valid"
                assert m["Text"][m["start"]:m["end"]]==m["word"]
                if kind=="selected":
                    assert m["entity_group"] not in {"TIME","EVENT"}
                    assert m["score"]==m["selection_score"]
                elif kind == "conflicts":
                    assert m["conflict_status"] in {"unresolved", "resolved_by_primary", "no_selected_candidate"}
                    if m["conflict_status"] == "unresolved":
                        assert not m["is_selected"]
                else:
                    assert m["entity_group"] in {"TIME","EVENT"}
                    assert m["selection_status"].startswith("candidate") and not m["verified_event_fact"]
            actual+=len(saved["records"])
        assert actual==index["record_count"]==EXPORT_COUNTS[kind]
        if kind == "selected" and CONFLICT_POLICY == "quarantine_exact_span":
            for part in index["parts"]:
                at_span = defaultdict(set)
                for record in load_json(EXPORT_DIR / part["path"])["records"]:
                    at_span[(record["Source"],record["Index"],record["start"],record["end"])].add(record["entity_group"])
                assert all(len(labels) == 1 for labels in at_span.values())
        assert verify_parquet_roundtrip(kind)==actual
    batch_event_count=sum(1 for _ in (RUN_DIR/"models").rglob("batch_event_*.json"))
    print("Raw reference audit cache:", dict(raw_audit.stats))
    RAW_REFERENCE_CACHE.clear()
    RUN_STATUS="completed_with_errors" if any(s["partial_invalid"] or s["error"] or s["counts"]["pending"] or s["counts"]["invalid"] for s in summary.values()) else "completed_with_warnings" if batch_event_count or EXPORT_COUNTS["unresolved_conflict_groups"] else "completed"
    atomic_json(EXPORT_DIR/"validation_summary.json",{"schema_version":SCHEMA_VERSION,
        "sample_sha256":SAMPLE_HASH,"run_status":RUN_STATUS,"mode":CONFIG["mode"],
        "batch_event_count":batch_event_count,"eligible":TOTAL_ELIGIBLE,"processed":len(SAMPLE),"models":summary,
        "exports":dict(EXPORT_COUNTS),"selected_by_label":dict(EXPORT_LABELS["selected"]),
        "candidate_by_label":dict(EXPORT_LABELS["candidates"]),"preparation":PREPARATION_REPORT,"stage_timings":STAGE_TIMINGS,"alignment_policy":ALIGNMENT_POLICY,"alignment_replay":ALIGNMENT_REPORT,"selftests":load_json(PATHS["selftests"])})
    print("Run status:",RUN_STATUS,"models:",json.dumps(summary,indent=2),"exports:",dict(EXPORT_COUNTS))


### 12 — החלפת הפלט הקבוע ב־GitHub

מפרסם את Parquet ואת run.json באותו commit. כשל מתועד ואינו מוחק את הפלטים המקומיים.


In [ ]:
with cell("12 — publication attempt; inspect GitHub publication status"):
    def safe_publication_error(exc):
        message = str(exc)
        values = list(globals().get("_GITHUB_TOKEN_RUNTIME", {}).values())
        values += [os.environ.get(n) for n in [GITHUB_TOKEN_SECRET, "GITHUB_TOKEN", "GH_TOKEN"]]
        for value in sorted((v for v in values if isinstance(v, str) and v), key=len, reverse=True):
            message = message.replace(value, "[REDACTED]")
        return {"type": type(exc).__name__, "message": message}
    if GITHUB_UPLOAD_ENABLED:
        persisted_summary = load_json(EXPORT_DIR / "validation_summary.json")
        assert persisted_summary["sample_sha256"] == SAMPLE_HASH
        assert all(persisted_summary["models"][n]["covered"] == len(SAMPLE) for n in MODELS)
        assert load_json(RUN_DIR / "run.json")["inference_sha256"] == INFERENCE_HASH
        assert persisted_summary["exports"]["selected"] == EXPORT_COUNTS["selected"]
        EXTRACTION_RUN_STATUS = RUN_STATUS = persisted_summary["run_status"]
        try:
            GITHUB_UPLOAD_RESULT = publish_current_parquet(OUTPUT_FILES["selected_parquet"], EXPORT_COUNTS["selected"])
        except Exception as exc:
            error = safe_publication_error(exc)
            GITHUB_UPLOAD_RESULT = {"schema_version": "github-current-publication-1.0", "status": "failed",
                "error": error, "destination": GITHUB_OUTPUT_DIR.rstrip("/") + "/" + GITHUB_FINAL_FILENAME,
                "local_path": str(OUTPUT_FILES["selected_parquet"]), "at_finished": utc_now()}
            RUN_STATUS = "completed_with_errors"
            print("WARNING fixed GitHub publication failed:", error)
            print("Local outputs remain available. Rerun this publication cell after fixing the cause.")
        else:
            print("GitHub fixed output:", GITHUB_UPLOAD_RESULT)
        atomic_json(EXPORT_DIR / f"github_upload_{secrets.token_hex(6)}.json", GITHUB_UPLOAD_RESULT)
    else:
        EXTRACTION_RUN_STATUS = RUN_STATUS
        GITHUB_UPLOAD_RESULT = {"status": "disabled"}
        print("GitHub publication disabled; local outputs retained")

    print("GITHUB PUBLICATION STATUS:", GITHUB_UPLOAD_RESULT["status"])


### 13 — ZIP ו־manifest

מכין ובודק ZIP, שומר manifest, מדפיס את כל הזמנים ומבקש הורדה ב־Colab.


In [ ]:
with cell("13 — ZIP preparation and verification"):
    assert all(model_complete(n) for n in MODELS)
    # Include source/checkpoints once, only the current export, and session selftests.
    artifacts=[RUN_DIR/"source.parquet",RUN_DIR/"sample.json",RUN_DIR/"run.json",PATHS["selftests"]]
    artifacts+=sorted(p for p in (RUN_DIR/"prepared").rglob("*") if p.is_file() and p.suffix in {".json",".parquet"})
    artifacts+=sorted(p for p in (RUN_DIR/"models").rglob("*.json") if p.is_file())
    artifacts+=sorted((RUN_DIR/"models/compact_storage/payloads").rglob("*.parquet"))
    artifacts+=sorted(p for p in (RUN_DIR/"recovery").rglob("*.json") if p.is_file())
    artifacts+=sorted(p for p in EXPORT_DIR.rglob("*") if p.is_file())
    members={p.relative_to(RUN_DIR).as_posix():p for p in artifacts}
    assert len(members)==len(artifacts)
    with zipfile.ZipFile(ZIP_PATH,"x",zipfile.ZIP_DEFLATED) as z:
        for member,path in members.items():
            z.write(path,f"{RUN_NAME}/{member}", compress_type=zipfile.ZIP_STORED if path.suffix == ".parquet" else zipfile.ZIP_DEFLATED)
    with zipfile.ZipFile(ZIP_PATH) as z:
        assert z.testzip() is None
        for member,path in members.items():
            with z.open(f"{RUN_NAME}/{member}") as f:
                assert stream_digest(f)==digest_file(path)
try:
    STAGE_TIMINGS["selection_and_export_seconds"] = sum(t["seconds"] for t in CELL_TIMINGS if t["title"].startswith("10 "))
    STAGE_TIMINGS["final_zip_save_seconds"] = sum(t["seconds"] for t in CELL_TIMINGS if t["title"].startswith("13 "))
    manifest={"schema_version":SCHEMA_VERSION,"run_name":RUN_NAME,"run_status":RUN_STATUS,
        "notebook_version":"v18-git-publication", "compact_storage":COMPACT_STORAGE_REPORT, "compact_restore_proof":COMPACT_RESTORE_PROOF, "compact_cleanup":COMPACT_CLEANUP_REPORT, "alignment_policy":ALIGNMENT_POLICY, "alignment_replay":ALIGNMENT_REPORT, "github_transport":dict(GITHUB_HTTP_STATS), "preparation":PREPARATION_REPORT, "stage_timings":STAGE_TIMINGS, "delta_reuse":DELTA_IMPORT_MANIFEST,
        "fixed_publication":CONFIG["fixed_publication"], "text_preprocessing":TEXT_PREPROCESSING,
        "prediction_cache_policy":CONFIG["prediction_cache"], "conflict_policy":CONFIG["conflicts"], "input_validation":INPUT_VALIDATION,
        "github_checkpoint_head":CHECKPOINT_GITHUB.published_head if CHECKPOINT_GITHUB else None, "resilience_policy":CONFIG["resilience"], "initial_resources":INITIAL_RESOURCES,
        "backup_packets":CHECKPOINT_BACKUPS.receipts if CHECKPOINT_BACKUPS else [], "batch_preflight":PREFLIGHT_STATUS, "batch_event_count":batch_event_count, "runtime":RUNTIME_INFO,
        "github_auth_source":globals().get("_GITHUB_AUTH_SOURCE", "not_requested"),
        "extraction_status":globals().get("EXTRACTION_RUN_STATUS", RUN_STATUS),
        "github_publication":GITHUB_UPLOAD_RESULT,
        "mode":CONFIG["mode"],"configuration":CONFIG,"configuration_sha256":CONFIG_HASH,
        "inference_sha256":INFERENCE_HASH,"sample_sha256":SAMPLE_HASH,"source":SOURCE_META,
        "legacy":LEGACY,"models":summary,"versions":VERSIONS,"cell_timings":CELL_TIMINGS,
        "total_session_measured_seconds":sum(t["seconds"] for t in CELL_TIMINGS),
        "selection_export_seconds":sum(t["seconds"] for t in CELL_TIMINGS if t["title"].startswith("10 —")),
        "inference_seconds_all_checkpoints":{n:summary[n]["inference_seconds"] for n in MODELS},
        "timing_scope":"cell blocks including ZIP preparation; excludes manifest bookkeeping/final verification/browser transfer",
        "exports":dict(EXPORT_COUNTS),"artifact_sha256":{k:digest_file(p) for k,p in members.items()},
        "artifact_paths":{k:str(p) for k,p in members.items()},"zip_path":str(ZIP_PATH)}
    manifest_path=EXPORT_DIR/"manifest.json"
    atomic_json(manifest_path,manifest)
    member=manifest_path.relative_to(RUN_DIR).as_posix()
    with zipfile.ZipFile(ZIP_PATH,"a",zipfile.ZIP_DEFLATED) as z:
        z.write(manifest_path,f"{RUN_NAME}/{member}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        assert z.testzip() is None
        expected={f"{RUN_NAME}/{k}" for k in members}|{f"{RUN_NAME}/{member}"}
        assert set(z.namelist())==expected and len(z.namelist())==len(expected)
        assert z.read(f"{RUN_NAME}/{member}")==manifest_path.read_bytes()
        for key,want in manifest["artifact_sha256"].items():
            with z.open(f"{RUN_NAME}/{key}") as f:
                assert stream_digest(f)==want
    STAGE_TIMINGS["selection_and_export_seconds"] = sum(t["seconds"] for t in CELL_TIMINGS if t["title"].startswith("10 "))
    STAGE_TIMINGS["final_zip_save_seconds"] = sum(t["seconds"] for t in CELL_TIMINGS if t["title"].startswith("13 "))
    print("STAGE TIMINGS:", json.dumps(STAGE_TIMINGS, indent=2))
    print("GITHUB TRANSPORT:", json.dumps(dict(GITHUB_HTTP_STATS), indent=2))
    print_cell_timings()
    print("Status:",RUN_STATUS,"ZIP:",ZIP_PATH,"SHA256:",digest_file(ZIP_PATH))
    delivery_status = deliver_run_archive(ZIP_PATH)
    print("SUCCESS manifest and final ZIP verified; delivery:", delivery_status)
except BaseException as exc:
    print(f"FAILED final manifest/archive: {type(exc).__name__}: {exc}")
    raise
